In [ ]:
from __future__ import annotations  # must stay the first statement of the cell

# =====================================================================================
# SEC valuation & fundamentals review: everything in ONE notebook cell / script.
# Generated from the valuation/ package by tools/build_single_cell.py (edit the package,
# then rebuild; do not edit this file by hand).
#
# Needs: pandas, numpy, requests, openpyxl (altair is optional, for charts).
#   %pip install pandas numpy requests openpyxl altair
# Nothing is written to disk unless OUTPUT_DIR is set, and OUTPUT_DIR may not be on C:.
# =====================================================================================

# ----------------------------------- SETTINGS -----------------------------------------
TICKERS = None            # None = the default 497-ticker list; or ["AAPL", "MSFT"]; or a pasted "'AAPL','MSFT'"
LIMIT = None              # review only the first N tickers (quick test), e.g. 5
AS_OF = None              # Input D as-of date: None = today, or "2026-09-14"
CURRENT_PRICES = {}       # optional Input D prices, e.g. {"AAPL": 333.08}; default = last close on/before AS_OF
WORKERS = 4               # parallel threads for fetching and reviewing
OUTPUT_DIR = None         # None = keep everything in memory; or a folder NOT on C:, e.g. r"D:\valuation"
WRITE_FILES = True        # with OUTPUT_DIR: write <ticker>_valuation.xlsx and <ticker>_report.md per company
REFERENCE_XLSX = None     # optional CompanyCharts-style workbook to compare #Obs/#Inc/#Rec, e.g. r"D:\data\spy.xlsx"
REFERENCE_COMPATIBLE = False  # True = apply the reference file's own conventions in that comparison (see notes below)
SEC_USER_AGENT = "ValuationReview research-client admin@example.com"  # SEC asks for "Name email"; set your own
SHOW_FIRST_REPORT = True  # display the first company's report (sections 1-10) after the run
# --------------------------------------------------------------------------------------
# Reference conventions (REFERENCE_COMPATIBLE=True), found by comparing with the reference file:
#   * the first observation in the window counts as a record high (the written spec says it does not);
#   * equity is counted over 60 observations, the TTM metrics over 57;
#   * CFO and FCF: the first value in the window includes the prior year-to-date amount, an artifact of
#     building quarters from year-to-date data that cannot be differenced at the start of the dataset;
#   * FCF = CFO where capex is missing.
# They only change the comparison, never the review, the workbook or the statistics.

# ============================== valuation/tags.py ==============================
"""XBRL tag catalog.

Each metric lists candidate tags per taxonomy in priority order. For every
period the first candidate with a reported value is used; any candidate other
than the first is a *fallback* and is written to the data-quality log.
"""


from dataclasses import dataclass, field

US = "us-gaap"
IFRS = "ifrs-full"
DEI = "dei"


@dataclass(frozen=True)
class MetricDef:
    key: str
    label: str
    kind: str  # "duration" or "instant"
    unit: str  # "money", "per_share" or "shares"
    additive: bool  # quarters may be derived by YTD differencing
    candidates: dict[str, tuple[str, ...]]
    notes: dict[str, str] = field(default_factory=dict)


def _m(key, label, kind, unit, additive, us=(), ifrs=(), notes=None) -> MetricDef:
    return MetricDef(key, label, kind, unit, additive, {US: tuple(us), IFRS: tuple(ifrs)}, notes or {})


METRICS: dict[str, MetricDef] = {
    m.key: m
    for m in [
        # ----- income statement / cash flow (durations) -----
        _m(
            "revenue", "Revenue", "duration", "money", True,
            us=(
                "Revenues",
                "RevenueFromContractWithCustomerExcludingAssessedTax",
                "RevenueFromContractWithCustomerIncludingAssessedTax",
                "SalesRevenueNet",
                "SalesRevenueGoodsNet",
                "SalesRevenueServicesNet",
                "RevenuesNetOfInterestExpense",
                "RegulatedAndUnregulatedOperatingRevenue",
                "OperatingLeaseLeaseIncome",
            ),
            ifrs=("Revenue", "RevenueFromContractsWithCustomers", "RevenueFromSaleOfGoods"),
            notes={"OperatingLeaseLeaseIncome": "lease income only (lessor with no total revenue tag)"},
        ),
        _m(
            "lease_income", "Operating lease income", "duration", "money", True,
            us=("OperatingLeaseLeaseIncome", "OperatingLeasesIncomeStatementLeaseRevenue"),
        ),
        _m(
            "cost_of_revenue", "Cost of revenue", "duration", "money", True,
            us=(
                "CostOfRevenue",
                "CostOfGoodsAndServicesSold",
                "CostOfGoodsSold",
                "CostOfServices",
                # Further cost-of-sales lines; each is screened per company as a possible component (see COMPONENT_SCREENED_COST_TAGS).
                "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization",
                "CostOfGoodsSoldExcludingDepreciationDepletionAndAmortization",
                "CostOfServicesExcludingDepreciationDepletionAndAmortization",
                "CostOfPurchasedOilAndGas",
            ),
            ifrs=("CostOfSales",),
            notes={
                "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization": "cost of sales excluding depreciation, depletion and amortization; gross profit is before D&A",
                "CostOfGoodsSoldExcludingDepreciationDepletionAndAmortization": "cost of goods sold excluding depreciation, depletion and amortization; gross profit is before D&A",
                "CostOfServicesExcludingDepreciationDepletionAndAmortization": "cost of services excluding depreciation, depletion and amortization; gross profit is before D&A",
                "CostOfPurchasedOilAndGas": "purchased crude oil, gas and products (the cost-of-sales line of energy filers)",
            },
        ),
        _m("gross_profit", "Gross profit", "duration", "money", True, us=("GrossProfit",), ifrs=("GrossProfit",)),
        _m(
            "operating_income", "Operating income", "duration", "money", True,
            us=("OperatingIncomeLoss",),
            ifrs=("ProfitLossFromOperatingActivities",),
        ),
        _m(
            "net_income", "Net income attributable to parent", "duration", "money", True,
            us=("NetIncomeLoss", "NetIncomeLossAvailableToCommonStockholdersBasic", "ProfitLoss"),
            ifrs=("ProfitLossAttributableToOwnersOfParent", "ProfitLoss"),
            notes={
                "NetIncomeLossAvailableToCommonStockholdersBasic": "after preferred dividends; not identical to NI attributable to parent",
                "ProfitLoss": "includes noncontrolling interests; NCI income subtracted where tagged, otherwise used as reported",
            },
        ),
        _m(
            "nci_income", "Net income attributable to noncontrolling interest", "duration", "money", True,
            us=("NetIncomeLossAttributableToNoncontrollingInterest",),
            ifrs=("ProfitLossAttributableToNoncontrollingInterests",),
        ),
        _m(
            "eps_diluted", "Diluted EPS", "duration", "per_share", False,
            us=("EarningsPerShareDiluted", "EarningsPerShareBasicAndDiluted"),
            ifrs=("DilutedEarningsLossPerShare", "BasicAndDilutedEarningsLossPerShare"),
        ),
        _m(
            "diluted_shares", "Weighted diluted shares", "duration", "shares", False,
            us=(
                "WeightedAverageNumberOfDilutedSharesOutstanding",
                "WeightedAverageNumberOfShareOutstandingBasicAndDiluted",
                "WeightedAverageNumberOfSharesOutstandingBasic",
            ),
            ifrs=("AdjustedWeightedAverageShares", "WeightedAverageShares"),
            notes={
                "WeightedAverageNumberOfSharesOutstandingBasic": "basic count used because no diluted count is tagged",
                "WeightedAverageShares": "basic count used because no diluted count is tagged",
            },
        ),
        _m(
            "basic_shares", "Weighted basic shares", "duration", "shares", False,
            us=("WeightedAverageNumberOfSharesOutstandingBasic", "WeightedAverageNumberOfShareOutstandingBasicAndDiluted"),
            ifrs=("WeightedAverageShares",),
        ),
        _m(
            "ocf", "Operating cash flow", "duration", "money", True,
            us=("NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"),
            ifrs=("CashFlowsFromUsedInOperatingActivities",),
            notes={"NetCashProvidedByUsedInOperatingActivitiesContinuingOperations": "continuing operations only"},
        ),
        # Continuing-operations OCF: used instead of the total wherever both are tagged for a period
        # (values.continuing_operations_ocf), so the series never mixes the two across periods.
        _m(
            "ocf_continuing", "Operating cash flow (continuing operations)", "duration", "money", True,
            us=("NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",),
        ),
        _m(
            "ocf_discontinued", "Operating cash flow (discontinued operations)", "duration", "money", True,
            us=("CashProvidedByUsedInOperatingActivitiesDiscontinuedOperations",),
        ),
        _m(
            "capex", "Capital expenditures", "duration", "money", True,
            us=(
                "PaymentsToAcquirePropertyPlantAndEquipment",
                "PaymentsToAcquireProductiveAssets",
                "PaymentsForCapitalImprovements",
                "PaymentsToAcquireOtherProductiveAssets",
                "PaymentsToAcquireOilAndGasPropertyAndEquipment",
                "PaymentsToExploreAndDevelopOilAndGasProperties",
                "PaymentsToAcquireOtherPropertyPlantAndEquipment",
                "PaymentsForConstructionInProcess",
                "PaymentsForProceedsFromProductiveAssets",
                "PaymentsToAcquireFurnitureAndFixtures",
            ),
            ifrs=("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "PurchaseOfPropertyPlantAndEquipment"),
            notes={
                "PaymentsToAcquireProductiveAssets": "may include intangibles and other productive assets",
                "PaymentsToAcquireOtherProductiveAssets": "'other productive assets' line used as capex (no PP&E capex tag for the period)",
                "PaymentsToAcquireOilAndGasPropertyAndEquipment": "oil and gas property additions; other PP&E lines of the same period are added",
                "PaymentsToAcquireOtherPropertyPlantAndEquipment": "'other PP&E' line; other PP&E lines of the same period are added",
                "PaymentsToExploreAndDevelopOilAndGasProperties": "exploration and development additions; other PP&E lines of the same period are added",
                "PaymentsForConstructionInProcess": "construction expenditures (utility/REIT presentation)",
                "PaymentsForProceedsFromProductiveAssets": "net of disposal proceeds (the only capex line tagged)",
                "PaymentsToAcquireFurnitureAndFixtures": "furniture and fixtures purchases are the only PP&E purchase line tagged",
            },
        ),
        _m(
            "da", "Depreciation & amortization (cash flow)", "duration", "money", True,
            us=(
                "DepreciationDepletionAndAmortization",
                "DepreciationAmortizationAndAccretionNet",
                "DepreciationAndAmortization",
                "Depreciation",
            ),
            ifrs=(
                "AdjustmentsForDepreciationAndAmortisationExpense",
                "DepreciationAndAmortisationExpense",
            ),
            notes={
                "Depreciation": "depreciation only; amortization not included",
                "DepreciationAmortizationAndAccretionNet": "includes accretion",
                "DepreciationAndAmortisationExpense": "income-statement D&A, not the cash-flow add-back",
            },
        ),
        _m(
            "dividends_paid", "Dividends paid (cash flow)", "duration", "money", True,
            us=("PaymentsOfDividendsCommonStock", "PaymentsOfDividends"),
            ifrs=("DividendsPaidClassifiedAsFinancingActivities", "DividendsPaid"),
            notes={"PaymentsOfDividends": "may include preferred and NCI dividends"},
        ),
        _m(
            "dps_declared", "Dividends declared per share (reference only)", "duration", "per_share", False,
            us=("CommonStockDividendsPerShareDeclared", "CommonStockDividendsPerShareCashPaid"),
            ifrs=("DividendsRecognisedAsDistributionsToOwnersPerShare",),
        ),
        # one-time items used only for the red-flag screen
        _m(
            "impairment", "Impairment charges", "duration", "money", True,
            us=(
                "AssetImpairmentCharges",
                "GoodwillImpairmentLoss",
                "ImpairmentOfLongLivedAssetsHeldForUse",
                "ImpairmentOfIntangibleAssetsExcludingGoodwill",
                "GoodwillAndIntangibleAssetImpairment",
            ),
            ifrs=("ImpairmentLossRecognisedInProfitOrLoss",),
        ),
        _m(
            "disc_ops", "Income from discontinued operations", "duration", "money", True,
            us=(
                "IncomeLossFromDiscontinuedOperationsNetOfTaxAttributableToReportingEntity",
                "IncomeLossFromDiscontinuedOperationsNetOfTax",
            ),
            ifrs=("ProfitLossFromDiscontinuedOperations",),
        ),
        _m(
            "disposal_gains", "Gain (loss) on disposals", "duration", "money", True,
            us=(
                "GainLossOnDispositionOfAssets",
                "GainLossOnSaleOfPropertyPlantEquipment",
                "DisposalGroupNotDiscontinuedOperationGainLossOnDisposal",
                "GainLossOnSaleOfBusiness",
            ),
            ifrs=("GainsLossesOnDisposalsOfPropertyPlantAndEquipment",),
        ),
        # ----- balance sheet (instants) -----
        _m(
            "cash", "Cash and cash equivalents", "instant", "money", False,
            us=(
                "CashAndCashEquivalentsAtCarryingValue",
                "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
                "Cash",
            ),
            ifrs=("CashAndCashEquivalents",),
            notes={"CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents": "includes restricted cash"},
        ),
        _m("total_assets", "Total assets", "instant", "money", False, us=("Assets",), ifrs=("Assets",)),
        _m("total_liabilities", "Total liabilities", "instant", "money", False, us=("Liabilities",), ifrs=("Liabilities",)),
        _m(
            "equity", "Parent equity", "instant", "money", False,
            us=("StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"),
            ifrs=("EquityAttributableToOwnersOfParent", "Equity"),
            notes={
                "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest": "total equity; noncontrolling interest subtracted where tagged at the same date",
                "Equity": "total equity; noncontrolling interest subtracted where tagged at the same date",
            },
        ),
        _m(
            "long_term_debt", "Long-term debt", "instant", "money", False,
            us=("LongTermDebtNoncurrent", "LongTermDebt", "LongTermDebtAndCapitalLeaseObligations"),
            ifrs=("NoncurrentPortionOfNoncurrentBorrowings", "LongtermBorrowings"),
            notes={
                "LongTermDebt": "includes current maturities",
                "LongTermDebtAndCapitalLeaseObligations": "includes finance-lease obligations",
            },
        ),
        _m("goodwill", "Goodwill", "instant", "money", False, us=("Goodwill",), ifrs=("Goodwill",)),
        _m(
            "intangibles", "Intangible assets excl. goodwill", "instant", "money", False,
            us=("IntangibleAssetsNetExcludingGoodwill", "FiniteLivedIntangibleAssetsNet"),
            ifrs=("IntangibleAssetsOtherThanGoodwill",),
            notes={"FiniteLivedIntangibleAssetsNet": "finite-lived only"},
        ),
        _m("nci", "Noncontrolling interest", "instant", "money", False, us=("MinorityInterest",), ifrs=("NoncontrollingInterests",)),
        _m("preferred", "Preferred stock", "instant", "money", False, us=("PreferredStockValue", "PreferredStockValueOutstanding"), ifrs=()),
        _m(
            "bs_shares", "Shares outstanding (balance sheet)", "instant", "shares", False,
            us=("CommonStockSharesOutstanding",),
            ifrs=("NumberOfSharesOutstanding",),
        ),
    ]
}

# Borrowing components for EV. Each slot is resolved separately so the
# workbook can show what was tagged. "LongTermDebt" (which includes current
# maturities) is only used when no noncurrent tag exists.
DEBT_TAGS = {
    US: {
        "debt_current_total": ("DebtCurrent",),
        "ltd_current": ("LongTermDebtCurrent", "LongTermDebtAndCapitalLeaseObligationsCurrent"),
        "short_term": ("ShortTermBorrowings", "CommercialPaper"),
        "ltd_noncurrent": ("LongTermDebtNoncurrent", "LongTermDebtAndCapitalLeaseObligations"),
        "ltd_total": ("LongTermDebt",),
    },
    IFRS: {
        "debt_current_total": ("CurrentBorrowingsAndCurrentPortionOfNoncurrentBorrowings",),
        "ltd_current": ("CurrentPortionOfNoncurrentBorrowings", "CurrentPortionOfLongtermBorrowings"),
        "short_term": ("ShorttermBorrowings",),
        "ltd_noncurrent": ("NoncurrentPortionOfNoncurrentBorrowings", "LongtermBorrowings"),
        "ltd_total": ("Borrowings",),
    },
}

# Cost-of-revenue fallbacks that some filers use for a single component of cost of sales (e.g. equipment
# cost only). Per company, such a tag is used only if it never disagrees with a higher-priority cost tag,
# or with revenue minus tagged GrossProfit, in a period both report (values.screen_component_cost_tags).
COMPONENT_SCREENED_COST_TAGS = frozenset(
    {
        "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization",
        "CostOfGoodsSoldExcludingDepreciationDepletionAndAmortization",
        "CostOfServicesExcludingDepreciationDepletionAndAmortization",
        "CostOfPurchasedOilAndGas",
    }
)

# Screened tags that are used only when at least one period confirms them against a higher-priority cost tag or
# revenue minus GrossProfit (as for CVX, COP and PSX before 2017); without that, they are often a single component
# (a utility's gas purchased, a REIT's service costs).
CONFIRMATION_REQUIRED_COST_TAGS = frozenset({"CostOfPurchasedOilAndGas", "CostOfServicesExcludingDepreciationDepletionAndAmortization"})

SPLIT_RATIO_TAGS = ("StockholdersEquityNoteStockSplitConversionRatio1",)
COVER_SHARES_TAG = "EntityCommonStockSharesOutstanding"

# Tags whose periods define the fiscal calendar and filing report periods.
CALENDAR_METRICS = ("revenue", "net_income", "operating_income", "ocf", "eps_diluted", "diluted_shares", "gross_profit")
BALANCE_ANCHOR_METRICS = ("total_assets", "equity", "cash")

PERIODIC_FORMS = {"10-K", "10-Q", "20-F", "40-F", "10-KT", "10-QT", "10-K405"}

# Split-like factors (shares multiply by k). Reverse splits are 1/k.
SPLIT_FACTORS = (1.5, 2, 2.5, 3, 4, 5, 6, 7, 8, 10, 12, 15, 20, 25, 30, 40, 50, 100)


def all_tags_for(metric: MetricDef) -> list[tuple[str, str, int]]:
    """(taxonomy, tag, priority) for every candidate of a metric."""
    out = []
    for taxonomy, tags in metric.candidates.items():
        for priority, tag in enumerate(tags):
            out.append((taxonomy, tag, priority))
    return out

# Capex presentations that split PP&E purchases over several cash-flow lines. When the line chosen
# for a period is one of the COMPONENT tags, the other component lines of the same period are added
# (they are separate rows of the same statement, e.g. oil and gas additions + other PP&E, or capital
# improvements + construction in process). A TOTAL tag is used alone, unless construction
# expenditures for the same period exceed it (then the "total" tag holds a smaller separate line,
# e.g. a utility's asset acquisitions, and construction expenditures are the capex line).
CAPEX_TOTAL_TAGS = ("PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets")
CAPEX_COMPONENT_TAGS = (
    "PaymentsToAcquireOilAndGasPropertyAndEquipment",
    "PaymentsToExploreAndDevelopOilAndGasProperties",
    "PaymentsToAcquireOtherPropertyPlantAndEquipment",
    "PaymentsForCapitalImprovements",
    "PaymentsForConstructionInProcess",
)

# ============================== valuation/dq.py ==============================
"""Data-quality log shared by every stage of the pipeline."""


from dataclasses import dataclass, field

import pandas as pd

CATEGORIES = (
    "input",
    "coverage",
    "fallback_tag",
    "derived_value",
    "restatement",
    "split",
    "amendment",
    "derived_quarter",
    "quarter_mismatch",
    "ttm_method",
    "eps_crosscheck",
    "shares_method",
    "period_label",
    "fiscal_calendar",
    "taxonomy_change",
    "currency_change",
    "missing_period",
    "price_gap",
    "stale_price",
    "price_basis",
    "ev_choice",
    "extreme_multiple",
    "assumption",
)


@dataclass
class DQLog:
    rows: list[dict] = field(default_factory=list)

    def add(self, category: str, item: str, detail: str, period=None, filing: str | None = None, severity: str = "info") -> None:
        if category not in CATEGORIES:
            raise ValueError(f"Unknown data-quality category {category!r}")
        if isinstance(period, pd.Timestamp):
            period = period.date().isoformat()
        self.rows.append(
            {
                "category": category,
                "severity": severity,
                "item": item,
                "period": "" if period is None else str(period),
                "detail": detail,
                "filing": filing or "",
            }
        )

    def frame(self) -> pd.DataFrame:
        if not self.rows:
            return pd.DataFrame(columns=["category", "severity", "item", "period", "detail", "filing"])
        order = {name: index for index, name in enumerate(CATEGORIES)}
        frame = pd.DataFrame(self.rows)
        frame["_order"] = frame["category"].map(order)
        return frame.sort_values(["_order", "item", "period"], kind="stable").drop(columns="_order").reset_index(drop=True)

# ============================== valuation/storage.py ==============================
"""Disk-use policy.

Nothing is written to disk unless the caller names an output directory, and
that directory may not be on the C: drive. Inputs fetched from SEC and Yahoo
stay in memory; workbooks and CSVs are built as in-memory bytes.
"""


import os
import re
from pathlib import Path, PureWindowsPath

ENV_DIR = "VALUATION_OUTPUT_DIR"
_C_DRIVE = re.compile(r"^[cC]:")


class StorageRefused(ValueError):
    pass


def _drive_of(path: Path) -> str:
    text = str(path)
    if os.name == "nt":
        return PureWindowsPath(text).drive
    # WSL / Git-Bash style mounts of the Windows C: drive.
    lowered = text.replace("\\", "/").lower()
    if lowered.startswith("/mnt/c/") or lowered == "/mnt/c" or lowered.startswith("/c/"):
        return "C:"
    return PureWindowsPath(text).drive


def is_c_drive(path: str | Path) -> bool:
    """True for C:\\..., c:/..., /mnt/c/..., and relative paths that resolve onto C: (e.g. a repo on C:)."""
    text = str(path).strip()
    lowered = text.replace("\\", "/").lower()
    if _C_DRIVE.match(text) or lowered.startswith(("/mnt/c/", "/c/")) or lowered in ("/mnt/c", "/c"):
        return True
    if PureWindowsPath(text).drive:  # another drive letter, e.g. D:
        return False
    resolved = Path(text).expanduser()
    try:
        resolved = resolved.resolve()
    except OSError:
        pass
    return bool(_C_DRIVE.match(_drive_of(resolved)))


def output_dir(requested: str | Path | None) -> Path | None:
    """The directory files may be written to, or None for memory-only operation."""
    chosen = requested or os.environ.get(ENV_DIR) or None
    if not chosen:
        return None
    path = Path(chosen).expanduser()
    if is_c_drive(path):
        raise StorageRefused(
            f"Refusing to write to {path}: it is on the C: drive. Choose a folder on another drive "
            f"(for example D:\\valuation_out) or omit the output folder to keep everything in memory."
        )
    path.mkdir(parents=True, exist_ok=True)
    return path

# ============================== valuation/facts.py ==============================
"""Load and normalize SEC companyfacts JSON (Input A)."""


import json
from dataclasses import dataclass
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd



@dataclass
class CompanyFacts:
    cik: str
    entity_name: str
    facts: pd.DataFrame  # one row per reported fact, filed on or before the as-of date
    filings: pd.DataFrame  # one row per accession
    excluded_after_asof: int  # facts dropped because they were filed after the as-of date
    taxonomies: list[str]


def _to_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def read_companyfacts(source) -> dict:
    if isinstance(source, dict):
        return source
    if isinstance(source, (str, Path)):
        return json.loads(Path(source).read_text(encoding="utf-8"))
    if hasattr(source, "read"):
        data = source.read()
        if isinstance(data, bytes):
            data = data.decode("utf-8")
        return json.loads(data)
    raise TypeError("Unsupported companyfacts source")


def load_companyfacts(source, as_of: date) -> CompanyFacts:
    raw = read_companyfacts(source)
    rows = []
    for taxonomy, concepts in (raw.get("facts") or {}).items():
        for tag, payload in concepts.items():
            for unit, items in (payload.get("units") or {}).items():
                for item in items:
                    rows.append(
                        (
                            taxonomy,
                            tag,
                            unit,
                            item.get("start"),
                            item.get("end"),
                            item.get("val"),
                            item.get("accn"),
                            item.get("fy"),
                            item.get("fp"),
                            item.get("form"),
                            item.get("filed"),
                            item.get("frame"),
                        )
                    )
    columns = ["taxonomy", "tag", "unit", "start", "end", "val", "accn", "fy", "fp", "form", "filed", "frame"]
    facts = pd.DataFrame(rows, columns=columns)
    if facts.empty:
        raise ValueError("Input A contains no facts.")
    facts["start"] = _to_date(facts["start"])
    facts["end"] = _to_date(facts["end"])
    facts["filed"] = _to_date(facts["filed"])
    facts["val"] = pd.to_numeric(facts["val"], errors="coerce")
    facts = facts.dropna(subset=["end", "filed", "val", "accn"]).copy()
    facts["form"] = facts["form"].fillna("").astype(str)
    facts["kind"] = np.where(facts["start"].isna(), "instant", "duration")
    # Inclusive day count: a 52-week year is 364 days, a 53-week year 371.
    facts["days"] = (facts["end"] - facts["start"]).dt.days + 1

    as_of_ts = pd.Timestamp(as_of)
    excluded = int((facts["filed"] > as_of_ts).sum())
    facts = facts[facts["filed"] <= as_of_ts].copy()
    if facts.empty:
        raise ValueError("Input A has no facts filed on or before the as-of date.")

    facts["currency"] = facts["unit"].map(_unit_currency)
    filings = _build_filings(facts)
    facts = facts.merge(filings[["accn", "dominant_currency", "regime_taxonomy"]], on="accn", how="left")
    cik = str(raw.get("cik", "")).zfill(10)
    taxonomies = sorted(set(facts["taxonomy"]) & {US, IFRS})
    return CompanyFacts(cik, str(raw.get("entityName", "")), facts, filings, excluded, taxonomies)


def _unit_currency(unit: str) -> str | None:
    """'USD' -> 'USD'; 'USD/shares' -> 'USD'; 'shares' -> None."""
    if not isinstance(unit, str):
        return None
    head = unit.split("/")[0]
    if len(head) == 3 and head.isalpha() and head.isupper():
        return head
    return None


def _calendar_tags() -> set[tuple[str, str]]:
    tags = set()
    for key in CALENDAR_METRICS:
        for taxonomy, names in METRICS[key].candidates.items():
            tags.update((taxonomy, name) for name in names)
    return tags


def _build_filings(facts: pd.DataFrame) -> pd.DataFrame:
    """One row per accession: form, filed date, regime and report period."""
    base = (
        facts.sort_values("filed")
        .groupby("accn", as_index=False)
        .agg(form=("form", "first"), filed=("filed", "min"))
    )
    money = facts[facts["currency"].notna() & facts["taxonomy"].isin([US, IFRS])]
    dominant = (
        money.groupby(["accn", "currency"]).size().reset_index(name="n").sort_values(["accn", "n"], ascending=[True, False])
        .drop_duplicates("accn")[["accn", "currency"]]
        .rename(columns={"currency": "dominant_currency"})
    )
    tax_counts = facts[facts["taxonomy"].isin([US, IFRS])].groupby(["accn", "taxonomy"]).size().unstack(fill_value=0)
    regime_tax = pd.Series(US, index=tax_counts.index, name="regime_taxonomy")
    if IFRS in tax_counts.columns:
        us_counts = tax_counts[US] if US in tax_counts.columns else 0
        regime_tax = regime_tax.where(~(tax_counts[IFRS] > us_counts), IFRS)
    regime_tax = regime_tax.reset_index()

    calendar_tags = _calendar_tags()
    keyed = facts[(facts["taxonomy"] + ":" + facts["tag"]).isin({f"{t}:{g}" for t, g in calendar_tags})]
    durations = keyed[keyed["kind"] == "duration"]
    report_period = durations.groupby("accn")["end"].max().rename("report_period").reset_index()
    filings = base.merge(dominant, on="accn", how="left").merge(regime_tax, on="accn", how="left").merge(report_period, on="accn", how="left")
    filings["regime_taxonomy"] = filings["regime_taxonomy"].fillna(US)
    filings["is_amendment"] = filings["form"].str.endswith("/A")
    return filings.sort_values(["filed", "accn"]).reset_index(drop=True)

# ============================== valuation/splits.py ==============================
"""Stock-split detection and split-basis normalization.

Evidence comes from Input A: the same period's share count (or per-share
value) reported in two filings at a ratio close to a split factor. The split
ex-date is located in Input C when prices are not split-adjusted. Explicit
split events supplied with Input C are also honoured.

Basis convention: every share count and per-share value is restated to the
split basis in force on the as-of date ("current basis").
"""


import math
from dataclasses import dataclass, field

import numpy as np
import pandas as pd


SHARE_TOL = 0.02  # share counts are reported precisely
PER_SHARE_TOL = 0.03  # EPS is rounded to cents
MIN_EPS_FOR_EVIDENCE = 0.50


@dataclass
class SplitEvent:
    factor: float  # shares multiply by this factor (2.0 for 2-for-1, 0.1 for 1-for-10)
    window_start: pd.Timestamp | None  # last filing observed on the old basis
    window_end: pd.Timestamp | None  # first filing observed on the new basis
    effective: pd.Timestamp  # date from which the new basis applies
    source: str  # how `effective` was determined
    evidence: int = 0
    examples: list[str] = field(default_factory=list)
    price_located: bool = False
    share_evidence: int = 0

    @property
    def label(self) -> str:
        if self.factor >= 1:
            return f"{_fmt_factor(self.factor)}-for-1"
        return f"1-for-{_fmt_factor(1 / self.factor)}"


def _fmt_factor(value: float) -> str:
    return f"{value:g}"


def match_factor(ratio: float, tolerance: float) -> float | None:
    if not np.isfinite(ratio) or ratio <= 0:
        return None
    if abs(ratio - 1) <= max(tolerance * 2, 0.05):
        return None
    for k in SPLIT_FACTORS:
        if abs(ratio / k - 1) <= tolerance:
            return float(k)
        if abs(ratio * k - 1) <= tolerance:
            return 1.0 / k
    return None


def _evidence_rows(facts: pd.DataFrame) -> list[tuple[float, pd.Timestamp, pd.Timestamp, str]]:
    share_tags = set(METRICS["diluted_shares"].candidates["us-gaap"]) | set(METRICS["diluted_shares"].candidates["ifrs-full"])
    share_tags |= {"WeightedAverageNumberOfSharesOutstandingBasic", "WeightedAverageShares", "CommonStockSharesOutstanding"}
    per_share_tags = set(METRICS["eps_diluted"].candidates["us-gaap"]) | set(METRICS["eps_diluted"].candidates["ifrs-full"])
    per_share_tags |= {"EarningsPerShareBasic", "BasicEarningsLossPerShare"}

    events = []
    subset = facts[facts["tag"].isin(share_tags | per_share_tags) & (facts["taxonomy"] != "dei")]
    for (tag, unit, start, end), group in subset.groupby(["tag", "unit", "start", "end"], dropna=False):
        group = group.sort_values(["filed", "accn"]).drop_duplicates("accn", keep="last")
        if len(group) < 2:
            continue
        values = group["val"].to_numpy(dtype=float)
        filed = group["filed"].tolist()
        is_share = tag in share_tags
        for i in range(len(group) - 1):
            a, b = values[i], values[i + 1]
            if filed[i] == filed[i + 1] or a == 0 or b == 0 or np.sign(a) != np.sign(b):
                continue
            if is_share:
                ratio, tol = b / a, SHARE_TOL
            else:
                if min(abs(a), abs(b)) < MIN_EPS_FOR_EVIDENCE / 2:
                    continue  # cent rounding makes small per-share values unreliable evidence
                ratio, tol = a / b, PER_SHARE_TOL
            k = match_factor(ratio, tol)
            if k is None:
                continue
            period = f"{start.date() if pd.notna(start) else ''}..{end.date()}"
            events.append((k, filed[i], filed[i + 1], f"{tag} {period}: {a:g} -> {b:g}", is_share))
    return events


def _cluster(events) -> list[SplitEvent]:
    clusters: list[SplitEvent] = []
    for k, a, b, example, is_share in sorted(events, key=lambda e: (e[0], e[2], e[1])):
        placed = False
        for cluster in clusters:
            if abs(cluster.factor - k) > 1e-9:
                continue
            lo = max(cluster.window_start, a)
            hi = min(cluster.window_end, b)
            if lo < hi:
                cluster.window_start, cluster.window_end = lo, hi
                cluster.evidence += 1
                cluster.share_evidence += int(is_share)
                if len(cluster.examples) < 4:
                    cluster.examples.append(example)
                placed = True
                break
        if not placed:
            clusters.append(SplitEvent(k, a, b, b, "first filing on the new basis", 1, [example], False, int(is_share)))
    return clusters


def locate_in_prices(event: SplitEvent, raw_close: pd.Series) -> pd.Timestamp | None:
    """Find the trading day in the filing window whose close-to-close move matches 1/k."""
    if raw_close is None or raw_close.empty:
        return None
    series = raw_close.dropna()
    ratio = series / series.shift(1)
    lo = event.window_start if event.window_start is not None else series.index.min()
    hi = event.window_end if event.window_end is not None else series.index.max()
    window = ratio[(ratio.index > lo) & (ratio.index <= hi)].dropna()
    if window.empty:
        return None
    distance = (np.log(window) + math.log(event.factor)).abs()
    best = distance.idxmin()
    if distance.loc[best] < math.log(1.35):
        return best
    return None


def detect_splits(
    facts: pd.DataFrame,
    dq: DQLog,
    raw_close: pd.Series | None,
    prices_split_adjusted: bool | None,
    explicit: pd.DataFrame | None,
    as_of: pd.Timestamp,
) -> list[SplitEvent]:
    clusters = _cluster(_evidence_rows(facts))
    tagged = facts[facts["tag"].isin(SPLIT_RATIO_TAGS)].drop_duplicates(["val", "end"])
    for _, row in tagged.iterrows():
        dq.add("split", "tagged split ratio", f"{row['tag']} = {row['val']:g} (period ending {row['end'].date()}).", period=row["end"], filing=row["accn"])
    explicit_events = []
    if explicit is not None and not explicit.empty:
        for _, row in explicit.iterrows():
            explicit_events.append((pd.Timestamp(row["date"]), float(row["ratio"])))

    def corroborated(cluster: SplitEvent) -> bool:
        lo, hi = cluster.window_start, cluster.window_end + pd.Timedelta(days=5)
        if any(abs(r / cluster.factor - 1) < 0.02 and lo < d <= hi for d, r in explicit_events):
            return True
        return any(abs(v / cluster.factor - 1) < 0.02 for v in tagged["val"])

    confirmed: list[SplitEvent] = []
    for cluster in clusters:
        if corroborated(cluster) or (cluster.share_evidence >= 1 and cluster.evidence >= 2):
            confirmed.append(cluster)
            continue
        why = "per-share values only (often a restatement, not a split)" if cluster.share_evidence == 0 else "a single restated value"
        dq.add(
            "split",
            "possible split (not applied)",
            f"A {cluster.label} change is supported by {why}: {'; '.join(cluster.examples[:2])}. No share-count, tagged-ratio or Input C confirmation; not applied.",
            period=cluster.window_end,
            severity="warning",
        )
    first_filed = facts["filed"].min()

    for cluster in confirmed:
        matched = None
        for ex_date, ratio in explicit_events:
            if abs(ratio / cluster.factor - 1) < 0.02 and cluster.window_start < ex_date <= cluster.window_end + pd.Timedelta(days=5):
                matched = (ex_date, ratio)
                break
        if matched:
            cluster.effective, cluster.source, cluster.price_located = matched[0], "Input C split event", True
            explicit_events.remove(matched)
        elif prices_split_adjusted is False:
            located = locate_in_prices(cluster, raw_close)
            if located is not None:
                cluster.effective, cluster.source, cluster.price_located = located, "price move in Input C", True
        dq.add(
            "split",
            f"likely split {cluster.label}",
            (
                f"Factor {cluster.factor:g} (shares x{cluster.factor:g}, per-share /{cluster.factor:g}). "
                f"Old basis last seen in a filing on {cluster.window_start.date()}, new basis first seen {cluster.window_end.date()}. "
                f"Effective date used: {cluster.effective.date()} ({cluster.source}). Evidence: {cluster.evidence} restated values, e.g. "
                + "; ".join(cluster.examples[:3])
            ),
            period=cluster.effective,
            severity="warning",
        )

    for ex_date, ratio in explicit_events:
        if ex_date > as_of:
            continue
        before = ex_date < first_filed
        event = SplitEvent(ratio, None, None, ex_date, "Input C split event" + (" (before Input A history)" if before else " (not confirmed by filings)"), 0, [], True)
        confirmed.append(event)
        dq.add(
            "split",
            f"split from Input C {event.label}",
            (
                f"Split on {ex_date.date()} predates the first filing in Input A ({first_filed.date()}); it changes no filing value and only matters for unadjusted prices."
                if before
                else f"Split on {ex_date.date()} listed in Input C but not confirmed by restated filing values; applied as supplied."
            ),
            period=ex_date,
            severity="info" if before else "warning",
        )
    return sorted(confirmed, key=lambda e: e.effective)


class SplitBasis:
    """Cumulative split factors to restate values onto the as-of date's basis."""

    def __init__(self, events: list[SplitEvent], as_of: pd.Timestamp):
        self.events = [e for e in events if e.effective <= as_of]

    def share_factor(self, on_basis_of: pd.Timestamp) -> float:
        """Multiply a share count published on `on_basis_of` by this factor."""
        factor = 1.0
        for event in self.events:
            if event.effective > on_basis_of:
                factor *= event.factor
        return factor

    def price_factor(self, day: pd.Timestamp) -> float:
        """Divide an unadjusted price on `day` by this factor."""
        return self.share_factor(day)

    def uncertain(self, day: pd.Timestamp) -> SplitEvent | None:
        """A split whose ex-date was not located and whose filing window contains `day`."""
        for event in self.events:
            if not event.price_located and event.window_start is not None and event.window_start < day < event.window_end:
                return event
        return None

    def describe(self) -> str:
        if not self.events:
            return "no splits detected"
        return "; ".join(f"{e.label} effective {e.effective.date()} ({e.source})" for e in self.events)

    def describe_within(self, start: pd.Timestamp) -> str:
        events = [e for e in self.events if e.effective >= start]
        if not events:
            return "no splits during the filing history"
        return "; ".join(f"{e.label} effective {e.effective.date()} ({e.source})" for e in events)

# ============================== valuation/values.py ==============================
"""Per-metric value tables built from Input A.

For each period (start/end for durations, end for instants):
  * the value is the latest restated value filed on or before the as-of date;
  * the first filing that reported the period is kept for timing and audit;
  * share counts and per-share values are restated to the as-of split basis;
  * the first candidate tag with a value wins, other tags are logged fallbacks.
"""


from dataclasses import dataclass, replace

import numpy as np
import pandas as pd


RESTATEMENT_TOL = 0.005
ANNUAL_FORMS = {"10-K", "10-K/A", "10-K405", "10-KT", "10-KT/A", "20-F", "20-F/A", "40-F", "40-F/A"}
TOL_DAYS = 3


def _unit_mask(frame: pd.DataFrame, unit_type: str) -> pd.Series:
    if unit_type == "money":
        return frame["unit"].eq(frame["dominant_currency"]) & frame["currency"].notna()
    if unit_type == "per_share":
        return frame["unit"].str.endswith("/shares") & frame["currency"].eq(frame["dominant_currency"])
    if unit_type == "shares":
        return frame["unit"].eq("shares")
    raise ValueError(unit_type)


def _normalize(values: pd.Series, filed: pd.Series, unit_type: str, basis: SplitBasis) -> pd.Series:
    if unit_type == "money":
        return values.astype(float)
    factors = filed.map(basis.share_factor).astype(float)
    if unit_type == "shares":
        return values.astype(float) * factors
    return values.astype(float) / factors


def raw_tag_table(cf: CompanyFacts, taxonomy: str, tag: str, kind: str, unit_type: str, basis: SplitBasis) -> pd.DataFrame:
    """Latest and first reported value per period for a single tag."""
    facts = cf.facts
    subset = facts[(facts["taxonomy"] == taxonomy) & (facts["tag"] == tag) & (facts["kind"] == kind)]
    if subset.empty:
        return pd.DataFrame()
    subset = subset[_unit_mask(subset, unit_type)]
    if subset.empty:
        return pd.DataFrame()
    subset = subset.sort_values(["filed", "accn"]).copy()
    subset["norm"] = _normalize(subset["val"], subset["filed"], unit_type, basis)
    keys = ["start", "end"] if kind == "duration" else ["end"]
    ignored = pd.Series(dtype=float)
    if kind == "duration":
        # A full-year period takes its value from annual reports when one reports it; a full-year
        # value that appears in a 10-Q (often a mis-dated comparative) is ignored and logged.
        days = (subset["end"] - subset["start"]).dt.days + 1
        annual = days.between(350, 380)
        from_annual_form = subset["form"].isin(ANNUAL_FORMS)
        period = subset["start"].astype(str) + "|" + subset["end"].astype(str)
        covered = set(period[annual & from_annual_form])
        drop = annual & ~from_annual_form & period.isin(covered)
        if drop.any():
            ignored = subset[drop].groupby(keys)["norm"].last()
            subset = subset[~drop]
    latest = subset.groupby(keys, dropna=False).tail(1).set_index(keys)
    first = subset.groupby(keys, dropna=False).head(1).set_index(keys)
    n_filings = subset.groupby(keys, dropna=False)["accn"].nunique()
    table = pd.DataFrame(
        {
            "value": latest["norm"],
            "value_reported": latest["val"],
            "unit": latest["unit"],
            "accn": latest["accn"],
            "filed": latest["filed"],
            "form": latest["form"],
            "first_value": first["norm"],
            "first_accn": first["accn"],
            "first_filed": first["filed"],
            "first_form": first["form"],
            "n_filings": n_filings,
            "regime_taxonomy": latest["regime_taxonomy"],
            "currency": latest["dominant_currency"],
        }
    ).reset_index()
    if kind == "duration":
        table["days"] = (table["end"] - table["start"]).dt.days + 1
    else:
        table["start"] = pd.NaT
        table["days"] = np.nan
    table["taxonomy"] = taxonomy
    table["tag"] = tag
    table["ignored_quarterly_report_value"] = np.nan
    if not ignored.empty and kind == "duration":
        lookup = {(a, b): v for (a, b), v in ignored.items()}
        table["ignored_quarterly_report_value"] = [lookup.get((a, b), np.nan) for a, b in zip(table["start"], table["end"])]
        same = (table["ignored_quarterly_report_value"] - table["value"]).abs() <= table["value"].abs() * RESTATEMENT_TOL
        table.loc[same, "ignored_quarterly_report_value"] = np.nan
    diff = (table["value"] - table["first_value"]).abs()
    scale = table[["value", "first_value"]].abs().max(axis=1).replace(0, np.nan)
    table["restated"] = (diff / scale > RESTATEMENT_TOL) & (diff > 0)
    return table


@dataclass
class MetricTable:
    metric: MetricDef
    table: pd.DataFrame  # one row per period

    def __post_init__(self) -> None:
        t = self.table
        if t.empty:
            self._start = self._end = np.array([], dtype="int64")
            self._records: list[dict] = []
            return
        self._end = pd.to_datetime(t["end"]).to_numpy(dtype="datetime64[ns]").astype("int64")
        starts = pd.to_datetime(t["start"]) if "start" in t else pd.Series(pd.NaT, index=t.index)
        self._start = starts.to_numpy(dtype="datetime64[ns]").astype("int64")
        self._records = t.to_dict("records")

    def _match(self, start, end, tol: int = TOL_DAYS) -> dict | None:
        if not self._records:
            return None
        tol_ns = tol * 86_400_000_000_000
        end_ns = pd.Timestamp(end).value
        gap = np.abs(self._end - end_ns)
        mask = gap <= tol_ns
        if self.metric.kind == "duration":
            if start is None:
                return None
            sgap = np.abs(self._start - pd.Timestamp(start).value)
            mask &= sgap <= tol_ns
            gap = gap + sgap
        hits = np.flatnonzero(mask)
        if hits.size == 0:
            return None
        best = hits[np.argmin(gap[hits])] if hits.size > 1 else hits[0]
        return self._records[best]

    def get(self, start, end, tol: int = TOL_DAYS) -> dict | None:
        return self._match(start, end, tol)

    def value(self, start, end, tol: int = TOL_DAYS) -> float | None:
        row = self._match(start, end, tol)
        return None if row is None else float(row["value"])

    def annual(self) -> pd.DataFrame:
        t = self.table
        return t[(t["days"] >= 350) & (t["days"] <= 380)] if not t.empty else t


def build_metric_table(cf: CompanyFacts, metric: MetricDef, basis: SplitBasis, dq: DQLog) -> MetricTable:
    pieces = []
    for taxonomy, tags in metric.candidates.items():
        for priority, tag in enumerate(tags):
            table = raw_tag_table(cf, taxonomy, tag, metric.kind, metric.unit, basis)
            if table.empty:
                continue
            table["priority"] = priority
            pieces.append(table)
    if not pieces:
        return MetricTable(metric, pd.DataFrame())
    combined = pd.concat(pieces, ignore_index=True)
    keys = ["start", "end"] if metric.kind == "duration" else ["end"]
    # The taxonomy of the most recent filing that reported the period wins; then tag priority.
    latest_tax = combined.sort_values("filed").groupby(keys, dropna=False)["taxonomy"].last().rename("period_taxonomy")
    combined = combined.merge(latest_tax.reset_index(), on=keys, how="left")
    combined = combined[combined["taxonomy"] == combined["period_taxonomy"]].drop(columns="period_taxonomy")
    chosen = combined.sort_values(["priority"]).groupby(keys, dropna=False).head(1)
    chosen = chosen.sort_values(keys).reset_index(drop=True)

    for (taxonomy, tag), group in chosen[chosen["priority"] > 0].groupby(["taxonomy", "tag"]):
        primary = metric.candidates[taxonomy][0]
        note = metric.notes.get(tag, "")
        dq.add(
            "fallback_tag",
            metric.label,
            f"{taxonomy}:{tag} used instead of {primary} for {len(group)} period(s) ending "
            f"{group['end'].min().date()} to {group['end'].max().date()}." + (f" Note: {note}." if note else ""),
            severity="warning" if note else "info",
        )
    if "ignored_quarterly_report_value" in chosen:
        for _, row in chosen[chosen["ignored_quarterly_report_value"].notna()].iterrows():
            dq.add(
                "restatement",
                metric.label,
                f"{row['tag']} {row['start'].date()}..{row['end'].date()}: a quarterly report shows {row['ignored_quarterly_report_value']:,.4g} for this full year; "
                f"the annual report's {row['value']:,.4g} ({row['form']} {row['accn']}) is used.",
                period=row["end"],
                filing=row["accn"],
                severity="warning",
            )
    for _, row in chosen[chosen["restated"]].iterrows():
        period = f"{row['start'].date()}..{row['end'].date()}" if pd.notna(row["start"]) else f"{row['end'].date()}"
        change = (row["value"] - row["first_value"]) / abs(row["first_value"]) if row["first_value"] else np.nan
        dq.add(
            "restatement",
            metric.label,
            f"{row['tag']} {period}: first reported {row['first_value']:,.4g} ({row['first_form']} {row['first_accn']}), "
            f"latest {row['value']:,.4g} ({row['form']} {row['accn']}); change {change:+.2%}. Latest value used.",
            period=row["end"],
            filing=row["accn"],
        )
    return MetricTable(metric, chosen)


COMPONENT_TOL = 0.005


def _first_conflict(table: pd.DataFrame, references: list[pd.DataFrame], overlaps: list | None = None) -> tuple | None:
    """First period where `table` and a reference table both have a value and differ by more than COMPONENT_TOL.
    The number of overlapping periods is appended to `overlaps` when given."""
    mine = table[["start", "end", "value"]]
    for ref in references:
        both = mine.merge(ref[["start", "end", "value", "tag"]], on=["start", "end"], suffixes=("", "_ref"))
        if overlaps is not None:
            overlaps.append(len(both))
        if both.empty:
            continue
        a, b = both["value"].astype(float), both["value_ref"].astype(float)
        scale = pd.concat([a.abs(), b.abs()], axis=1).max(axis=1).replace(0, np.nan)
        bad = both[((a - b).abs() / scale > COMPONENT_TOL).fillna(False)]
        if not bad.empty:
            row = bad.sort_values("end").iloc[-1]
            return row["tag"], row["start"], row["end"], float(row["value"]), float(row["value_ref"])
    return None


def screen_component_cost_tags(cf: CompanyFacts, metric: MetricDef, gross_profit: MetricDef, revenue: MetricTable, basis: SplitBasis, dq: DQLog) -> MetricDef:
    """Drop, for this company, cost-of-revenue fallback tags that are shown to be only a component of cost of sales.

    A screened tag (tags.COMPONENT_SCREENED_COST_TAGS) is kept only if, in every period where it and a
    higher-priority cost tag are both reported, the two agree, and likewise against revenue minus tagged
    GrossProfit. A tag with no overlapping period is kept (nothing contradicts it) and is logged as a fallback
    when used.
    """
    pass  # was: relative import (names are global in the single cell)

    candidates: dict[str, tuple[str, ...]] = {}
    for taxonomy, tags in metric.candidates.items():
        references: list[pd.DataFrame] = []
        if not revenue.table.empty:
            for gp_tag in gross_profit.candidates.get(taxonomy, ()):
                gp = raw_tag_table(cf, taxonomy, gp_tag, gross_profit.kind, gross_profit.unit, basis)
                if gp.empty:
                    continue
                implied = revenue.table[["start", "end", "value"]].merge(gp[["start", "end", "value"]], on=["start", "end"], suffixes=("_rev", "_gp"))
                if not implied.empty:
                    implied = implied.assign(value=implied["value_rev"] - implied["value_gp"], tag=f"revenue - {gp_tag}")
                    references.append(implied[["start", "end", "value", "tag"]])
        kept = []
        for tag in tags:
            table = raw_tag_table(cf, taxonomy, tag, metric.kind, metric.unit, basis)
            overlaps: list[int] = []
            conflict = _first_conflict(table, references, overlaps) if tag in COMPONENT_SCREENED_COST_TAGS and not table.empty else None
            if conflict is None and tag in CONFIRMATION_REQUIRED_COST_TAGS and not table.empty and sum(overlaps) == 0:
                dq.add(
                    "fallback_tag",
                    metric.label,
                    f"{taxonomy}:{tag} not used: no period where it can be confirmed against a total cost-of-revenue line or revenue minus "
                    "GrossProfit, and for many filers it is only one component (e.g. gas purchased).",
                    severity="info",
                )
                continue
            if not table.empty:
                # Every higher-priority line is a reference, used or not: two different cost lines for the
                # same period mean the later one is not the total.
                references.append(table[["start", "end", "value", "tag"]])
            if conflict is not None:
                ref_tag, start, end, value, ref_value = conflict
                dq.add(
                    "fallback_tag",
                    metric.label,
                    f"{taxonomy}:{tag} not used: for {start.date()}..{end.date()} it is {value:,.6g} while {ref_tag} is {ref_value:,.6g}, "
                    "so it covers only part of cost of revenue for this company.",
                    severity="info",
                )
                continue
            kept.append(tag)
        candidates[taxonomy] = tuple(kept)
    return replace(metric, candidates=candidates)


def derive_gross_profit(gp: MetricTable, revenue: MetricTable, cost: MetricTable, dq: DQLog) -> MetricTable:
    """Fill gross profit as revenue minus cost of revenue where GrossProfit is not tagged."""
    if revenue.table.empty or cost.table.empty:
        return gp
    merged = revenue.table.merge(cost.table, on=["start", "end"], suffixes=("_rev", "_cor"))
    existing = set()
    if not gp.table.empty:
        existing = {(s, e) for s, e in zip(gp.table["start"], gp.table["end"])}
    rows = []
    for _, row in merged.iterrows():
        if (row["start"], row["end"]) in existing:
            continue
        rows.append(
            {
                "start": row["start"],
                "end": row["end"],
                "value": row["value_rev"] - row["value_cor"],
                "value_reported": np.nan,
                "unit": row["unit_rev"],
                "accn": row["accn_rev"],
                "filed": max(row["filed_rev"], row["filed_cor"]),
                "form": row["form_rev"],
                "first_value": row["first_value_rev"] - row["first_value_cor"],
                "first_accn": row["first_accn_rev"],
                "first_filed": max(row["first_filed_rev"], row["first_filed_cor"]),
                "first_form": row["first_form_rev"],
                "n_filings": row["n_filings_rev"],
                "regime_taxonomy": row["regime_taxonomy_rev"],
                "currency": row["currency_rev"],
                "days": row["days_rev"],
                "taxonomy": row["taxonomy_rev"],
                "tag": f"derived: {row['tag_rev']} - {row['tag_cor']}",
                "restated": bool(row["restated_rev"] or row["restated_cor"]),
                "priority": 99,
            }
        )
    if not rows:
        return gp
    derived = pd.DataFrame(rows)
    dq.add(
        "derived_value",
        "Gross profit",
        f"GrossProfit not tagged for {len(derived)} period(s) ending {derived['end'].min().date()} to "
        f"{derived['end'].max().date()}; computed as revenue minus cost of revenue from the same periods.",
        severity="warning",
    )
    table = pd.concat([gp.table, derived], ignore_index=True) if not gp.table.empty else derived
    return MetricTable(gp.metric, table.sort_values(["start", "end"]).reset_index(drop=True))


def parent_share_of_profit_loss(ni: MetricTable, nci: MetricTable, dq: DQLog) -> MetricTable:
    """Where ProfitLoss (which includes NCI) was the only net income tag, subtract tagged NCI income."""
    t = ni.table
    if t.empty or not (t["tag"] == "ProfitLoss").any():
        return ni
    t = t.copy()
    adjusted = 0
    for i in t.index[t["tag"] == "ProfitLoss"]:
        row = nci.get(t.at[i, "start"], t.at[i, "end"], tol=0)
        if row is not None and row["value"]:
            t.at[i, "value"] = float(t.at[i, "value"]) - float(row["value"])
            t.at[i, "first_value"] = float(t.at[i, "first_value"]) - float(row["value"])
            t.at[i, "tag"] = "derived: ProfitLoss - NetIncomeLossAttributableToNoncontrollingInterest"
            adjusted += 1
    if adjusted:
        dq.add("derived_value", ni.metric.label, f"{adjusted} period(s): NI attributable to parent = ProfitLoss - NCI income (both tagged for the same period).", severity="warning")
    return MetricTable(ni.metric, t)


def parent_equity(equity: MetricTable, nci: MetricTable, dq: DQLog) -> MetricTable:
    """Where only total equity (including NCI) is tagged, subtract NCI tagged at the same instant."""
    t = equity.table
    total_tags = {"StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest", "Equity"}
    if t.empty or not t["tag"].isin(total_tags).any():
        return equity
    t = t.copy()
    adjusted = 0
    for i in t.index[t["tag"].isin(total_tags)]:
        row = nci.get(None, t.at[i, "end"], tol=0)
        if row is not None and row["value"]:
            t.at[i, "value"] = float(t.at[i, "value"]) - float(row["value"])
            t.at[i, "first_value"] = float(t.at[i, "first_value"]) - float(row["value"])
            t.at[i, "tag"] = f"derived: {t.at[i, 'tag']} - {row['tag']}"
            adjusted += 1
    if adjusted:
        dq.add("derived_value", equity.metric.label, f"{adjusted} instant(s): parent equity = total equity - noncontrolling interest (both tagged at the same date).", severity="warning")
    return MetricTable(equity.metric, t)


CONTINUING_OCF_TAG = "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"
_REPLACE_COLUMNS = (
    "value", "value_reported", "unit", "accn", "filed", "form", "first_value", "first_accn", "first_filed", "first_form",
    "n_filings", "restated", "tag", "ignored_quarterly_report_value",
)


def continuing_operations_ocf(ocf: MetricTable, cont: MetricTable, dq: DQLog, disc: MetricTable | None = None) -> MetricTable:
    """Use continuing-operations OCF for every period where it is tagged alongside the total.

    The total (NetCashProvidedByUsedInOperatingActivities) includes discontinued operations' cash flows.
    Filers with discontinued operations often tag only the continuing line in some filings and both in
    others, so preferring the total mixes the two across periods and inside YTD differencing (e.g.
    Q2 = 6M total - 3M continuing), which puts spurious steps into the TTM series. Where the continuing
    line is not tagged but discontinued-operations OCF is, continuing = total - discontinued.
    """
    t = ocf.table
    c = cont.table
    d = disc.table if disc is not None else pd.DataFrame()
    if t.empty or (c.empty and d.empty):
        return ocf
    lookup = {(s, e): row for s, e, row in zip(c["start"], c["end"], c.to_dict("records"))} if not c.empty else {}
    disc_lookup = {(s, e): row for s, e, row in zip(d["start"], d["end"], d.to_dict("records"))} if not d.empty else {}
    t = t.copy()
    replaced, ends = 0, []
    derived, dends = 0, []
    for i in t.index:
        if t.at[i, "tag"] == CONTINUING_OCF_TAG:
            continue
        row = lookup.get((t.at[i, "start"], t.at[i, "end"]))
        if row is None or row.get("taxonomy") != t.at[i, "taxonomy"]:
            drow = disc_lookup.get((t.at[i, "start"], t.at[i, "end"]))
            if t.at[i, "tag"] == "NetCashProvidedByUsedInOperatingActivities" and drow is not None and drow.get("taxonomy") == t.at[i, "taxonomy"] and float(drow["value"]):
                t.at[i, "value"] = float(t.at[i, "value"]) - float(drow["value"])
                t.at[i, "first_value"] = float(t.at[i, "first_value"]) - float(drow["value"])
                t.at[i, "tag"] = f"derived: NetCashProvidedByUsedInOperatingActivities - {drow['tag']}"
                derived += 1
                dends.append(t.at[i, "end"])
            continue
        differs = abs(float(row["value"]) - float(t.at[i, "value"])) > RESTATEMENT_TOL * max(abs(float(row["value"])), abs(float(t.at[i, "value"])), 1.0)
        for col in _REPLACE_COLUMNS:
            if col in t.columns and col in row:
                t.at[i, col] = row[col]
        if differs:
            replaced += 1
            ends.append(t.at[i, "end"])
    if replaced:
        dq.add(
            "fallback_tag",
            ocf.metric.label,
            f"{replaced} period(s) ending {min(ends).date()} to {max(ends).date()}: {CONTINUING_OCF_TAG} used instead of the total "
            "NetCashProvidedByUsedInOperatingActivities, which includes discontinued operations (both tagged for the same period).",
            severity="warning",
        )
    if derived:
        dq.add(
            "derived_value",
            ocf.metric.label,
            f"{derived} period(s) ending {min(dends).date()} to {max(dends).date()}: continuing-operations OCF = NetCashProvidedByUsedInOperatingActivities - "
            "CashProvidedByUsedInOperatingActivitiesDiscontinuedOperations (both tagged for the same period; no continuing-operations tag).",
            severity="warning",
        )
    return MetricTable(ocf.metric, t)


def lessor_revenue(revenue: MetricTable, lease: MetricTable, dq: DQLog) -> MetricTable:
    """Total revenue for lessors that tag contract revenue and lease income separately but no total."""
    t = revenue.table
    if t.empty or lease.table.empty:
        return revenue
    t = t.copy()
    added = 0
    for i in t.index[t["tag"].astype(str).str.startswith("RevenueFromContractWithCustomer")]:
        row = lease.get(t.at[i, "start"], t.at[i, "end"], tol=0)
        # Only when lease income exceeds contract revenue can contract revenue not already be the total
        # (many lessors tag total revenue, lease income included, under the contract-revenue tag).
        if row is not None and row["value"] and float(row["value"]) > float(t.at[i, "value"]):
            t.at[i, "value"] = float(t.at[i, "value"]) + float(row["value"])
            t.at[i, "first_value"] = float(t.at[i, "first_value"]) + float(row["value"])
            t.at[i, "tag"] = f"derived: {t.at[i, 'tag']} + {row['tag']}"
            added += 1
    if added:
        dq.add(
            "derived_value",
            revenue.metric.label,
            f"{added} period(s): total revenue not tagged and lease income exceeds contract revenue; revenue = contract revenue + operating lease income for the same period (lessor presentation).",
            severity="warning",
        )
    return MetricTable(revenue.metric, t)


def capex_components(capex: MetricTable, components: dict[str, MetricTable], dq: DQLog) -> MetricTable:
    """Capex for companies that split PP&E purchases over several cash-flow lines.

    `components` maps each tag in CAPEX_COMPONENT_TAGS to its own single-tag table. When the line chosen
    for a period is a component, the other component lines of the exact same period are added. When it
    is a total tag but construction expenditures of the same period exceed it, the "total" tag holds a
    smaller separate line (e.g. a utility's asset acquisitions) and the component lines are used instead.
    """
    t = capex.table
    if t.empty or not components:
        return capex
    t = t.copy()
    summed, replaced = 0, 0
    construction = "PaymentsForConstructionInProcess"
    for i in t.index:
        tag = t.at[i, "tag"]
        if t.at[i, "taxonomy"] != US or (tag not in CAPEX_COMPONENT_TAGS and tag not in CAPEX_TOTAL_TAGS):
            continue
        start, end = t.at[i, "start"], t.at[i, "end"]
        parts = {}
        for ctag, table in components.items():
            row = table.get(start, end, tol=0)
            if row is not None and row["value"]:
                parts[ctag] = float(row["value"])
        if tag in CAPEX_COMPONENT_TAGS:
            extra = {k: v for k, v in parts.items() if k != tag}
            if not extra:
                continue
            t.at[i, "value"] = float(t.at[i, "value"]) + sum(extra.values())
            t.at[i, "first_value"] = float(t.at[i, "first_value"]) + sum(extra.values())
            t.at[i, "tag"] = "derived: " + " + ".join([tag, *extra])
            summed += 1
        elif construction in parts and parts[construction] > float(t.at[i, "value"]):
            t.at[i, "value"] = sum(parts.values())
            t.at[i, "first_value"] = sum(parts.values())
            t.at[i, "tag"] = "derived: " + " + ".join(parts) + f" (instead of {tag})"
            replaced += 1
    if summed:
        dq.add(
            "derived_value",
            capex.metric.label,
            f"{summed} period(s): PP&E purchases are split over several cash-flow lines ({', '.join(CAPEX_COMPONENT_TAGS)}); "
            "capex = sum of the lines tagged for the same period.",
            severity="warning",
        )
    if replaced:
        dq.add(
            "derived_value",
            capex.metric.label,
            f"{replaced} period(s): construction expenditures (PaymentsForConstructionInProcess) exceed the value tagged as total PP&E purchases, "
            "so that tag holds a smaller separate line; capex = the construction/PP&E component lines of the same period.",
            severity="warning",
        )
    return MetricTable(capex.metric, t)

# ============================== valuation/periods.py ==============================
"""Fiscal calendar, quarterly values and TTM construction.

Periods are identified by start/end dates and duration only (never by the
fy/fp fields). Durations: annual 350-380 days, quarter 80-100 days, 6M YTD
170-200, 9M YTD 260-290. Anything else is a stub or transition period and is
kept out of trend math.
"""


from collections import Counter
from dataclasses import dataclass, field

import numpy as np
import pandas as pd


TOL = pd.Timedelta(days=3)
TTM_DISAGREE_TOL = 0.001
QUARTER_MISMATCH_TOL = 0.005


def classify_days(days: float) -> str:
    if pd.isna(days):
        return "instant"
    if 350 <= days <= 380:
        return "annual"
    if 80 <= days <= 100:
        return "quarter"
    if 170 <= days <= 200:
        return "ytd6"
    if 260 <= days <= 290:
        return "ytd9"
    if 101 <= days <= 125:
        return "long_quarter"  # e.g. a 16-week first quarter; valid only as Q1 starting at the fiscal-year start
    return "other"


@dataclass
class FiscalYear:
    index: int
    start: pd.Timestamp
    end: pd.Timestamp | None
    quarter_ends: dict[int, pd.Timestamp] = field(default_factory=dict)
    consecutive_with_prior: bool = False

    @property
    def days(self) -> int | None:
        return None if self.end is None else int((self.end - self.start).days + 1)

    @property
    def week_note(self) -> str:
        if self.days == 364:
            return "52-week"
        if self.days == 371:
            return "53-week"
        return ""

    @property
    def label(self) -> str:
        return f"FY{self.end.year}" if self.end is not None else f"FY in progress (from {self.start.date()})"


@dataclass(frozen=True)
class Quarter:
    fy: int
    k: int
    start: pd.Timestamp | None  # None when the preceding quarter end is unknown (e.g. annual-only 20-F reporting)
    end: pd.Timestamp

    @property
    def label(self) -> str:
        return f"Q{self.k} ending {self.end.date()}"


@dataclass
class Calendar:
    years: list[FiscalYear]
    quarters: list[Quarter]

    def quarter(self, fy: int, k: int) -> Quarter | None:
        for q in self.quarters:
            if q.fy == fy and q.k == k:
                return q
        return None

    def previous_quarters(self, q: Quarter, n: int) -> list[Quarter] | None:
        """The n consecutive quarters ending at q (inclusive), or None if any is missing."""
        out = [q]
        while len(out) < n:
            prior = self._preceding(out[0])
            if prior is None:
                return None
            out.insert(0, prior)
        return out

    def _preceding(self, q: Quarter) -> Quarter | None:
        """The quarter that ends the day before q starts.

        Within a fiscal year that is Q(k-1); across consecutive years it is the prior year's Q4. When the
        prior year is provisional (a transition period after a fiscal-year change, or a year without an
        annual value) the chain continues through any calendar quarter ending the day before q starts.
        """
        if q.k > 1:
            return self.quarter(q.fy, q.k - 1)
        if self.years[q.fy].consecutive_with_prior:
            prior = self.quarter(q.fy - 1, 4)
            if prior is not None:
                return prior
        if q.start is None:
            return None
        target = q.start - pd.Timedelta(days=1)
        hits = [p for p in self.quarters if p.end < q.end and abs(p.end - target) <= TOL]
        return hits[-1] if hits else None


def _cluster_dates(dates: list[pd.Timestamp]) -> list[pd.Timestamp]:
    """Merge dates within 3 days; the most frequent member represents the cluster."""
    if not dates:
        return []
    ordered = sorted(dates)
    clusters: list[list[pd.Timestamp]] = [[ordered[0]]]
    for d in ordered[1:]:
        if d - clusters[-1][-1] <= TOL:
            clusters[-1].append(d)
        else:
            clusters.append([d])
    return [Counter(c).most_common(1)[0][0] for c in clusters]


def _mode_end(periods: pd.DataFrame, start: pd.Timestamp, kinds: tuple[str, ...]) -> pd.Timestamp | None:
    hits = periods[(periods["cls"].isin(kinds)) & ((periods["start"] - start).abs() <= TOL)]
    if hits.empty:
        return None
    return Counter(hits["end"]).most_common(1)[0][0]


def _following_quarter_end(periods: pd.DataFrame, prior_end: pd.Timestamp) -> pd.Timestamp | None:
    nxt = prior_end + pd.Timedelta(days=1)
    hits = periods[(periods["cls"] == "quarter") & ((periods["start"] - nxt).abs() <= TOL)]
    if hits.empty:
        return None
    return Counter(hits["end"]).most_common(1)[0][0]


def build_calendar(tables: list[MetricTable], dq: DQLog, as_of: pd.Timestamp) -> Calendar:
    frames = [t.table[["start", "end", "days"]] for t in tables if not t.table.empty]
    if not frames:
        return Calendar([], [])
    periods = pd.concat(frames, ignore_index=True).drop_duplicates()
    periods = periods[periods["end"] <= as_of].copy()
    periods["cls"] = periods["days"].map(classify_days)

    starts = list(periods.loc[periods["cls"].isin(["annual", "ytd6", "ytd9"]), "start"])
    starts += [e + pd.Timedelta(days=1) for e in periods.loc[periods["cls"] == "annual", "end"]]
    fy_starts = _cluster_dates(starts)

    years: list[FiscalYear] = []
    for start in fy_starts:
        end = _mode_end(periods, start, ("annual",))
        q1 = _mode_end(periods, start, ("quarter",)) or _mode_end(periods, start, ("long_quarter",))
        q2 = _mode_end(periods, start, ("ytd6",)) or (q1 and _following_quarter_end(periods, q1))
        q3 = _mode_end(periods, start, ("ytd9",)) or (q2 and _following_quarter_end(periods, q2))
        # Q4 is only ever the annual period's end; a trailing 3M period alone does not define a year end.
        ends ={k: v for k, v in {1: q1, 2: q2, 3: q3, 4: end}.items() if v is not None}
        if not ends:
            continue
        # Discard quarter ends that are out of order.
        valid, last = {}, start
        for k in sorted(ends):
            if ends[k] > last:
                valid[k] = ends[k]
                last = ends[k]
        years.append(FiscalYear(len(years), start, end, valid))

    # Drop provisional years that overlap a real fiscal year (e.g. E+1 of a changed fiscal year). A boundary
    # that differs by no more than the date tolerance is the same boundary, not an overlap: a company moving
    # from month-end to 52/53-week years restates the prior year to end a day or two later (e.g. 2016-10-30
    # instead of 2016-10-31) while the next year starts on 2016-10-31.
    cleaned: list[FiscalYear] = []
    for fy in years:
        if cleaned and cleaned[-1].end is not None and fy.start + TOL <= cleaned[-1].end:
            continue
        if cleaned and cleaned[-1].end is None:
            # A provisional year that precedes the next year (a transition period, or a year without an annual
            # value) ends where the next year starts; quarter ends it borrowed from beyond that boundary belong
            # to the next year. A provisional year with no quarter before that boundary is a parallel calendar
            # (e.g. a predecessor's fiscal quarters in the same filings) and is left as it is.
            prev = cleaned[-1]
            before = {k: v for k, v in prev.quarter_ends.items() if v <= fy.start - pd.Timedelta(days=1) + TOL}
            if before:
                prev.quarter_ends = before
        cleaned.append(fy)
    years = cleaned
    for i, fy in enumerate(years):
        fy.index = i
        if i > 0 and years[i - 1].end is not None:
            fy.consecutive_with_prior = abs(fy.start - (years[i - 1].end + pd.Timedelta(days=1))) <= TOL

    quarters: list[Quarter] = []
    for fy in years:
        for k in (1, 2, 3, 4):
            if k not in fy.quarter_ends:
                continue  # a missing quarter end breaks quarter chains; the fiscal-year end still stands on its own
            if k == 1:
                q_start = fy.start
            elif (k - 1) in fy.quarter_ends:
                q_start = fy.quarter_ends[k - 1] + pd.Timedelta(days=1)
            else:
                q_start = None
            quarters.append(Quarter(fy.index, k, q_start, fy.quarter_ends[k]))

    _log_calendar(years, periods, dq)
    return Calendar(years, quarters)


def _log_calendar(years: list[FiscalYear], periods: pd.DataFrame, dq: DQLog) -> None:
    for fy in years:
        if fy.week_note == "53-week":
            dq.add("period_label", "53-week fiscal year", f"{fy.label} runs {fy.start.date()} to {fy.end.date()} (371 days). Comparisons with 52-week years are not adjusted.", period=fy.end)
    for i in range(1, len(years)):
        prev, cur = years[i - 1], years[i]
        if prev.end is None:
            continue
        if not cur.consecutive_with_prior:
            dq.add(
                "fiscal_calendar",
                "fiscal year gap or change",
                f"Fiscal year starting {cur.start.date()} does not follow the year ending {prev.end.date()}; TTM chains break here.",
                period=cur.start,
                severity="warning",
            )
        if cur.end is not None and abs((cur.end.month * 31 + cur.end.day) - (prev.end.month * 31 + prev.end.day)) > 7 and abs(
            (cur.end.month * 31 + cur.end.day) - (prev.end.month * 31 + prev.end.day)
        ) < 365 - 7:
            dq.add("fiscal_calendar", "fiscal year end change", f"Fiscal year end moved from {prev.end.strftime('%m-%d')} to {cur.end.strftime('%m-%d')}.", period=cur.end, severity="warning")
    long_q1 = [fy for fy in years if 1 in fy.quarter_ends and (fy.quarter_ends[1] - fy.start).days + 1 > 100]
    if long_q1:
        days = sorted({(fy.quarter_ends[1] - fy.start).days + 1 for fy in long_q1})
        dq.add(
            "period_label",
            "non-standard first quarter",
            f"{len(long_q1)} fiscal year(s) have a first quarter of {', '.join(map(str, days))} days (e.g. a 16-week Q1). It is used only inside YTD "
            "differencing and TTM sums, which still cover full 52/53-week years; it is never trended on its own.",
        )
    starts_of_years = {fy.start for fy in years}
    other = periods[(periods["cls"] == "other") | ((periods["cls"] == "long_quarter") & ~periods["start"].isin(starts_of_years))].drop_duplicates(["start", "end"])
    for _, row in other.iterrows():
        dq.add(
            "period_label",
            "stub/transition period",
            f"Period {row['start'].date()} to {row['end'].date()} ({int(row['days'])} days) is neither annual nor quarterly; excluded from trend math.",
            period=row["end"],
        )


# ---------------------------------------------------------------- quarterly values

def quarterly_values(metric: MetricTable, cal: Calendar, dq: DQLog) -> pd.DataFrame:
    rows = []
    additive = metric.metric.additive
    for q in cal.quarters:
        fy = cal.years[q.fy]
        reported = metric.get(q.start, q.end) if q.start is not None else None
        if reported is not None and not (classify_days(reported["days"]) == "quarter" or (q.k == 1 and classify_days(reported["days"]) == "long_quarter")):
            reported = None
        ytd = metric.get(fy.start, q.end)
        prev_ytd = None
        if q.k >= 2:
            prev_q = cal.quarter(q.fy, q.k - 1)
            prev_ytd = metric.get(fy.start, prev_q.end) if prev_q is not None else None
        derived = None
        if additive and q.k >= 2 and ytd is not None and prev_ytd is not None:
            derived = float(ytd["value"]) - float(prev_ytd["value"])
        value, source = None, ""
        if reported is not None:
            value, source = float(reported["value"]), "reported"
            if derived is not None and _rel_diff(value, derived) > QUARTER_MISMATCH_TOL:
                dq.add(
                    "quarter_mismatch",
                    metric.metric.label,
                    f"{q.label}: reported 3M {value:,.4g} vs YTD difference {derived:,.4g}; reported value used.",
                    period=q.end,
                    filing=reported["accn"],
                )
        elif derived is not None:
            value, source = derived, "derived"
            label = {2: "Q2 = 6M - 3M", 3: "Q3 = 9M - 6M", 4: "Q4 = FY - 9M"}[q.k]
            dq.add(
                "derived_quarter",
                metric.metric.label,
                f"{q.label}: {label} = {float(ytd['value']):,.6g} ({ytd['tag']}, {ytd['accn']}) - {float(prev_ytd['value']):,.6g} "
                f"({prev_ytd['tag']}, {prev_ytd['accn']}) = {derived:,.6g}"
                + ("; components use different tags" if ytd["tag"] != prev_ytd["tag"] else ""),
                period=q.end,
                filing=ytd["accn"],
            )
        rows.append(
            {
                "fy": q.fy,
                "k": q.k,
                "q_start": q.start,
                "q_end": q.end,
                "value": value,
                "source": source,
                "ytd": None if ytd is None else float(ytd["value"]),
                "ytd_accn": None if ytd is None else ytd["accn"],
                "accn": None if reported is None else reported["accn"],
            }
        )
    if metric.metric.key in ("revenue", "gross_profit"):
        _exclude_misdated_quarters(rows, metric.metric.label, dq)
    frame = pd.DataFrame(rows)
    frame.attrs["index"] = {(r["fy"], r["k"]): r for r in rows}
    return frame


MISDATED_RATIO = 2.5


def _exclude_misdated_quarters(rows: list[dict], label: str, dq: DQLog) -> None:
    """Drop a quarter far larger than both neighbours (e.g. a full-year value tagged with quarter dates)."""
    values = [r["value"] for r in rows]
    for i in range(1, len(rows) - 1):
        v, prev, nxt = values[i], values[i - 1], values[i + 1]
        if not all(isinstance(x, float) and x > 0 for x in (v, prev, nxt)):
            continue
        if v > MISDATED_RATIO * max(prev, nxt):
            dq.add(
                "quarter_mismatch",
                label,
                f"Q{rows[i]['k']} ending {rows[i]['q_end'].date()}: {v:,.6g} is more than {MISDATED_RATIO}x both neighbouring quarters "
                f"({prev:,.6g}, {nxt:,.6g}); treated as a mis-dated fact (often a full-year value tagged with quarter dates) and excluded.",
                period=rows[i]["q_end"],
                filing=rows[i]["accn"] or rows[i]["ytd_accn"] or "",
                severity="warning",
            )
            rows[i]["value"], rows[i]["source"] = None, "excluded"
    return None


def _qv_row(qv: pd.DataFrame, fy: int, k: int) -> dict | None:
    index = qv.attrs.get("index")
    if index is None:
        index = {(r["fy"], r["k"]): r for r in qv.to_dict("records")}
        qv.attrs["index"] = index
    return index.get((fy, k))


def _rel_diff(a: float, b: float) -> float:
    scale = max(abs(a), abs(b))
    return 0.0 if scale == 0 else abs(a - b) / scale


# ---------------------------------------------------------------- TTM

@dataclass
class TTMValue:
    value: float | None
    method: str  # "FY", "FY+YTD-PYTD", "4Q" or ""
    components: list[float | None]
    component_labels: list[str]
    alt_value: float | None
    alt_method: str
    reason: str  # why the value is blank
    sources: str


def ttm_flow(metric: MetricTable, qv: pd.DataFrame, cal: Calendar, q: Quarter, dq: DQLog | None = None) -> TTMValue:
    fy = cal.years[q.fy]
    label = metric.metric.label
    four = cal.previous_quarters(q, 4)
    quarter_sum, quarter_parts, quarter_labels = None, [], []
    if four is not None:
        for item in four:
            match = _qv_row(qv, item.fy, item.k)
            v = None if match is None or match["value"] is None or pd.isna(match["value"]) else float(match["value"])
            quarter_parts.append(v)
            quarter_labels.append(f"Q{item.k} {item.end.date()}")
        if all(v is not None for v in quarter_parts):
            quarter_sum = float(sum(quarter_parts))

    primary, method, parts, labels, sources, reason = None, "", [], [], "", ""
    if q.k == 4:
        row = metric.get(fy.start, fy.end)
        if row is not None:
            primary, method = float(row["value"]), "FY"
            parts, labels = [primary], [f"FY {fy.start.date()}..{fy.end.date()}"]
            sources = f"FY {row['accn']}"
        else:
            reason = f"{label}: no annual value for {fy.label}"
    else:
        prior = cal.years[q.fy - 1] if q.fy > 0 else None
        if prior is None or not fy.consecutive_with_prior or prior.end is None:
            reason = f"{label}: no consecutive prior fiscal year"
        else:
            fy_prev = metric.get(prior.start, prior.end)
            ytd = metric.get(fy.start, q.end)
            prior_q = cal.quarter(prior.index, q.k)
            pytd = metric.get(prior.start, prior_q.end) if prior_q is not None else None
            missing = [n for n, r in (("prior FY", fy_prev), ("current YTD", ytd), ("prior-year YTD", pytd)) if r is None]
            if not missing:
                parts = [float(fy_prev["value"]), float(ytd["value"]), float(pytd["value"])]
                primary, method = parts[0] + parts[1] - parts[2], "FY+YTD-PYTD"
                labels = [f"FY {prior.start.date()}..{prior.end.date()}", f"YTD {fy.start.date()}..{q.end.date()}", f"PYTD {prior.start.date()}..{prior_q.end.date()}"]
                sources = f"FY {fy_prev['accn']}; YTD {ytd['accn']}; PYTD {pytd['accn']}"
            else:
                reason = f"{label}: missing {', '.join(missing)}"

    alt, alt_method = None, ""
    if primary is not None:
        if method != "4Q" and quarter_sum is not None:
            alt, alt_method = quarter_sum, "4Q"
    elif quarter_sum is not None:
        primary, method, parts, labels, reason = quarter_sum, "4Q", quarter_parts, quarter_labels, ""
        sources = "sum of four quarters"
    if primary is not None and alt is not None and dq is not None and _rel_diff(primary, alt) > TTM_DISAGREE_TOL:
        dq.add(
            "ttm_method",
            label,
            f"TTM ending {q.end.date()}: {method} = {primary:,.6g} vs sum of four quarters = {alt:,.6g} ({_rel_diff(primary, alt):.2%} apart). {method} used.",
            period=q.end,
            severity="warning",
        )
    return TTMValue(primary, method, parts, labels, alt, alt_method, reason, sources)


def ttm_shares(shares: MetricTable, qv: pd.DataFrame, cal: Calendar, q: Quarter) -> TTMValue:
    four = cal.previous_quarters(q, 4)
    if four is not None:
        vals = []
        for item in four:
            match = _qv_row(qv, item.fy, item.k)
            vals.append(float(match["value"]) if match is not None and match["source"] == "reported" else None)
        if all(v is not None for v in vals):
            return TTMValue(float(np.mean(vals)), "avg4Q", vals, [f"Q{i.k} {i.end.date()}" for i in four], None, "", "", "four reported quarterly counts")
    fy_index = q.fy if q.k == 4 else q.fy - 1
    # After a transition period (a provisional year without an annual end) the latest fiscal year is the one before it.
    while q.k != 4 and fy_index >= 0 and cal.years[fy_index].end is None:
        fy_index -= 1
    if fy_index < 0:
        return TTMValue(None, "", [], [], None, "", "no fiscal-year diluted share count", "")
    fy = cal.years[fy_index]
    if fy.end is None:
        return TTMValue(None, "", [], [], None, "", "no fiscal-year diluted share count", "")
    row = shares.get(fy.start, fy.end)
    if row is None:
        return TTMValue(None, "", [], [], None, "", f"no diluted share count for {fy.label}", "")
    return TTMValue(float(row["value"]), "FY", [float(row["value"])], [f"FY {fy.start.date()}..{fy.end.date()}"], None, "", "", f"FY count ({fy.label}, {row['accn']}): quarterly counts unavailable")


def reported_ttm_eps(eps: MetricTable, cal: Calendar, q: Quarter) -> float | None:
    """FY EPS + YTD EPS - prior-year YTD EPS, from reported values (cross-check only)."""
    fy = cal.years[q.fy]
    if q.k == 4:
        return eps.value(fy.start, fy.end)
    if q.fy == 0 or not fy.consecutive_with_prior:
        return None
    prior = cal.years[q.fy - 1]
    prior_q = cal.quarter(prior.index, q.k)
    if prior.end is None or prior_q is None:
        return None
    parts = [eps.value(prior.start, prior.end), eps.value(fy.start, q.end), eps.value(prior.start, prior_q.end)]
    if any(p is None for p in parts):
        return None
    return parts[0] + parts[1] - parts[2]

# ============================== valuation/prices.py ==============================
"""Input C: daily closes, dividends with ex-dates, and the stated adjustment basis."""


import io
import json
from dataclasses import dataclass, field
from datetime import date, timedelta
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd


STALE_TRADING_DAYS = 5


# ------------------------------------------------------------------ exchange calendar
# Used only to tell a data gap from an exchange holiday. It follows NYSE rules;
# a listing on another exchange can produce false gap entries, which are logged
# as "possible" gaps for that reason.

def _easter(year: int) -> date:
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    return date(year, month, day)


def _nth_weekday(year: int, month: int, weekday: int, n: int) -> date:
    first = date(year, month, 1)
    offset = (weekday - first.weekday()) % 7
    return first + timedelta(days=offset + 7 * (n - 1))


def _last_weekday(year: int, month: int, weekday: int) -> date:
    nxt = date(year + (month == 12), month % 12 + 1, 1)
    last = nxt - timedelta(days=1)
    return last - timedelta(days=(last.weekday() - weekday) % 7)


def _observed(d: date, saturday_to_friday: bool = True) -> date | None:
    if d.weekday() == 5:
        return d - timedelta(days=1) if saturday_to_friday else None
    if d.weekday() == 6:
        return d + timedelta(days=1)
    return d


SPECIAL_CLOSURES = {
    date(1994, 4, 27), date(2001, 9, 11), date(2001, 9, 12), date(2001, 9, 13), date(2001, 9, 14),
    date(2004, 6, 11), date(2007, 1, 2), date(2012, 10, 29), date(2012, 10, 30),
    date(2018, 12, 5), date(2025, 1, 9),
}


@lru_cache(maxsize=None)
def exchange_holidays(year: int) -> frozenset[date]:
    days = {
        _observed(date(year, 1, 1), saturday_to_friday=False),
        _nth_weekday(year, 1, 0, 3),
        _nth_weekday(year, 2, 0, 3),
        _easter(year) - timedelta(days=2),
        _last_weekday(year, 5, 0),
        _observed(date(year, 7, 4)),
        _nth_weekday(year, 9, 0, 1),
        _nth_weekday(year, 11, 3, 4),
        _observed(date(year, 12, 25)),
    }
    if year >= 2022:
        days.add(_observed(date(year, 6, 19)))
    days |= {d for d in SPECIAL_CLOSURES if d.year == year}
    days.discard(None)
    return frozenset(days)


def expected_trading_days(start: pd.Timestamp, end: pd.Timestamp) -> pd.DatetimeIndex:
    if start > end:
        return pd.DatetimeIndex([])
    days = pd.bdate_range(start, end)
    holidays = set()
    for year in range(start.year, end.year + 1):
        holidays |= exchange_holidays(year)
    return days[[d.date() not in holidays for d in days]]


# ------------------------------------------------------------------ loading

@dataclass
class PriceMeta:
    ticker: str = ""
    split_adjusted: bool | None = None
    dividend_adjusted: bool | None = None
    currency: str = ""

    @classmethod
    def from_json(cls, source) -> "PriceMeta":
        if source is None:
            return cls()
        if isinstance(source, dict):
            data = source
        elif isinstance(source, (str, Path)):
            data = json.loads(Path(source).read_text(encoding="utf-8"))
        else:
            raw = source.read()
            data = json.loads(raw.decode("utf-8") if isinstance(raw, bytes) else raw)
        return cls(
            ticker=str(data.get("ticker", "") or ""),
            split_adjusted=_as_bool(data.get("split_adjusted")),
            dividend_adjusted=_as_bool(data.get("dividend_adjusted")),
            currency=str(data.get("currency", "") or "").upper(),
        )


def _as_bool(value) -> bool | None:
    if isinstance(value, bool):
        return value
    if value is None:
        return None
    text = str(value).strip().lower()
    if text in {"yes", "y", "true", "1"}:
        return True
    if text in {"no", "n", "false", "0"}:
        return False
    return None


@dataclass
class PriceData:
    meta: PriceMeta
    close_raw: pd.Series  # valid closes as supplied, indexed by date
    invalid_days: list[pd.Timestamp]
    dividends_raw: pd.DataFrame  # ex_date, amount as supplied
    splits: pd.DataFrame  # date, ratio (optional, from Input C)
    close: pd.Series = field(default=None)  # current split basis
    dividends: pd.DataFrame = field(default=None)  # current split basis
    uncertain_days: pd.Series = field(default=None)  # True where the split basis is unknown
    missing_days: list[pd.Timestamp] = field(default_factory=list)

    @property
    def first_date(self) -> pd.Timestamp:
        return self.close_raw.index.min()

    @property
    def last_date(self) -> pd.Timestamp:
        return self.close_raw.index.max()


def _read_table(source) -> pd.DataFrame:
    if source is None:
        return pd.DataFrame()
    if isinstance(source, pd.DataFrame):
        return source.copy()
    if isinstance(source, (str, Path)):
        return pd.read_csv(source)
    raw = source.getvalue() if hasattr(source, "getvalue") else source.read()
    return pd.read_csv(io.BytesIO(raw) if isinstance(raw, bytes) else io.StringIO(raw))


def _col(frame: pd.DataFrame, *names: str) -> str | None:
    lookup = {str(c).strip().lower().replace(" ", "_"): c for c in frame.columns}
    for name in names:
        if name in lookup:
            return lookup[name]
    return None


def load_prices(prices_source, meta: PriceMeta, dividends_source=None, dq: DQLog | None = None) -> PriceData:
    frame = _read_table(prices_source)
    date_col = _col(frame, "date")
    close_col = _col(frame, "close", "closing_price", "price")
    if date_col is None or close_col is None:
        raise ValueError("Input C must have 'date' and 'close' columns.")
    frame["_date"] = pd.to_datetime(frame[date_col], errors="coerce").dt.normalize()
    frame["_close"] = pd.to_numeric(frame[close_col], errors="coerce")
    frame = frame.dropna(subset=["_date"]).sort_values("_date").drop_duplicates("_date", keep="last")
    valid = frame["_close"].notna() & (frame["_close"] > 0)
    event_cols = [c for c in (_col(frame, "dividend", "dividends", "dividend_per_share", "dps"), _col(frame, "split_ratio", "split", "stock_splits", "splits")) if c is not None]
    has_event = frame[event_cols].apply(pd.to_numeric, errors="coerce").fillna(0).ne(0).any(axis=1) if event_cols else pd.Series(False, index=frame.index)
    # A row that only carries a dividend or split event is not a missing close.
    invalid_days = list(frame.loc[~valid & ~(frame["_close"].isna() & has_event), "_date"])
    close_raw = frame.loc[valid].set_index("_date")["_close"].astype(float)
    close_raw.index.name = "date"

    div_col = _col(frame, "dividend", "dividends", "dividend_per_share", "dps")
    dividends = pd.DataFrame(columns=["ex_date", "amount"])
    if div_col is not None:
        d = frame[["_date", div_col]].copy()
        d["amount"] = pd.to_numeric(d[div_col], errors="coerce")
        d = d[d["amount"].notna() & (d["amount"] != 0)]
        dividends = pd.DataFrame({"ex_date": d["_date"].to_numpy(), "amount": d["amount"].to_numpy()})
    extra = _read_table(dividends_source)
    if not extra.empty:
        ex_col = _col(extra, "ex_date", "exdate", "ex-date", "date")
        amt_col = _col(extra, "amount", "dividend", "dividends", "dps", "value")
        if ex_col is None or amt_col is None:
            raise ValueError("Dividend table must have 'ex_date' and 'amount' columns.")
        more = pd.DataFrame(
            {"ex_date": pd.to_datetime(extra[ex_col], errors="coerce").dt.normalize(), "amount": pd.to_numeric(extra[amt_col], errors="coerce")}
        ).dropna()
        dividends = pd.concat([dividends, more], ignore_index=True).drop_duplicates(["ex_date", "amount"])
    dividends = dividends.sort_values("ex_date").reset_index(drop=True)

    split_col = _col(frame, "split_ratio", "split", "stock_splits", "splits")
    splits = pd.DataFrame(columns=["date", "ratio"])
    if split_col is not None:
        s = frame[["_date", split_col]].copy()
        s["ratio"] = pd.to_numeric(s[split_col], errors="coerce")
        s = s[s["ratio"].notna() & (s["ratio"] > 0) & (s["ratio"] != 1)]
        splits = pd.DataFrame({"date": s["_date"].to_numpy(), "ratio": s["ratio"].to_numpy()})

    # Rows without a valid close are logged by apply_basis, limited to the span the review uses.
    return PriceData(meta, close_raw, invalid_days, dividends, splits)


def apply_basis(prices: PriceData, basis: SplitBasis, dq: DQLog, check_from: pd.Timestamp | None = None) -> None:
    """Put closes and dividends on the as-of split basis; find data gaps."""
    close = prices.close_raw.copy()
    dividends = prices.dividends_raw.copy()
    gap_start = max(prices.first_date, check_from) if check_from is not None else prices.first_date
    uncertain = pd.Series(False, index=close.index)
    if prices.meta.split_adjusted is False and basis.events:
        factors = pd.Series([basis.price_factor(d) for d in close.index], index=close.index)
        close = close / factors
        if not dividends.empty:
            dividends["amount"] = dividends["amount"] / dividends["ex_date"].map(basis.price_factor)
        uncertain = pd.Series([basis.uncertain(d) is not None for d in close.index], index=close.index)
        for event in basis.events:
            if not event.price_located and event.window_start is not None:
                dq.add(
                    "price_basis",
                    f"split {event.label} ex-date not found",
                    f"No close-to-close move near 1/{event.factor:g} between {event.window_start.date()} and {event.window_end.date()} in Input C. "
                    "Price ratios inside that window are left blank because the price basis is uncertain.",
                    period=event.window_end,
                    severity="warning",
                )
        dq.add("price_basis", "split adjustment applied", f"Input C stated not split-adjusted; closes before each ex-date divided by the cumulative factor ({basis.describe()}).")
    elif prices.meta.split_adjusted is True:
        dq.add("price_basis", "split basis", "Input C stated split-adjusted; closes used as supplied and assumed adjusted through the as-of date. Dividends assumed on the same basis.")
        _check_unexplained_jumps(close, dq, stated_adjusted=True)
    if prices.meta.split_adjusted is False:
        _check_unexplained_jumps(close, dq, stated_adjusted=False)
    if prices.meta.dividend_adjusted:
        dq.add(
            "price_basis",
            "dividend-adjusted prices",
            "Input C stated dividend-adjusted: historical closes are reduced for later dividends, so historical price ratios and yields "
            "are not on traded prices. Not reversed (no method supplied).",
            severity="warning",
        )
    for day in prices.invalid_days:
        if day >= gap_start:
            dq.add("price_gap", "missing close", f"Input C row for {day.date()} has no valid close; day skipped (no carry-forward).", period=day, severity="warning")
    prices.close = close
    prices.dividends = dividends
    prices.uncertain_days = uncertain

    expected = expected_trading_days(gap_start, prices.last_date)
    present = set(prices.close_raw.index)
    invalid = set(prices.invalid_days)
    missing = [d for d in expected if d not in present and d not in invalid]
    prices.missing_days = missing + sorted(d for d in invalid if d >= gap_start)
    for run in _runs(missing):
        dq.add(
            "price_gap",
            "possible missing trading day(s)",
            f"No Input C row for {run[0].date()}" + (f" to {run[-1].date()} ({len(run)} days)" if len(run) > 1 else "")
            + " (exchange calendar assumption: NYSE). Days skipped; no price carried across.",
            period=run[0],
            severity="warning",
        )


def _check_unexplained_jumps(close: pd.Series, dq: DQLog, stated_adjusted: bool) -> None:
    ratio = (close / close.shift(1)).dropna()
    big = ratio[(ratio < 1 / 1.8) | (ratio > 1.8)]
    for day, r in big.items():
        dq.add(
            "price_basis",
            "large one-day price move",
            f"Close changed by a factor of {r:.3f} on {day.date()}"
            + (" although Input C is stated split-adjusted" if stated_adjusted else " after split adjustment")
            + "; possible unrecorded split or data error. Not adjusted.",
            period=day,
            severity="warning",
        )


def _runs(days: list[pd.Timestamp]) -> list[list[pd.Timestamp]]:
    runs: list[list[pd.Timestamp]] = []
    for d in days:
        if runs and (d - runs[-1][-1]).days <= 4:
            runs[-1].append(d)
        else:
            runs.append([d])
    return runs


# ------------------------------------------------------------------ lookups

@dataclass
class PriceLookup:
    date: pd.Timestamp | None
    close: float | None
    raw_close: float | None
    factor: float
    stale_days: int
    reason: str


def price_on_or_before(prices: PriceData, when: pd.Timestamp, allow_stale: bool) -> PriceLookup:
    """Close for a valuation date (allow_stale=False) or the as-of snapshot (allow_stale=True)."""
    if prices is None or prices.close is None or prices.close.empty:
        return PriceLookup(None, None, None, 1.0, 0, "no price data")
    if when < prices.first_date:
        return PriceLookup(None, None, None, 1.0, 0, f"before price coverage (starts {prices.first_date.date()})")
    sub = prices.close[prices.close.index <= when]
    day = sub.index.max()
    gap = expected_trading_days(day + pd.Timedelta(days=1), when)
    stale = len(gap)
    raw = float(prices.close_raw.loc[day])
    adj = float(prices.close.loc[day])
    factor = raw / adj if adj else 1.0
    if bool(prices.uncertain_days.get(day, False)):
        return PriceLookup(day, None, raw, factor, stale, "split basis uncertain on this date")
    if allow_stale:
        reason = f"stale: {stale} trading days old" if stale > STALE_TRADING_DAYS else ""
        return PriceLookup(day, adj, raw, factor, stale, reason)
    if stale > 0:
        where = "after price coverage ends" if when > prices.last_date else "no close on an expected trading day"
        return PriceLookup(None, None, None, 1.0, stale, f"{where} ({gap[0].date()}); price not carried forward")
    return PriceLookup(day, adj, raw, factor, 0, "")


def ttm_dividends(prices: PriceData | None, end: pd.Timestamp) -> tuple[float | None, int, str]:
    """Sum of dividends with ex-dates in (end - 1 year, end]."""
    if prices is None or prices.close is None:
        return None, 0, "no price data"
    start = end - pd.DateOffset(years=1)
    if start < prices.first_date - pd.Timedelta(days=7):
        return None, 0, f"dividend history in Input C starts {prices.first_date.date()}; trailing year not covered"
    if end > prices.last_date + pd.Timedelta(days=7):
        return None, 0, f"dividend history in Input C ends {prices.last_date.date()}"
    d = prices.dividends
    mask = (d["ex_date"] > start) & (d["ex_date"] <= end)
    return float(d.loc[mask, "amount"].sum()), int(mask.sum()), ""

# ============================== valuation/ratios.py ==============================
"""Ratio definitions shared by the snapshot, the per-filing table and the daily series."""


import math
from dataclasses import dataclass

NM = "NM"
EXTREME = 100.0


@dataclass(frozen=True)
class RatioDef:
    key: str
    label: str
    numerator: str
    denominator: str
    kind: str  # "multiple", "yield", "ratio", "value"
    formula: str
    shares: str
    daily: bool = True


RATIOS: list[RatioDef] = [
    RatioDef("pe", "P/E", "price", "eps", "multiple", "Price / TTM diluted EPS", "TTM average diluted shares (in EPS)"),
    RatioDef("earnings_yield", "Earnings yield", "eps", "price", "yield", "TTM diluted EPS / Price", "TTM average diluted shares (in EPS)"),
    RatioDef("ps", "P/S", "mcap", "revenue", "multiple", "Market cap / TTM revenue", "cover-page shares outstanding (market cap)"),
    RatioDef("pgp", "P/GP", "mcap", "gross_profit", "multiple", "Market cap / TTM gross profit", "cover-page shares outstanding (market cap)"),
    RatioDef("pb", "P/B", "mcap", "equity", "multiple", "Market cap / parent equity", "cover-page shares outstanding (market cap)"),
    RatioDef("ptbv", "P/TBV", "mcap", "tbv", "multiple", "Market cap / (parent equity - goodwill - intangibles)", "cover-page shares outstanding (market cap)"),
    RatioDef("pcfo", "P/CFO", "mcap", "ocf", "multiple", "Market cap / TTM operating cash flow", "cover-page shares outstanding (market cap)"),
    RatioDef("pfcf", "P/FCF", "mcap", "fcf", "multiple", "Market cap / TTM free cash flow", "cover-page shares outstanding (market cap)"),
    RatioDef("fcf_yield", "FCF yield", "fcf", "mcap", "yield", "TTM free cash flow / Market cap", "cover-page shares outstanding (market cap)"),
    RatioDef("div_yield", "Dividend yield", "dps", "price", "yield", "TTM dividends per share (Input C ex-dates) / Price", "per share as supplied in Input C"),
    RatioDef("payout", "Payout ratio", "dps", "eps", "ratio", "TTM dividends per share / TTM diluted EPS", "Input C DPS; TTM average diluted shares (in EPS)", daily=False),
    RatioDef("ev_revenue", "EV/Revenue", "ev", "revenue", "multiple", "EV / TTM revenue", "cover-page shares outstanding (in EV)"),
    RatioDef("ev_gp", "EV/Gross profit", "ev", "gross_profit", "multiple", "EV / TTM gross profit", "cover-page shares outstanding (in EV)"),
    RatioDef("ev_ebit", "EV/EBIT", "ev", "operating_income", "multiple", "EV / TTM operating income", "cover-page shares outstanding (in EV)"),
    RatioDef("ev_ebitda", "EV/EBITDA", "ev", "ebitda", "multiple", "EV / (TTM operating income + TTM D&A)", "cover-page shares outstanding (in EV)"),
    RatioDef("ev_cfo", "EV/CFO", "ev", "ocf", "multiple", "EV / TTM operating cash flow", "cover-page shares outstanding (in EV)"),
    RatioDef("ev_fcf", "EV/FCF", "ev", "fcf", "multiple", "EV / TTM free cash flow", "cover-page shares outstanding (in EV)"),
    RatioDef("peg", "PEG", "pe", "cagr_pct", "multiple", "P/E / (5-year TTM EPS CAGR x 100), only when CAGR > 0", "TTM average diluted shares (in EPS)"),
    RatioDef("graham", "Graham number", "eps", "bvps", "value", "sqrt(22.5 x TTM EPS x BVPS), only when both > 0", "EPS: TTM average diluted; BVPS: cover-page shares", daily=False),
]
RATIO_BY_KEY = {r.key: r for r in RATIOS}
DAILY_RATIOS = [r for r in RATIOS if r.daily]


def _num(x) -> bool:
    return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)


def compute(ratio: RatioDef, inputs: dict) -> tuple[object, str]:
    """Return (value | 'NM' | None, reason). None means missing."""
    num = inputs.get(ratio.numerator)
    den = inputs.get(ratio.denominator)
    if ratio.key == "graham":
        if not (_num(num) and _num(den)):
            return None, "missing " + " and ".join(n for n, v in (("TTM EPS", num), ("BVPS", den)) if not _num(v))
        if num <= 0 or den <= 0:
            return NM, "EPS or BVPS not positive"
        return math.sqrt(22.5 * num * den), ""
    if ratio.key == "peg":
        cagr = inputs.get("cagr")
        if not _num(cagr):
            return None, inputs.get("cagr_reason") or "5-year EPS CAGR unavailable"
        if cagr <= 0:
            return None, "5-year EPS CAGR not positive; PEG not computed"
        if num == NM:
            return NM, "P/E is NM"
        if not _num(num):
            return None, "P/E unavailable"
        return num / (cagr * 100.0), ""
    if not _num(num) or not _num(den):
        missing = [name for name, value in ((ratio.numerator, num), (ratio.denominator, den)) if not _num(value)]
        reasons = [str(inputs.get(f"{m}_reason")) if isinstance(inputs.get(f"{m}_reason"), str) and inputs.get(f"{m}_reason") else f"{m} missing" for m in missing]
        return None, "; ".join(reasons)
    if den <= 0:
        return NM, f"{ratio.denominator} is zero or negative"
    return num / den, ""


def compute_all(inputs: dict, keys: list[str] | None = None) -> dict[str, tuple[object, str]]:
    out: dict[str, tuple[object, str]] = {}
    for ratio in RATIOS:
        if keys is not None and ratio.key not in keys:
            continue
        if ratio.key == "peg":
            continue
        out[ratio.key] = compute(ratio, inputs)
    if keys is None or "peg" in keys:
        pe_value = out.get("pe", compute(RATIO_BY_KEY["pe"], inputs))[0]
        peg_inputs = dict(inputs, pe=pe_value, cagr_pct=None)
        out["peg"] = compute(RATIO_BY_KEY["peg"], peg_inputs)
    return out


def is_extreme(ratio: RatioDef, value) -> bool:
    return ratio.kind == "multiple" and _num(value) and abs(value) > EXTREME

# ============================== valuation/stats.py ==============================
"""RAW and ADJUSTED trend statistics over quarter-end valuation dates.

The workbook reproduces every number here with live formulas; tests check the
two agree.
"""


import math
from dataclasses import dataclass

import pandas as pd


@dataclass(frozen=True)
class SeriesDef:
    key: str  # column in the TTM table
    label: str
    basis: str  # "TTM" or "Per share"
    family: str  # groups the TTM and per-share versions of one metric
    focus: bool
    daily_ratio: str | None  # daily ratio whose valid-day count is reported
    unit: str  # "money", "per_share", "shares"


SERIES: list[SeriesDef] = [
    SeriesDef("ttm_revenue", "Revenue", "TTM", "revenue", True, "ps", "money"),
    SeriesDef("ttm_gross_profit", "Gross profit", "TTM", "gross_profit", True, "pgp", "money"),
    SeriesDef("ttm_net_income", "Earnings (NI attributable to parent)", "TTM", "earnings", True, "pe", "money"),
    SeriesDef("equity", "Parent equity", "TTM", "equity", True, "pb", "money"),
    SeriesDef("ttm_ocf", "Operating cash flow", "TTM", "ocf", True, "pcfo", "money"),
    SeriesDef("ttm_fcf", "Free cash flow", "TTM", "fcf", True, "pfcf", "money"),
    SeriesDef("revenue_ps", "Revenue per share", "Per share", "revenue", True, "ps", "per_share"),
    SeriesDef("gp_ps", "Gross profit per share", "Per share", "gross_profit", True, "pgp", "per_share"),
    SeriesDef("eps_ttm", "Diluted EPS", "Per share", "earnings", True, "pe", "per_share"),
    SeriesDef("bvps", "Book value per share", "Per share", "equity", True, "pb", "per_share"),
    SeriesDef("ocf_ps", "Operating cash flow per share", "Per share", "ocf", True, "pcfo", "per_share"),
    SeriesDef("fcf_ps", "Free cash flow per share", "Per share", "fcf", True, "pfcf", "per_share"),
    SeriesDef("ttm_da", "D&A", "TTM", "da", False, None, "money"),
    SeriesDef("ttm_ebitda", "EBITDA", "TTM", "ebitda", False, "ev_ebitda", "money"),
    SeriesDef("ttm_operating_income", "Operating income", "TTM", "operating_income", False, "ev_ebit", "money"),
    SeriesDef("ttm_dividends_paid", "Dividends paid", "TTM", "dividends", False, "div_yield", "money"),
    SeriesDef("da_ps", "D&A per share", "Per share", "da", False, None, "per_share"),
    SeriesDef("ebitda_ps", "EBITDA per share", "Per share", "ebitda", False, "ev_ebitda", "per_share"),
    SeriesDef("oi_ps", "Operating income per share", "Per share", "operating_income", False, "ev_ebit", "per_share"),
    SeriesDef("dps_ttm", "Dividends per share", "Per share", "dividends", False, "div_yield", "per_share"),
    SeriesDef("ttm_capex", "Capital expenditures", "TTM", "capex", False, None, "money"),
    SeriesDef("avg_diluted_shares", "Average diluted shares", "TTM", "shares", False, None, "shares"),
]
SERIES_BY_KEY = {s.key: s for s in SERIES}
FAMILIES = list(dict.fromkeys(s.family for s in SERIES))


def _is_num(v) -> bool:
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)


def compute_stats(dates: list, values: list) -> dict:
    """Statistics for one series. `values[i]` is a number or None for each valuation date."""
    vals = [float(v) if _is_num(v) else None for v in values]
    present = [i for i, v in enumerate(vals) if v is not None]
    out: dict = {"start": dates[0] if dates else None}
    pairs = [(vals[i], vals[i + 1]) for i in range(len(vals) - 1) if vals[i] is not None and vals[i + 1] is not None]

    # RAW
    out["raw_n"] = len(present)
    out["raw_comparisons"] = len(pairs)
    out["raw_increases"] = sum(1 for a, b in pairs if b > a)
    out["raw_decreases"] = sum(1 for a, b in pairs if b < a)
    out["raw_unchanged"] = sum(1 for a, b in pairs if b == a)
    out["raw_increase_rate"] = out["raw_increases"] / len(pairs) if pairs else None
    records, running = 0, None
    for i in present:
        v = vals[i]
        if running is not None and v > running:
            records += 1
        running = v if running is None else max(running, v)
    out["raw_record_highs"] = records

    latest_i = present[-1] if present else None
    latest = vals[latest_i] if latest_i is not None else None
    earlier = [vals[i] for i in present[:-1]]
    if latest is None:
        out["raw_latest_record"] = "n/a (no values)"
    elif not earlier:
        out["raw_latest_record"] = "n/a (single value)"
    else:
        out["raw_latest_record"] = "yes" if latest > max(earlier) else "no"
    out["raw_latest_rank"] = (1 + sum(1 for i in present if vals[i] > latest)) if latest is not None else None

    # ADJUSTED
    eligible = [i for i in present if vals[i] > 0]
    adj_pairs = [(a, b) for a, b in pairs if a > 0 and b > 0]
    out["adj_n"] = len(eligible)
    out["adj_comparisons"] = len(adj_pairs)
    out["adj_pairs_skipped"] = len(pairs) - len(adj_pairs)
    out["adj_increases"] = sum(1 for a, b in adj_pairs if b > a)
    out["adj_decreases"] = sum(1 for a, b in adj_pairs if b < a)
    out["adj_unchanged"] = sum(1 for a, b in adj_pairs if b == a)
    out["adj_increase_rate"] = out["adj_increases"] / len(adj_pairs) if adj_pairs else None
    records, running = 0, None
    for i in eligible:
        v = vals[i]
        if running is not None and v > running:
            records += 1
        running = v if running is None else max(running, v)
    out["adj_record_highs"] = records
    if latest is None:
        out["adj_latest_record"] = "n/a (no values)"
    elif latest <= 0:
        out["adj_latest_record"] = "excluded (≤ 0)"
    else:
        earlier_eligible = [vals[i] for i in eligible if i != latest_i]
        out["adj_latest_record"] = "yes" if earlier_eligible and latest > max(earlier_eligible) else ("n/a (single value)" if not earlier_eligible else "no")
    out["adj_excluded"] = sum(1 for i in present if vals[i] <= 0)
    out["adj_sign_changes"] = sum(1 for a, b in pairs if (a > 0 and b < 0) or (a < 0 and b > 0))

    # COMMON
    out["latest"] = latest
    out["latest_date"] = dates[latest_i] if latest_i is not None else None
    prior_i = present[-2] if len(present) >= 2 else None
    out["prior"] = vals[prior_i] if prior_i is not None else None
    out["prior_date"] = dates[prior_i] if prior_i is not None else None
    if out["prior"] is None or latest is None:
        out["pct_change"] = None
    elif out["prior"] == 0:
        out["pct_change"] = "NM"
    else:
        out["pct_change"] = (latest - out["prior"]) / abs(out["prior"])
    if present:
        min_i = min(present, key=lambda i: (vals[i], i))
        max_i = min(present, key=lambda i: (-vals[i], i))
        out.update(min=vals[min_i], min_date=dates[min_i], max=vals[max_i], max_date=dates[max_i])
    else:
        out.update(min=None, min_date=None, max=None, max_date=None)
    return out


def stats_table(ttm: pd.DataFrame, start_index: int, daily_counts: dict[str, int] | None = None) -> pd.DataFrame:
    rows = []
    sub = ttm.iloc[start_index:]
    dates = list(sub["valuation_date"])
    for s in SERIES:
        values = list(sub[s.key]) if s.key in sub else [None] * len(sub)
        stats = compute_stats(dates, values)
        stats.update(series=s.key, label=s.label, basis=s.basis, focus=s.focus, family=s.family)
        stats["daily_obs"] = (daily_counts or {}).get(s.daily_ratio) if s.daily_ratio else None
        stats["daily_ratio"] = s.daily_ratio
        rows.append(stats)
    return pd.DataFrame(rows)

# ============================== valuation/redflags.py ==============================
"""Red-flag screens. Every flag cites the filing it comes from.

Data screens use Input A. Text screens (going concern, auditor opinion,
material weakness, auditor change) need Input B; without it they are reported
as not assessable rather than clean.
"""


import html
import re
from dataclasses import dataclass

import pandas as pd


ACCN_RE = re.compile(r"\b\d{10}-\d{2}-\d{6}\b")

DEADLINE_DAYS = {"10-K": 90, "10-Q": 45, "20-F": 120, "10-KT": 90, "10-QT": 45}


@dataclass
class InputBDocument:
    name: str
    text: str

    @property
    def citation(self) -> str:
        match = ACCN_RE.search(self.name) or ACCN_RE.search(self.text[:5000])
        return f"{self.name}" + (f" (accession {match.group(0)})" if match else "")


def clean_text(raw: str) -> str:
    if "<" in raw and ">" in raw:
        raw = re.sub(r"(?is)<(script|style).*?</\1>", " ", raw)
        raw = re.sub(r"(?s)<[^>]+>", " ", raw)
        raw = html.unescape(raw)
    return re.sub(r"\s+", " ", raw)


def _sentences_with(text: str, pattern: re.Pattern, limit: int = 3) -> list[str]:
    out = []
    for match in pattern.finditer(text):
        lo = text.rfind(".", 0, match.start()) + 1
        hi = text.find(".", match.end())
        sentence = text[lo : hi + 1 if hi != -1 else len(text)].strip()
        if sentence and sentence not in out:
            out.append(sentence[:400])
        if len(out) >= limit:
            break
    return out


TEXT_SCREENS = [
    ("going concern", re.compile(r"substantial doubt[^.]{0,200}going concern|going concern[^.]{0,200}substantial doubt", re.I), "critical"),
    ("non-clean auditor opinion", re.compile(r"adverse opinion|qualified opinion|disclaimer of opinion|do not express an opinion|except for the (?:effects|possible effects)", re.I), "critical"),
    ("material weakness", re.compile(r"material weakness(?:es)?[^.]{0,160}(?:identified|exist|not effective|ineffective)|(?:not effective|ineffective)[^.]{0,160}material weakness", re.I), "serious"),
    ("auditor change", re.compile(r"(?:dismiss(?:ed|al)|resign(?:ed|ation)|engaged)[^.]{0,200}independent registered public accounting firm|change(?:s)? in (?:the )?(?:company'?s )?(?:independent )?(?:registered public )?account(?:ant|ing firm)", re.I), "serious"),
    ("impairment (text)", re.compile(r"impairment (?:charge|loss)[^.]{0,120}(?:\$|million|billion)", re.I), "warning"),
]


def text_flags(docs: list[InputBDocument]) -> list[dict]:
    flags = []
    for doc in docs:
        text = clean_text(doc.text)
        clean_opinion = re.search(r"present fairly, in all material respects", text, re.I)
        for name, pattern, severity in TEXT_SCREENS:
            hits = _sentences_with(text, pattern)
            if name == "non-clean auditor opinion" and hits and clean_opinion:
                severity = "warning"  # wording found alongside standard clean-opinion language; read in context
            for sentence in hits:
                flags.append({"flag": name, "severity": severity, "evidence": f"“{sentence}”", "source": doc.citation})
    return flags


def data_flags(review) -> list[dict]:
    flags: list[dict] = []
    ttm: pd.DataFrame = review.ttm
    annual: pd.DataFrame = review.annual_all

    # Negative equity at any valuation date.
    neg = ttm[pd.to_numeric(ttm["equity"], errors="coerce") < 0] if "equity" in ttm else ttm.iloc[0:0]
    if not neg.empty:
        last = neg.iloc[-1]
        flags.append(
            {
                "flag": "negative equity",
                "severity": "serious",
                "evidence": f"Parent equity negative at {len(neg)} valuation date(s); latest {last['equity']:,.0f} at {last['bs_instant'].date()}.",
                "source": f"{last['bs_form']} {last['bs_accn']}",
            }
        )

    # Persistent negative FCF: three or more consecutive negative fiscal years, or 3 of the last 5.
    fcf = annual[["fy_end", "fcf", "accn_ocf"]].dropna(subset=["fcf"]) if not annual.empty and "fcf" in annual else pd.DataFrame()
    if not fcf.empty:
        negative = list(fcf["fcf"] < 0)
        run, best = 0, 0
        for is_neg in negative:
            run = run + 1 if is_neg else 0
            best = max(best, run)
        last5 = negative[-5:]
        if best >= 3 or sum(last5) >= 3:
            years = fcf[fcf["fcf"] < 0]
            flags.append(
                {
                    "flag": "persistent negative FCF",
                    "severity": "serious",
                    "evidence": f"Annual FCF negative in {len(years)} fiscal year(s), longest run {best}; years ending "
                    + ", ".join(str(d.date()) for d in years["fy_end"].tail(6)),
                    "source": "; ".join(str(a) for a in years["accn_ocf"].tail(3)),
                }
            )

    # Dilution not explained by a detected split (shares are already split-normalized).
    if not annual.empty and "diluted_shares" in annual:
        sh = annual[["fy_end", "diluted_shares", "accn_diluted_shares"]].dropna()
        for (_, a), (_, b) in zip(sh.iloc[:-1].iterrows(), sh.iloc[1:].iterrows()):
            change = b["diluted_shares"] / a["diluted_shares"] - 1 if a["diluted_shares"] else 0
            if change > 0.05:
                flags.append(
                    {
                        "flag": "dilution",
                        "severity": "warning",
                        "evidence": f"Diluted weighted shares (split-normalized) rose {change:.1%} in the fiscal year ending {b['fy_end'].date()}; not explained by a detected split.",
                        "source": str(b["accn_diluted_shares"]),
                    }
                )

    # Late periodic filings (beyond the longest standard deadline for the form).
    filings = review.filings
    seen: set[tuple[str, pd.Timestamp]] = set()
    for row in filings.sort_values(["filed", "accn"]).itertuples():
        base_form = row.form.replace("/A", "")
        if row.form.endswith("/A") or base_form not in DEADLINE_DAYS or pd.isna(row.report_period):
            continue
        key = (base_form, row.report_period)
        if key in seen:
            continue  # a later filing for an already-reported period is a re-filing, not a late original
        seen.add(key)
        limit = DEADLINE_DAYS[base_form]
        if base_form == "20-F" and row.report_period < pd.Timestamp("2011-12-15"):
            limit = 183
        days = (row.filed - row.report_period).days
        deadline = row.report_period + pd.Timedelta(days=limit)
        while deadline.weekday() >= 5 or deadline.date() in exchange_holidays(deadline.year):
            deadline += pd.Timedelta(days=1)  # a deadline on a weekend or holiday moves to the next business day
        if row.filed > deadline:
            flags.append(
                {
                    "flag": "possible late filing",
                    "severity": "warning",
                    "evidence": f"{row.form} for the period ending {row.report_period.date()} filed {row.filed.date()} ({days} days; longest standard deadline {limit} days, due {deadline.date()}). "
                    "It is the first filing in Input A for this period, but it may be a re-filing of an earlier report without XBRL; verify on EDGAR (NT 10-K/10-Q notices are not in Input A).",
                    "source": f"{row.form} {row.accn}",
                }
            )

    # Impairments by fiscal year.
    if not annual.empty and "impairment" in annual:
        imp = annual[["fy_end", "impairment", "accn_impairment"]].dropna()
        for _, row in imp[imp["impairment"] > 0].iterrows():
            flags.append(
                {
                    "flag": "impairment",
                    "severity": "warning",
                    "evidence": f"Impairment charges {row['impairment']:,.0f} in the fiscal year ending {row['fy_end'].date()}.",
                    "source": str(row["accn_impairment"]),
                }
            )

    # Multiples driven by one-time items (latest TTM).
    latest = review.latest_one_time
    if latest:
        ni = latest.get("net_income")
        items = {
            k: float(v)
            for k, v in latest.items()
            if k != "net_income" and not k.startswith("_") and isinstance(v, (int, float)) and v == v and v != 0
        }
        if isinstance(ni, (int, float)) and ni == ni and ni and items:
            total = sum(abs(v) for v in items.values())
            if total > 0.2 * abs(ni):
                flags.append(
                    {
                        "flag": "multiples driven by one-time items",
                        "severity": "serious",
                        "evidence": "Latest TTM one-time items "
                        + ", ".join(f"{k} {v:,.0f}" for k, v in items.items())
                        + f" equal {total / abs(ni):.0%} of TTM net income {ni:,.0f}; earnings-based multiples are affected.",
                        "source": latest.get("_source", ""),
                    }
                )

    for row in review.amendments.itertuples():
        flags.append({"flag": "amendment", "severity": "info", "evidence": f"{row.form} filed {row.filed.date()}.", "source": f"{row.form} {row.accn}"})
    return flags

# ============================== valuation/pipeline.py ==============================
"""Run the full valuation and fundamentals review for one company."""


import math
from dataclasses import dataclass, field
from datetime import date

import numpy as np
import pandas as pd


FLOW_KEYS = ["revenue", "gross_profit", "operating_income", "net_income", "ocf", "capex", "da", "dividends_paid", "impairment", "disc_ops", "disposal_gains"]
TTM_COLUMNS = {
    "revenue": "ttm_revenue",
    "gross_profit": "ttm_gross_profit",
    "operating_income": "ttm_operating_income",
    "net_income": "ttm_net_income",
    "ocf": "ttm_ocf",
    "capex": "ttm_capex",
    "da": "ttm_da",
    "dividends_paid": "ttm_dividends_paid",
}
INSTANT_KEYS = ["cash", "total_assets", "total_liabilities", "equity", "long_term_debt", "goodwill", "intangibles", "nci", "preferred", "bs_shares"]
DISPLAY_YEARS = 15
EPS_CHECK_TOL = 0.05
THIN_MIN_YEARS = 3
THIN_MIN_TTM = 8
COVER_CLASS_RATIO = 0.85  # cover-page shares below this share of weighted basic shares cover one class only
# Opt-in (off by default): replace total OCF by continuing-operations OCF wherever both are tagged (or total -
# discontinued). Measured on 486 tickers it fixes some series (BDX, DLTR, HUBB, SLB) but per-period substitution
# mixes the two definitions for more filers than it repairs, so the default keeps the total.
PREFER_CONTINUING_OCF = False


@dataclass
class ReviewInputs:
    facts: object  # Input A: dict, path or file
    as_of: date  # Input D
    prices: object | None = None  # Input C: DataFrame, path or file (date, close[, dividend, split_ratio])
    price_meta: PriceMeta = field(default_factory=PriceMeta)
    dividends: object | None = None
    filings_text: list[InputBDocument] = field(default_factory=list)  # Input B
    current_price: float | None = None  # Input D price, if given
    ticker: str = ""
    source_notes: list[str] = field(default_factory=list)  # provenance of Inputs A and C


@dataclass
class Review:
    inputs: ReviewInputs
    cf: CompanyFacts
    dq: DQLog
    calendar: Calendar
    tables: dict[str, MetricTable]
    prices: PriceData | None
    basis: SplitBasis
    ttm: pd.DataFrame
    build: pd.DataFrame
    annual_all: pd.DataFrame
    coverage: pd.DataFrame
    coverage_summary: pd.DataFrame
    stats: pd.DataFrame
    stats_start_index: int
    snapshot: pd.DataFrame
    snapshot_inputs: dict
    growth: pd.DataFrame
    red_flags: pd.DataFrame
    header: dict
    thin: bool
    thin_reason: str
    display_start: pd.Timestamp | None
    price_block_reason: str
    filings: pd.DataFrame
    amendments: pd.DataFrame
    latest_one_time: dict
    daily: pd.DataFrame | None = None
    percentiles: dict = field(default_factory=dict)
    daily_counts: dict = field(default_factory=dict)


# ---------------------------------------------------------------- helpers

def _num(v) -> bool:
    return isinstance(v, (int, float, np.floating)) and not isinstance(v, bool) and math.isfinite(float(v))


def _slot_metric(key: str, taxonomy_tags: dict[str, tuple[str, ...]]) -> MetricDef:
    return MetricDef(key, key.replace("_", " "), "instant", "money", False, taxonomy_tags, {})


class BalanceSheet:
    """Balance sheet and share count from the most recent filing on or before a date."""

    def __init__(self, cf: CompanyFacts, tables: dict[str, MetricTable], debt: dict[str, MetricTable], basis: SplitBasis, dq: DQLog):
        self.cf, self.tables, self.debt, self.basis, self.dq = cf, tables, debt, basis, dq
        anchor_tags = set()
        for key in BALANCE_ANCHOR_METRICS:
            for taxonomy, tags in METRICS[key].candidates.items():
                anchor_tags |= {(taxonomy, t) for t in tags}
        f = cf.facts
        keys = f["taxonomy"] + ":" + f["tag"]
        mask = (f["kind"] == "instant") & keys.isin({f"{t}:{g}" for t, g in anchor_tags})
        anchors = f[mask & (f["end"] <= f["filed"])]
        self.anchor = anchors.groupby("accn").agg(filed=("filed", "min"), instant=("end", "max"), form=("form", "first")).reset_index().sort_values(["filed", "accn"])
        cover = f[(f["taxonomy"] == DEI) & (f["tag"] == COVER_SHARES_TAG) & (f["unit"] == "shares")]
        self.cover = cover.sort_values(["filed", "accn"])
        self._logged_multi: set[str] = set()
        self._logged_partial: set[str] = set()
        self._logged_fallback = False

    def at(self, when: pd.Timestamp) -> dict:
        out: dict = {}
        cand = self.anchor[self.anchor["filed"] <= when]
        if cand.empty:
            out["bs_reason"] = "no balance sheet filed on or before this date"
            return out
        last = cand.iloc[-1]
        instant = last["instant"]
        out.update(bs_instant=instant, bs_accn=last["accn"], bs_form=last["form"], bs_filed=last["filed"])
        for key in INSTANT_KEYS:
            row = self.tables[key].get(None, instant)
            out[key] = None if row is None else float(row["value"])
            out[f"{key}_tag"] = "" if row is None else row["tag"]
            out[f"{key}_accn"] = "" if row is None else row["accn"]
        out.update(self._borrowings(instant))
        gw, ia, eq = out.get("goodwill"), out.get("intangibles"), out.get("equity")
        if eq is None:
            out["tbv"], out["tbv_reason"] = None, "parent equity not tagged"
        elif gw is None and ia is None:
            out["tbv"], out["tbv_reason"] = None, "goodwill and intangibles not tagged"
        else:
            out["tbv"] = eq - (gw or 0.0) - (ia or 0.0)
            out["tbv_reason"] = "" if (gw is not None and ia is not None) else ("intangibles not tagged" if ia is None else "goodwill not tagged") + " (treated as not present)"
        out.update(self._cover_shares(when, instant))
        return out

    def _borrowings(self, instant) -> dict:
        def get(slot):
            row = self.debt[slot].get(None, instant)
            return (None, "") if row is None else (float(row["value"]), row["tag"])

        dct, dct_tag = get("debt_current_total")
        ltdc, ltdc_tag = get("ltd_current")
        st, st_tag = get("short_term")
        ltdn, ltdn_tag = get("ltd_noncurrent")
        ltdt, ltdt_tag = get("ltd_total")
        notes = []
        if ltdn is not None:
            noncurrent, nc_src = ltdn, ltdn_tag
            if dct is not None:
                current, c_src = dct, dct_tag
            elif ltdc is not None or st is not None:
                current = (ltdc or 0.0) + (st or 0.0)
                c_src = " + ".join(t for t in (ltdc_tag, st_tag) if t)
            else:
                current, c_src = None, ""
        elif ltdt is not None:
            if ltdc is not None:
                noncurrent, nc_src = ltdt - ltdc, f"{ltdt_tag} - {ltdc_tag}"
                current = dct if dct is not None else ltdc + (st or 0.0)
                c_src = dct_tag if dct is not None else " + ".join(t for t in (ltdc_tag, st_tag) if t)
            else:
                noncurrent, nc_src = ltdt, f"{ltdt_tag} (includes current maturities)"
                current, c_src = st, st_tag
                if dct is not None:
                    notes.append("DebtCurrent not added: LongTermDebt already includes current maturities and the split is not tagged")
        else:
            noncurrent, nc_src = None, ""
            if dct is not None:
                current, c_src = dct, dct_tag
            elif ltdc is not None or st is not None:
                current = (ltdc or 0.0) + (st or 0.0)
                c_src = " + ".join(t for t in (ltdc_tag, st_tag) if t)
            else:
                current, c_src = None, ""
        total = None if current is None and noncurrent is None else (current or 0.0) + (noncurrent or 0.0)
        return {
            "debt_current": current,
            "debt_current_tag": c_src,
            "debt_noncurrent": noncurrent,
            "debt_noncurrent_tag": nc_src,
            "total_borrowings": total,
            "borrowings_note": "; ".join(notes) if notes else ("no borrowings tagged at this instant" if total is None else ""),
        }

    def _cover_shares(self, when, instant) -> dict:
        cand = self.cover[self.cover["filed"] <= when]
        if not cand.empty and cand["end"].max() < instant - pd.Timedelta(days=400):
            if "stale" not in self._logged_partial:
                self._logged_partial.add("stale")
                self.dq.add(
                    "fallback_tag",
                    "Shares outstanding",
                    f"The latest cover-page share count ({cand['end'].max().date()}) is more than a year older than the balance sheet "
                    f"({instant.date()}); it is not used and market cap falls back to other share counts.",
                    period=instant,
                    severity="warning",
                )
            cand = cand.iloc[0:0]
        if not cand.empty:
            accn = cand.iloc[-1]["accn"]
            rows = cand[cand["accn"] == accn]
            rows = rows[rows["end"] == rows["end"].max()].drop_duplicates("val")
            reported = float(rows["val"].sum())
            filed = rows.iloc[0]["filed"]
            if len(rows) > 1 and accn not in self._logged_multi:
                self._logged_multi.add(accn)
                self.dq.add("assumption", "cover-page shares", f"{len(rows)} distinct cover-page share counts in one filing (likely share classes); summed to {reported:,.0f}.", period=rows.iloc[0]["end"], filing=accn, severity="warning")
            factor = self.basis.share_factor(filed)
            weighted = self._weighted_basic(instant)
            if weighted and reported * factor < COVER_CLASS_RATIO * weighted:
                if accn not in self._logged_partial:
                    self._logged_partial.add(accn)
                    self.dq.add(
                        "fallback_tag",
                        "Shares outstanding",
                        f"Cover-page shares {reported * factor:,.0f} are {reported * factor / weighted:.0%} of weighted basic shares {weighted:,.0f} for the period; "
                        "they appear to cover one share class only (or are a placeholder), so market cap uses the weighted basic count.",
                        period=rows.iloc[0]["end"],
                        filing=accn,
                        severity="warning",
                    )
                return self._fallback_shares(instant, skip_balance_sheet=True)
            return {
                "cover_shares_reported": reported,
                "cover_date": rows.iloc[0]["end"],
                "cover_accn": accn,
                "cover_form": rows.iloc[0]["form"],
                "cover_filed": filed,
                "share_factor": factor,
                "shares_out": reported * factor,
                "shares_source": "dei:EntityCommonStockSharesOutstanding",
            }
        return self._fallback_shares(instant)

    def _weighted_basic(self, instant) -> float | None:
        basic = self.tables["basic_shares"].table
        if basic.empty:
            return None
        row = self._latest_basic(instant)
        return None if row is None else float(row["value"])

    def _latest_basic(self, instant) -> dict | None:
        """Weighted basic shares for the period ending at the instant, else the latest within 200 days before it."""
        basic = self.tables["basic_shares"].table
        if basic.empty:
            return None
        near = basic[(basic["end"] <= instant + pd.Timedelta(days=3)) & (basic["end"] >= instant - pd.Timedelta(days=200))]
        if near.empty:
            return None
        latest_end = near["end"].max()
        row = near[near["end"] == latest_end].sort_values("days").iloc[0].to_dict()
        earlier = basic[(basic["end"] < latest_end - pd.Timedelta(days=30)) & (basic["end"] >= latest_end - pd.Timedelta(days=400))]
        if not earlier.empty:
            prior = float(earlier["value"].median())  # median of the prior year resists a single mis-scaled period
            if prior > 0 and not (1 / 3 <= float(row["value"]) / prior <= 3):
                if "implausible_basic" not in self._logged_partial:
                    self._logged_partial.add("implausible_basic")
                    self.dq.add(
                        "fallback_tag",
                        "Weighted basic shares",
                        f"Weighted basic shares {float(row['value']):,.0f} for the period ending {latest_end.date()} are more than 3x away from the prior year's "
                        f"median {prior:,.0f} (likely a scaling error in the filing); not used to check or replace cover-page shares.",
                        period=latest_end,
                        severity="warning",
                    )
                return None
        return row

    def _fallback_shares(self, instant, skip_balance_sheet: bool = False) -> dict:
        row = None if skip_balance_sheet else self.tables["bs_shares"].get(None, instant)
        source = None
        if row is not None:
            source = f"{row['tag']} (balance sheet, fallback)"
        else:
            row = self._latest_basic(instant)
            if row is not None:
                source = f"{row['tag']} weighted basic, {int(row['days'])}-day period ending {row['end'].date()} (fallback)"
        if row is not None:
            if not self._logged_fallback:
                self._logged_fallback = True
                self.dq.add(
                    "fallback_tag",
                    "Shares outstanding",
                    "dei:EntityCommonStockSharesOutstanding is not available without share-class dimensions (companyfacts excludes them); "
                    f"market cap uses {source} instead. Check the share count in the workbook (Valuation_Inputs).",
                    severity="warning",
                )
            return {
                "cover_shares_reported": float(row["value"]),
                "cover_date": instant,
                "cover_accn": row["accn"],
                "cover_form": row["form"],
                "cover_filed": row["filed"],
                "share_factor": 1.0,
                "shares_out": float(row["value"]),
                "shares_source": source,
            }
        return {"shares_out": None, "shares_reason": "no shares outstanding filed on or before this date"}


# ---------------------------------------------------------------- main

def run_review(inp: ReviewInputs) -> Review:
    dq = DQLog()
    as_of = pd.Timestamp(inp.as_of).normalize()
    cf = load_companyfacts(inp.facts, inp.as_of)
    for note in inp.source_notes:
        dq.add("input", "provenance", note)
    if cf.excluded_after_asof:
        dq.add("input", "Input A", f"{cf.excluded_after_asof:,} facts filed after the as-of date {as_of.date()} were excluded (no look-ahead).")
    dq.add("input", "Input B", f"{len(inp.filings_text)} excerpt(s) supplied." if inp.filings_text else "Not supplied: going-concern, auditor-opinion, material-weakness and auditor-change screens are not assessable.")

    prices = None
    meta = inp.price_meta or PriceMeta()
    if inp.prices is not None:
        prices = load_prices(inp.prices, meta, inp.dividends, dq)
        post = prices.close_raw.index > as_of
        if post.any():
            dq.add("input", "Input C", f"{int(post.sum())} price rows after the as-of date ignored.")
        prices.close_raw = prices.close_raw[~post]
        prices.invalid_days = [d for d in prices.invalid_days if d <= as_of]
        prices.dividends_raw = prices.dividends_raw[prices.dividends_raw["ex_date"] <= as_of].reset_index(drop=True)
        if prices.close_raw.empty:
            dq.add("input", "Input C", "No prices on or before the as-of date.", severity="warning")
            prices = None
    else:
        dq.add("input", "Input C", "Not supplied: every price-based ratio is 'no price data'.", severity="warning")

    explicit = prices.splits if prices is not None else None
    events = detect_splits(cf.facts, dq, prices.close_raw if prices is not None else None, meta.split_adjusted, explicit, as_of)
    basis = SplitBasis(events, as_of)

    price_block = ""
    if prices is not None:
        apply_basis(prices, basis, dq, check_from=cf.filings["filed"].min() - pd.DateOffset(years=1))
        _restore_asof_basis(prices, as_of, dq)
        if meta.split_adjusted is None:
            if basis.events:
                price_block = "Input C does not state whether prices are split-adjusted and a split was detected"
                dq.add("price_basis", "split basis not stated", price_block + "; per-share price ratios left blank.", severity="critical")
            else:
                dq.add("price_basis", "split basis not stated", "Input C does not state its split basis; no split detected in Input A, so the basis does not change any ratio.", severity="warning")
        if meta.dividend_adjusted is None:
            dq.add("price_basis", "dividend basis not stated", "Input C does not state whether closes are dividend-adjusted; treated as traded closes.", severity="warning")

    tables = {key: build_metric_table(cf, METRICS[key], basis, dq) for key in METRICS if key != "cost_of_revenue"}
    tables["revenue"] = lessor_revenue(tables["revenue"], tables["lease_income"], dq)
    cost_metric = screen_component_cost_tags(cf, METRICS["cost_of_revenue"], METRICS["gross_profit"], tables["revenue"], basis, dq)
    tables["cost_of_revenue"] = build_metric_table(cf, cost_metric, basis, dq)
    tables["gross_profit"] = derive_gross_profit(tables["gross_profit"], tables["revenue"], tables["cost_of_revenue"], dq)
    tables["net_income"] = parent_share_of_profit_loss(tables["net_income"], tables["nci_income"], dq)
    tables["equity"] = parent_equity(tables["equity"], tables["nci"], dq)
    capex_parts = {tag: build_metric_table(cf, MetricDef(f"capex:{tag}", tag, "duration", "money", True, {US: (tag,)}, {}), basis, DQLog()) for tag in CAPEX_COMPONENT_TAGS}
    tables["capex"] = capex_components(tables["capex"], {k: v for k, v in capex_parts.items() if not v.table.empty}, dq)
    if PREFER_CONTINUING_OCF:
        tables["ocf"] = continuing_operations_ocf(tables["ocf"], tables["ocf_continuing"], dq, tables["ocf_discontinued"])
    debt = {slot: build_metric_table(cf, _slot_metric(slot, {US: DEBT_TAGS[US][slot], IFRS: DEBT_TAGS[IFRS][slot]}), basis, DQLog()) for slot in DEBT_TAGS[US]}

    currencies = sorted(set(cf.filings["dominant_currency"].dropna()))
    reporting_currency = _latest_regime(cf)[1]
    if prices is not None and not price_block:
        if meta.currency and reporting_currency and meta.currency != reporting_currency:
            price_block = f"price currency {meta.currency} differs from reporting currency {reporting_currency}; no FX input supplied"
            dq.add("price_basis", "currency mismatch", price_block + ". Price-based ratios left blank.", severity="critical")
        elif not meta.currency and reporting_currency and reporting_currency != "USD":
            price_block = f"price currency not stated and reporting currency is {reporting_currency}"
            dq.add("price_basis", "currency not stated", price_block + ". Price-based ratios left blank.", severity="critical")
        elif not meta.currency:
            dq.add("price_basis", "currency not stated", "Input C currency not stated; assumed USD, the reporting currency.", severity="warning")

    calendar = build_calendar([tables[k] for k in CALENDAR_METRICS], dq, as_of)
    qv = {key: quarterly_values(tables[key], calendar, dq) for key in FLOW_KEYS}
    qv_shares = quarterly_values(tables["diluted_shares"], calendar, DQLog())
    bs = BalanceSheet(cf, tables, debt, basis, dq)

    rows, build_rows = _valuation_rows(cf, calendar, tables, qv, qv_shares, bs, prices, price_block, dq)
    ttm = pd.DataFrame(rows)
    build = pd.DataFrame(build_rows)
    if not ttm.empty:
        _add_cagr(ttm)
    stats_start = _regime_breaks(ttm, dq)
    annual_all = _annual_table(calendar, tables, prices)
    coverage, coverage_summary = _coverage(calendar, tables, cf, prices, as_of)
    filings = cf.filings
    amendments = filings[filings["is_amendment"]]
    for row in amendments.itertuples():
        dq.add("amendment", row.form, f"Amended filing {row.form} filed {row.filed.date()}; its values supersede earlier ones for the periods it reports.", period=row.filed, filing=row.accn)
    _missing_periods(calendar, ttm, dq)

    display_start = None
    fy_done = [y for y in calendar.years if y.end is not None and y.end <= as_of]
    if fy_done:
        display_start = fy_done[-DISPLAY_YEARS:][0].start

    n_years = int(annual_all["revenue"].notna().sum()) if not annual_all.empty else 0
    n_years = max(n_years, int(annual_all["net_income"].notna().sum()) if not annual_all.empty else 0)
    n_ttm = int(ttm["ttm_net_income"].notna().sum()) if not ttm.empty else 0
    thin = n_years < THIN_MIN_YEARS or n_ttm < THIN_MIN_TTM
    thin_reason = f"{n_years} fiscal year(s) with revenue or net income and {n_ttm} TTM earnings observation(s); at least {THIN_MIN_YEARS} and {THIN_MIN_TTM} are needed for a trend review." if thin else ""
    if thin:
        dq.add("coverage", "coverage too thin", thin_reason, severity="critical")

    snapshot_inputs = _snapshot_inputs(inp, ttm, bs, prices, price_block, as_of, dq)
    review = Review(
        inputs=inp, cf=cf, dq=dq, calendar=calendar, tables=tables, prices=prices, basis=basis, ttm=ttm, build=build,
        annual_all=annual_all, coverage=coverage, coverage_summary=coverage_summary, stats=pd.DataFrame(), stats_start_index=stats_start,
        snapshot=pd.DataFrame(), snapshot_inputs=snapshot_inputs, growth=_growth(annual_all, ttm), red_flags=pd.DataFrame(),
        header={}, thin=thin, thin_reason=thin_reason, display_start=display_start, price_block_reason=price_block,
        filings=filings, amendments=amendments, latest_one_time=_latest_one_time(ttm),
    )
    review.daily = daily_table(review)
    review.daily_counts = _daily_counts(review)
    review.stats = stats_table(ttm, stats_start, review.daily_counts) if not ttm.empty else pd.DataFrame()
    review.snapshot, review.percentiles = _snapshot_table(review)
    _flag_extremes(review)
    flags = data_flags(review) + text_flags(inp.filings_text)
    if not inp.filings_text:
        flags.append({"flag": "text screens not assessable", "severity": "info", "evidence": "Input B not supplied: going-concern language, auditor opinion, material weakness and auditor change could not be checked.", "source": "-"})
    review.red_flags = pd.DataFrame(flags, columns=["flag", "severity", "evidence", "source"])
    review.header = _header(review, currencies)
    return review


def _restore_asof_basis(prices: PriceData, as_of: pd.Timestamp, dq: DQLog) -> None:
    """Split-adjusted feeds are adjusted through their download date; undo splits after the as-of date."""
    if prices.meta.split_adjusted is not True or prices.splits.empty:
        return
    later = prices.splits[prices.splits["date"] > as_of]
    if later.empty:
        return
    factor = float(np.prod(later["ratio"]))
    prices.close = prices.close * factor
    if not prices.dividends.empty:
        prices.dividends["amount"] = prices.dividends["amount"] * factor
    dq.add("price_basis", "splits after as-of date", f"Input C is adjusted for split(s) after the as-of date ({', '.join(str(d.date()) for d in later['date'])}); closes multiplied by {factor:g} to restore the as-of basis.", severity="warning")


def _latest_regime(cf: CompanyFacts) -> tuple[str, str]:
    periodic = cf.filings.dropna(subset=["report_period"])
    if periodic.empty:
        return US, ""
    last = periodic.iloc[-1]
    return last["regime_taxonomy"], last["dominant_currency"] or ""


def _valuation_rows(cf, cal: Calendar, tables, qv, qv_shares, bs: BalanceSheet, prices, price_block, dq):
    rows, build = [], []
    running = None
    triggers = []
    for f in cf.filings.itertuples():
        rp = f.report_period
        if pd.isna(rp):
            continue
        if running is not None and rp <= running + pd.Timedelta(days=3):
            continue
        running = rp
        quarter = next((q for q in cal.quarters if abs(q.end - rp) <= pd.Timedelta(days=3)), None)
        if quarter is None:
            dq.add("period_label", "filing period outside fiscal calendar", f"{f.form} {f.accn} first reports a period ending {rp.date()} that is not a fiscal quarter end; no valuation date created.", period=rp, filing=f.accn)
            continue
        if triggers and triggers[-1][1] == f.filed:
            dq.add("assumption", "same-day filings", f"{triggers[-1][2]} and {f.accn} were both filed {f.filed.date()}; the later period ({quarter.label}) is the valuation row.", period=f.filed)
            triggers.pop()
        triggers.append((quarter, f.filed, f.accn, f.form, f.regime_taxonomy, f.dominant_currency))

    fy_fallback: list[pd.Timestamp] = []
    fx_rows: list[pd.Timestamp] = []
    for quarter, filed, accn, form, taxonomy, currency in triggers:
        row = {
            "quarter_end": quarter.end,
            "fiscal_quarter": f"Q{quarter.k}",
            "fy_end": cal.years[quarter.fy].end,
            "valuation_date": filed,
            "form": form,
            "accn": accn,
            "regime": f"{taxonomy} {currency}",
        }
        reasons = []
        for key in FLOW_KEYS:
            t = ttm_flow(tables[key], qv[key], cal, quarter, dq if key in TTM_COLUMNS else None)
            col = TTM_COLUMNS.get(key, f"ttm_{key}")
            row[col] = t.value
            row[f"{col}_method"] = t.method
            if t.value is None and key in TTM_COLUMNS:
                reasons.append(t.reason)
            if key in TTM_COLUMNS:
                build.append(_build_row(row, key, t))
        sh = ttm_shares(tables["diluted_shares"], qv_shares, cal, quarter)
        row["avg_diluted_shares"] = sh.value
        row["shares_method"] = sh.method
        row["shares_note"] = sh.sources or sh.reason
        build.append(_build_row(row, "diluted_shares", sh))
        if sh.value is None:
            reasons.append(f"Diluted shares: {sh.reason}")
        elif sh.method == "FY" and quarter.k != 4:
            fy_fallback.append(quarter.end)

        ni, shares = row["ttm_net_income"], row["avg_diluted_shares"]
        row["eps_ttm"] = ni / shares if _num(ni) and _num(shares) and shares > 0 else None
        reported = reported_ttm_eps(tables["eps_diluted"], cal, quarter)
        row["eps_reported_ttm"] = reported
        if _num(row["eps_ttm"]) and _num(reported) and reported != 0:
            gap = abs(row["eps_ttm"] - reported) / abs(reported)
            row["eps_check_gap"] = gap
            if gap > EPS_CHECK_TOL:
                dq.add("eps_crosscheck", "TTM diluted EPS", f"TTM ending {quarter.end.date()}: NI/avg diluted shares = {row['eps_ttm']:.4f} vs reported FY+YTD-PYTD EPS = {reported:.4f} ({gap:.1%} apart).", period=quarter.end, severity="warning")
        else:
            row["eps_check_gap"] = None

        ocf, capex = row["ttm_ocf"], row["ttm_capex"]
        row["ttm_fcf"] = ocf - capex if _num(ocf) and _num(capex) else None
        oi, da = row["ttm_operating_income"], row["ttm_da"]
        row["ttm_ebitda"] = oi + da if _num(oi) and _num(da) else None
        missing_oi = "operating income (OperatingIncomeLoss) not available for this TTM period"
        missing = [n for n, v in (("operating income", oi), ("D&A", da)) if not _num(v)]
        row["ebitda_reason"] = f"TTM EBITDA unavailable: {' and '.join(missing)} missing" if missing else ""
        row["operating_income_reason"] = missing_oi if not _num(oi) else ""
        missing = [n for n, v in (("operating cash flow", ocf), ("capex", capex)) if not _num(v)]
        row["fcf_reason"] = f"TTM FCF unavailable: {' and '.join(missing)} missing" if missing else ""

        bal = bs.at(filed)
        row.update(bal)
        if row.get("equity") is None:
            reasons.append("Parent equity: " + ("not tagged at " + str(bal["bs_instant"].date()) if "bs_instant" in bal else bal.get("bs_reason", "")))

        dps, n_div, dps_reason = ttm_dividends(prices, quarter.end)
        row["dps_ttm"], row["dps_count"], row["dps_reason"] = dps, n_div, dps_reason

        for col, src in (("revenue_ps", "ttm_revenue"), ("gp_ps", "ttm_gross_profit"), ("oi_ps", "ttm_operating_income"), ("ocf_ps", "ttm_ocf"), ("fcf_ps", "ttm_fcf"), ("da_ps", "ttm_da"), ("ebitda_ps", "ttm_ebitda")):
            v = row[src]
            row[col] = v / shares if _num(v) and _num(shares) and shares > 0 else None
        so = row.get("shares_out")
        row["bvps"] = row["equity"] / so if _num(row.get("equity")) and _num(so) and so > 0 else None

        price_currency = prices.meta.currency if prices is not None else ""
        row["currency_ok"] = not (price_currency and currency and price_currency != currency)
        if prices is None:
            lookup_reason = "no price data"
            row.update(price=None, price_date=None, raw_price=None, price_factor=None)
        elif not row["currency_ok"]:
            lookup_reason = f"fundamentals reported in {currency}, price in {price_currency}; no FX input"
            row.update(price=None, price_date=None, raw_price=None, price_factor=None)
            fx_rows.append(quarter.end)
        elif price_block:
            lookup_reason = price_block
            row.update(price=None, price_date=None, raw_price=None, price_factor=None)
        else:
            look = price_on_or_before(prices, filed, allow_stale=False)
            lookup_reason = look.reason
            row.update(price=look.close, price_date=look.date, raw_price=look.raw_close, price_factor=look.factor if look.close is not None else None)
            if look.close is None and look.reason:
                dq.add("missing_period", "price at valuation date", f"Valuation date {filed.date()} ({quarter.label}): {look.reason}; price ratios left blank.", period=filed)
        row["price_reason"] = lookup_reason
        _market_values(row)
        row["notes"] = "; ".join(r for r in reasons if r)
        rows.append(row)
    if fx_rows:
        dq.add(
            "price_basis",
            "currency mismatch by period",
            f"{len(fx_rows)} valuation date(s) (TTM ends {fx_rows[0].date()} to {fx_rows[-1].date()}) report fundamentals in a currency other than Input C's "
            f"{prices.meta.currency} prices. No FX input was supplied, so price ratios for those dates (and the daily days they cover) are blank.",
            severity="warning",
        )
    if fy_fallback:
        dq.add(
            "shares_method",
            "TTM average diluted shares",
            f"For {len(fy_fallback)} TTM period(s) ending Q1-Q3 ({fy_fallback[0].date()} to {fy_fallback[-1].date()}) the four quarterly weighted diluted counts "
            "were not all reported (a 3-month Q4 count is rarely filed and is never derived), so the latest fiscal-year count is used. "
            "TTM EPS is cross-checked against reported EPS; gaps above 5% are logged under eps_crosscheck.",
        )
    return rows, build


def _build_row(row: dict, key: str, t) -> dict:
    comps = list(t.components) + [None] * (4 - len(t.components))
    labels = list(t.component_labels) + [""] * (4 - len(t.component_labels))
    return {
        "quarter_end": row["quarter_end"],
        "valuation_date": row["valuation_date"],
        "metric": key,
        "method": t.method,
        "a": comps[0], "b": comps[1], "c": comps[2], "d": comps[3],
        "a_label": labels[0], "b_label": labels[1], "c_label": labels[2], "d_label": labels[3],
        "ttm": t.value,
        "alt": t.alt_value,
        "alt_method": t.alt_method,
        "reason": t.reason,
        "sources": t.sources,
    }


def _market_values(row: dict) -> None:
    price, so = row.get("price"), row.get("shares_out")
    row["mcap"] = price * so if _num(price) and _num(so) else None
    cash = row.get("cash")
    if _num(row["mcap"]) and _num(cash):
        row["ev"] = row["mcap"] + sum(row.get(k) or 0.0 for k in ("total_borrowings", "nci", "preferred")) - cash
        row["ev_reason"] = ""
    else:
        row["ev"] = None
        row["ev_reason"] = "market cap unavailable" if not _num(row["mcap"]) else "cash not tagged"


def _add_cagr(ttm: pd.DataFrame) -> None:
    cagr, reasons, base_idx = [], [], []
    ends = list(ttm["quarter_end"])
    for i, end in enumerate(ends):
        target = end - pd.DateOffset(years=5)
        j = next((j for j in range(i) if abs(ends[j] - target) <= pd.Timedelta(days=10)), None)
        base_idx.append(j)
        if j is None:
            cagr.append(None)
            reasons.append("no TTM EPS five years earlier")
            continue
        now, then = ttm.at[i, "eps_ttm"], ttm.at[j, "eps_ttm"]
        if not (_num(now) and _num(then)):
            cagr.append(None)
            reasons.append("TTM EPS missing at an endpoint")
        elif now <= 0 or then <= 0:
            cagr.append(None)
            reasons.append("TTM EPS not positive at an endpoint; CAGR undefined")
        else:
            cagr.append((now / then) ** (1 / 5) - 1)
            reasons.append("")
    ttm["eps_cagr5"] = cagr
    ttm["eps_cagr5_reason"] = reasons
    ttm["eps_cagr5_base_row"] = base_idx


def _regime_breaks(ttm: pd.DataFrame, dq: DQLog) -> int:
    if ttm.empty:
        return 0
    start = 0
    regimes = list(ttm["regime"])
    for i in range(1, len(regimes)):
        if regimes[i] != regimes[i - 1]:
            prev_tax, prev_cur = regimes[i - 1].split(" ")
            tax, cur = regimes[i].split(" ")
            category = "taxonomy_change" if prev_tax != tax else "currency_change"
            dq.add(
                category,
                f"{regimes[i - 1]} -> {regimes[i]}",
                f"Reporting regime changed at the valuation date {ttm.at[i, 'valuation_date'].date()} ({ttm.at[i, 'form']} {ttm.at[i, 'accn']}). "
                "Series are not spliced across the break for statistics: statistics restart here. TTM values near the break can combine periods from both regimes.",
                period=ttm.at[i, "quarter_end"],
                filing=ttm.at[i, "accn"],
                severity="critical",
            )
            start = i
    return start


def _annual_table(cal: Calendar, tables, prices) -> pd.DataFrame:
    rows = []
    for fy in cal.years:
        if fy.end is None:
            continue
        row = {"fy_end": fy.end, "fy_start": fy.start, "fy_label": fy.label, "weeks": fy.week_note, "days": fy.days}
        for key in ["revenue", "gross_profit", "operating_income", "net_income", "eps_diluted", "diluted_shares", "ocf", "capex", "da", "dividends_paid", "dps_declared", "impairment"]:
            r = tables[key].get(fy.start, fy.end)
            row[key] = None if r is None else float(r["value"])
            row[f"accn_{key}"] = None if r is None else r["accn"]
        for key in ["cash", "total_assets", "total_liabilities", "equity", "long_term_debt"]:
            r = tables[key].get(None, fy.end)
            row[key] = None if r is None else float(r["value"])
            row[f"accn_{key}"] = None if r is None else r["accn"]
        row["fcf"] = row["ocf"] - row["capex"] if _num(row["ocf"]) and _num(row["capex"]) else None
        if prices is not None and prices.close is not None and fy.start >= prices.first_date - pd.Timedelta(days=7) and fy.end <= prices.last_date + pd.Timedelta(days=7):
            d = prices.dividends
            row["dps_inputc"] = float(d.loc[(d["ex_date"] >= fy.start) & (d["ex_date"] <= fy.end), "amount"].sum())
        else:
            row["dps_inputc"] = None
        rows.append(row)
    return pd.DataFrame(rows)


COVERAGE_ITEMS = [
    ("revenue", "Revenue"), ("gross_profit", "Gross profit"), ("operating_income", "Operating income"), ("net_income", "Net income (parent)"),
    ("eps_diluted", "Diluted EPS"), ("diluted_shares", "Diluted shares"), ("ocf", "Operating cash flow"), ("capex", "Capex"), ("da", "D&A"),
    ("dividends_paid", "Dividends paid"), ("cash", "Cash"), ("total_assets", "Total assets"), ("total_liabilities", "Total liabilities"),
    ("equity", "Parent equity"), ("long_term_debt", "Long-term debt"),
]


def _coverage(cal: Calendar, tables, cf: CompanyFacts, prices, as_of) -> tuple[pd.DataFrame, pd.DataFrame]:
    years = [fy for fy in cal.years if fy.end is not None and fy.end <= as_of]
    cols = [str(fy.end.year) if sum(1 for y in years if y.end.year == fy.end.year) == 1 else str(fy.end.date()) for fy in years]
    matrix, summary = {}, []
    for key, label in COVERAGE_ITEMS:
        marks = []
        for fy in years:
            r = tables[key].get(fy.start, fy.end) if METRICS[key].kind == "duration" else tables[key].get(None, fy.end)
            marks.append("✓" if r is not None else "")
        matrix[label] = marks
        have = [fy for fy, m in zip(years, marks) if m]
        summary.append({"item": label, "first": have[0].label if have else "", "last": have[-1].label if have else "", "years": len(have)})
    cover = cf.facts[(cf.facts["tag"] == COVER_SHARES_TAG)]
    matrix["Cover-page shares"] = ["✓" if ((cover["filed"] >= fy.start) & (cover["filed"] <= fy.end + pd.Timedelta(days=120))).any() else "" for fy in years]
    if prices is not None and prices.close is not None:
        marks = []
        for fy in years:
            expected = expected_trading_days(fy.start, fy.end)
            have = prices.close[(prices.close.index >= fy.start) & (prices.close.index <= fy.end)]
            share = len(have) / len(expected) if len(expected) else 0
            marks.append("✓" if share >= 0.95 else ("partial" if share > 0 else ""))
        matrix["Prices (Input C)"] = marks
        summary.append({"item": "Prices (Input C)", "first": str(prices.first_date.date()), "last": str(prices.last_date.date()), "years": sum(1 for m in marks if m)})
    else:
        matrix["Prices (Input C)"] = [""] * len(years)
        summary.append({"item": "Prices (Input C)", "first": "", "last": "", "years": 0})
    return pd.DataFrame(matrix, index=cols).T, pd.DataFrame(summary)


def _missing_periods(cal: Calendar, ttm: pd.DataFrame, dq: DQLog) -> None:
    annual_only = []
    for fy in cal.years:
        missing = [k for k in (1, 2, 3, 4) if k not in fy.quarter_ends]
        if fy.end is not None and missing == [1, 2, 3]:
            annual_only.append(fy)
        elif fy.end is not None and missing and len(missing) < 4:
            dq.add("missing_period", "fiscal quarter end not identified", f"{fy.label}: no period data ends Q{', Q'.join(map(str, missing))}; those quarters and the TTM values that need them are blank.", period=fy.end)
    if annual_only:
        dq.add(
            "missing_period",
            "annual-only reporting",
            f"{len(annual_only)} fiscal year(s) ({annual_only[0].label} to {annual_only[-1].label}) have no interim XBRL periods (e.g. 20-F/40-F filers). "
            "Those years contribute one fiscal-year-end valuation date each (TTM = FY); no quarters are derived.",
        )


def _growth(annual: pd.DataFrame, ttm: pd.DataFrame) -> pd.DataFrame:
    if annual.empty:
        return pd.DataFrame()
    items = [("revenue", "Revenue"), ("operating_income", "Operating income"), ("net_income", "Net income"), ("fcf", "Free cash flow"), ("diluted_shares", "Diluted shares"), ("eps_diluted", "Diluted EPS")]
    rows = []
    a = annual.reset_index(drop=True)
    last = len(a) - 1
    for key, label in items:
        row = {"metric": label}
        for n in (5, 10, 15):
            j = last - n
            if j < 0:
                row[f"cagr_{n}"], row[f"note_{n}"] = None, f"fewer than {n + 1} fiscal years"
                continue
            v1, v0 = a.at[last, key], a.at[j, key]
            if not (_num(v1) and _num(v0)):
                row[f"cagr_{n}"], row[f"note_{n}"] = None, "endpoint missing"
            elif v1 <= 0 or v0 <= 0:
                row[f"cagr_{n}"], row[f"note_{n}"] = None, "non-positive endpoint"
            else:
                years = (a.at[last, "fy_end"] - a.at[j, "fy_end"]).days / 365.25
                row[f"cagr_{n}"] = (v1 / v0) ** (1 / n) - 1
                notes = []
                if abs(years - n) > 0.1:
                    notes.append(f"span {years:.2f} years (fiscal calendar change)")
                weeks = {a.at[last, "weeks"], a.at[j, "weeks"]}
                if "53-week" in weeks and len(weeks) > 1:
                    notes.append("one endpoint is a 53-week year")
                row[f"note_{n}"] = "; ".join(notes)
            row[f"from_{n}"] = str(a.at[j, "fy_end"].date()) if j >= 0 else ""
        row["to"] = str(a.at[last, "fy_end"].date())
        rows.append(row)
    return pd.DataFrame(rows)


def _latest_one_time(ttm: pd.DataFrame) -> dict:
    if ttm.empty:
        return {}
    last = ttm.iloc[-1]
    return {
        "net_income": last.get("ttm_net_income"),
        "impairment": last.get("ttm_impairment"),
        "discontinued operations": last.get("ttm_disc_ops"),
        "disposal gains": last.get("ttm_disposal_gains"),
        "_source": f"{last['form']} {last['accn']}",
    }


def _snapshot_inputs(inp: ReviewInputs, ttm: pd.DataFrame, bs: BalanceSheet, prices, price_block, as_of, dq) -> dict:
    out: dict = {"as_of": as_of}
    if inp.current_price is not None:
        out.update(price=float(inp.current_price), price_date=as_of, price_source="Input D current price", raw_price=float(inp.current_price), price_factor=1.0, price_reason="")
    elif prices is None:
        out.update(price=None, price_date=None, price_source="", price_reason="no price data")
    elif price_block:
        out.update(price=None, price_date=None, price_source="", price_reason=price_block)
    else:
        look = price_on_or_before(prices, as_of, allow_stale=True)
        out.update(price=look.close, price_date=look.date, price_source="last Input C close on or before the as-of date", raw_price=look.raw_close, price_factor=look.factor, price_reason=look.reason)
        if look.stale_days > 5:
            dq.add("stale_price", "current price", f"Last close {look.date.date()} is {look.stale_days} trading days before the as-of date {as_of.date()}.", period=look.date, severity="warning")
    out.update(bs.at(as_of))
    out["ttm_row"] = len(ttm) - 1 if not ttm.empty else None
    _market_values(out)
    return out


SNAPSHOT_FIELDS = [
    ("ttm_revenue", "TTM revenue", "money"), ("ttm_gross_profit", "TTM gross profit", "money"), ("ttm_operating_income", "TTM operating income (EBIT)", "money"),
    ("ttm_net_income", "TTM net income attributable to parent", "money"), ("ttm_ocf", "TTM operating cash flow", "money"), ("ttm_capex", "TTM capex", "money"),
    ("ttm_fcf", "TTM free cash flow (OCF - capex)", "money"), ("ttm_da", "TTM D&A (cash flow statement)", "money"), ("ttm_ebitda", "TTM EBITDA (operating income + D&A)", "money"),
    ("ttm_dividends_paid", "TTM dividends paid", "money"), ("avg_diluted_shares", "TTM average diluted shares", "shares"), ("eps_ttm", "TTM diluted EPS", "per_share"),
    ("revenue_ps", "Revenue per share", "per_share"), ("gp_ps", "Gross profit per share", "per_share"), ("oi_ps", "Operating income per share", "per_share"),
    ("ocf_ps", "Operating cash flow per share", "per_share"), ("fcf_ps", "Free cash flow per share", "per_share"), ("da_ps", "D&A per share", "per_share"),
    ("ebitda_ps", "EBITDA per share", "per_share"), ("dps_ttm", "TTM dividends per share (Input C)", "per_share"), ("eps_cagr5", "5-year TTM EPS CAGR", "pct"),
]


def _text(src: dict, key: str) -> str:
    """A reason or note field as text; NaN and None become ''."""
    v = src.get(key)
    return v if isinstance(v, str) else ""


def ratio_inputs(row: dict | pd.Series, snap: dict | None = None) -> dict:
    """Map a TTM row (and optional snapshot market inputs) to ratio numerator/denominator names."""
    src = dict(row)
    if snap is not None:
        for key in ("price", "mcap", "ev", "equity", "tbv", "shares_out", "cash", "total_borrowings", "nci", "preferred", "price_reason", "ev_reason", "tbv_reason"):
            src[key] = snap.get(key)
        so, eq = snap.get("shares_out"), snap.get("equity")
        src["bvps"] = eq / so if _num(eq) and _num(so) and so > 0 else None
    clean = {k: (float(v) if _num(v) else (None if not isinstance(v, str) else v)) for k, v in src.items()}
    return {
        "price": clean.get("price"), "mcap": clean.get("mcap"), "ev": clean.get("ev"),
        "eps": clean.get("eps_ttm"), "revenue": clean.get("ttm_revenue"), "gross_profit": clean.get("ttm_gross_profit"),
        "equity": clean.get("equity"), "tbv": clean.get("tbv"), "ocf": clean.get("ttm_ocf"), "fcf": clean.get("ttm_fcf"),
        "dps": clean.get("dps_ttm"), "operating_income": clean.get("ttm_operating_income"), "ebitda": clean.get("ttm_ebitda"),
        "bvps": clean.get("bvps"), "cagr": clean.get("eps_cagr5"), "cagr_reason": _text(src, "eps_cagr5_reason"),
        "price_reason": _text(src, "price_reason") or "price unavailable",
        "mcap_reason": _text(src, "price_reason") or ("no shares outstanding available for market cap" if not _num(src.get("shares_out")) else "market cap unavailable"),
        "ev_reason": _text(src, "ev_reason") or "EV unavailable", "tbv_reason": _text(src, "tbv_reason") or "TBV unavailable",
        "dps_reason": _text(src, "dps_reason") or "TTM dividends per share unavailable",
        "eps_reason": "TTM EPS unavailable" + (f" ({_text(src, 'shares_note') or 'no diluted share count'})" if not _num(src.get("avg_diluted_shares")) else " (net income missing)"),
        "revenue_reason": "TTM revenue unavailable", "gross_profit_reason": "TTM gross profit unavailable",
        "equity_reason": "parent equity unavailable", "ocf_reason": "TTM operating cash flow unavailable", "fcf_reason": _text(src, "fcf_reason") or "TTM FCF unavailable",
        "operating_income_reason": _text(src, "operating_income_reason") or "TTM operating income unavailable",
        "ebitda_reason": _text(src, "ebitda_reason") or "TTM EBITDA unavailable",
        "bvps_reason": "BVPS unavailable",
    }


def _snapshot_table(review: Review) -> tuple[pd.DataFrame, dict]:
    ttm, snap = review.ttm, review.snapshot_inputs
    if ttm.empty:
        return pd.DataFrame(), {}
    last = ttm.iloc[-1]
    rows = []
    for key, label, unit in SNAPSHOT_FIELDS:
        rows.append({"group": "TTM / per-share TTM", "key": key, "item": label, "value": last.get(key), "unit": unit, "valuation_date": last["valuation_date"], "inputs": f"TTM ending {last['quarter_end'].date()} ({last['form']} {last['accn']})", "formula": "", "shares": "", "percentile": None, "reason": last.get("eps_cagr5_reason") if key == "eps_cagr5" else ""})
    so, eq = snap.get("shares_out"), snap.get("equity")
    bvps = eq / so if _num(eq) and _num(so) and so > 0 else None
    rows.append({"group": "Balance sheet / market", "key": "bvps", "item": "Book value per share", "value": bvps, "unit": "per_share", "valuation_date": snap.get("bs_filed"), "inputs": f"equity at {snap.get('bs_instant').date() if snap.get('bs_instant') is not None else '-'} / cover-page shares", "formula": "Parent equity / shares outstanding (cover page)", "shares": "cover-page shares outstanding", "percentile": None, "reason": ""})
    for key, label in (("price", "Price"), ("mcap", "Market cap"), ("ev", "Enterprise value")):
        rows.append({"group": "Balance sheet / market", "key": key, "item": label, "value": snap.get(key), "unit": "per_share" if key == "price" else "money", "valuation_date": snap.get("price_date"), "inputs": snap.get("price_source", ""), "formula": {"price": "", "mcap": "Price x shares outstanding (cover page)", "ev": "Market cap + borrowings + NCI + preferred - cash"}[key], "shares": "cover-page shares outstanding" if key != "price" else "", "percentile": None, "reason": snap.get("price_reason") if key == "price" else snap.get("ev_reason", "") if key == "ev" else ""})
    inputs = ratio_inputs(last, snap)
    values = compute_all(inputs)
    percentiles = {}
    daily = review.daily
    for ratio in RATIOS:
        value, reason = values[ratio.key]
        pct = None
        if ratio.daily and daily is not None and ratio.key in daily and _num(value):
            hist = daily[ratio.key].dropna()
            if len(hist):
                pct = 100.0 * float((hist <= value).sum()) / len(hist)
                percentiles[ratio.key] = {"percentile": pct, "n": int(len(hist)), "from": hist.index.min(), "to": hist.index.max()}
        shares_note = ratio.shares
        source = snap.get("shares_source") or ""
        if source and not source.startswith("dei:"):
            shares_note = shares_note.replace("cover-page shares outstanding", source)
        rows.append({"group": "Price ratios", "key": ratio.key, "item": ratio.label, "value": value, "unit": ratio.kind, "valuation_date": last["valuation_date"], "inputs": f"price {snap.get('price_date').date() if snap.get('price_date') is not None else '-'}; TTM ending {last['quarter_end'].date()}; balance sheet {snap.get('bs_instant').date() if snap.get('bs_instant') is not None else '-'}", "formula": ratio.formula, "shares": shares_note, "percentile": pct, "reason": reason})
    return pd.DataFrame(rows), percentiles


def _flag_extremes(review: Review) -> None:
    for _, row in review.snapshot.iterrows():
        if row["group"] == "Price ratios" and row["unit"] == "multiple" and _num(row["value"]) and abs(row["value"]) > 100:
            review.dq.add("extreme_multiple", row["item"], f"Current {row['item']} = {row['value']:,.1f}x (> 100x); check inputs: {row['inputs']}.", severity="warning")
    if review.ttm.empty:
        return
    for i, last in review.ttm.iterrows():
        values = compute_all(ratio_inputs(last))
        for ratio in RATIOS:
            v = values[ratio.key][0]
            if is_extreme(ratio, v):
                review.dq.add("extreme_multiple", ratio.label, f"{ratio.label} = {v:,.1f}x at valuation date {last['valuation_date'].date()} (TTM ending {last['quarter_end'].date()}); check inputs.", period=last["valuation_date"], filing=last["accn"])


def _header(review: Review, currencies: list[str]) -> dict:
    cf, inp = review.cf, review.inputs
    periodic = cf.filings.dropna(subset=["report_period"])
    latest = periodic.iloc[-1] if not periodic.empty else cf.filings.iloc[-1]
    regimes = sorted(set(review.ttm["regime"])) if not review.ttm.empty else []
    full_start = None
    if not review.ttm.empty:
        sub = review.ttm.iloc[review.stats_start_index:]
        have = sub[[s.key for s in SERIES if s.key in sub]].notna().any(axis=1)
        if have.any():
            full_start = sub.loc[have.idxmax(), "valuation_date"]
    p = review.prices
    meta = inp.price_meta
    return {
        "company": cf.entity_name,
        "ticker": inp.ticker or meta.ticker,
        "cik": cf.cik,
        "taxonomy": ", ".join(cf.taxonomies),
        "currency": ", ".join(currencies),
        "regimes": regimes,
        "latest_filing": f"{latest['form']} {latest['accn']} filed {latest['filed'].date()}",
        "as_of": review.snapshot_inputs["as_of"],
        "display_window": f"{review.display_start.date()} to {review.snapshot_inputs['as_of'].date()}" if review.display_start is not None else "n/a",
        "full_history_start": full_start,
        "price_range": f"{p.first_date.date()} to {p.last_date.date()}" if p is not None else "no price data",
        "price_basis": (
            f"split-adjusted: {_yn(meta.split_adjusted)}; dividend-adjusted: {_yn(meta.dividend_adjusted)}; currency: {meta.currency or 'not stated'}"
            if p is not None
            else "n/a"
        ),
        "splits": review.basis.describe_within(review.cf.facts["filed"].min()),
    }


def _yn(v):
    return "not stated" if v is None else ("yes" if v else "no")


# ---------------------------------------------------------------- daily

def daily_table(review: Review) -> pd.DataFrame | None:
    prices, ttm = review.prices, review.ttm
    if prices is None or prices.close is None or ttm.empty or review.price_block_reason:
        return None
    rows = ttm.dropna(subset=["valuation_date"]).sort_values("valuation_date").reset_index(drop=True)
    rows = rows.drop_duplicates("valuation_date", keep="last")
    rows["held_row"] = rows.index
    close = prices.close
    days = pd.DataFrame({"date": close.index, "price": close.to_numpy(), "raw_close": prices.close_raw.reindex(close.index).to_numpy()})
    days = days[days["date"] >= rows["valuation_date"].min()]
    keep = [
        "valuation_date", "quarter_end", "form", "accn", "held_row", "currency_ok", "shares_out", "cash", "total_borrowings", "nci", "preferred", "equity", "tbv",
        "eps_ttm", "ttm_revenue", "ttm_gross_profit", "ttm_ocf", "ttm_fcf", "dps_ttm", "ttm_operating_income", "ttm_ebitda", "eps_cagr5",
    ] + [s.key for s in SERIES if s.key not in ("eps_ttm", "ttm_revenue", "ttm_gross_profit", "ttm_ocf", "ttm_fcf", "dps_ttm", "ttm_operating_income", "ttm_ebitda", "equity")]
    keep = list(dict.fromkeys(k for k in keep if k in rows))
    days["date"] = days["date"].astype("datetime64[ns]")
    right = rows[keep].copy()
    right["valuation_date"] = right["valuation_date"].astype("datetime64[ns]")
    merged = pd.merge_asof(days.sort_values("date"), right.sort_values("valuation_date"), left_on="date", right_on="valuation_date", direction="backward")
    num = lambda c: pd.to_numeric(merged[c], errors="coerce")  # noqa: E731
    merged["mcap"] = merged["price"] * num("shares_out")
    merged["ev"] = merged["mcap"] + num("total_borrowings").fillna(0) + num("nci").fillna(0) + num("preferred").fillna(0) - num("cash")
    uncertain = merged["date"].map(lambda d: bool(prices.uncertain_days.get(d, False)))
    if "currency_ok" in merged:
        uncertain = uncertain | ~merged["currency_ok"].fillna(True).astype(bool)  # no FX input: leave blank
    src = {
        "price": merged["price"], "mcap": merged["mcap"], "ev": merged["ev"], "eps": num("eps_ttm"), "revenue": num("ttm_revenue"),
        "gross_profit": num("ttm_gross_profit"), "equity": num("equity"), "tbv": num("tbv"), "ocf": num("ttm_ocf"), "fcf": num("ttm_fcf"),
        "dps": num("dps_ttm"), "operating_income": num("ttm_operating_income"), "ebitda": num("ttm_ebitda"),
    }
    for ratio in DAILY_RATIOS:
        if ratio.key == "peg":
            continue
        n, d = src[ratio.numerator], src[ratio.denominator]
        value = n / d
        nm = d <= 0
        value = value.where(~nm & n.notna() & d.notna())
        merged[ratio.key] = value.where(~uncertain)
        merged[f"{ratio.key}_nm"] = nm & n.notna() & ~uncertain
    cagr = num("eps_cagr5")
    pe = merged["pe"]
    merged["peg"] = (pe / (cagr * 100)).where((cagr > 0) & pe.notna())
    merged["peg_nm"] = merged["pe_nm"] & (cagr > 0)
    merged["basis_uncertain"] = uncertain
    merged["in_display_window"] = merged["date"] >= review.display_start if review.display_start is not None else True
    return merged.set_index("date")


def _daily_counts(review: Review) -> dict:
    daily = review.daily
    if daily is None:
        return {}
    window = daily[daily["in_display_window"]]
    return {r.key: int(window[r.key].notna().sum()) for r in DAILY_RATIOS}

# ============================== valuation/sources.py ==============================
"""Fetch Inputs A and C for a ticker into memory.

Input A: SEC companyfacts JSON (data.sec.gov). Input C: Yahoo Finance chart
API daily history. Its `close` is split-adjusted and not dividend-adjusted;
dividends are cash amounts on their ex-dates (on the same split basis) and
split events carry their ex-dates. Nothing is written to disk here.

SEC asks automated clients to identify themselves. Set SEC_USER_AGENT to your
own "Name contact@domain" string; the default is a generic client name.
"""


import os
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib import resources
from pathlib import Path

import pandas as pd
import requests


SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_FACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
YAHOO_CHART_URL = "https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
DEFAULT_SEC_AGENT = "ValuationReview research-client admin@example.com"


def sec_headers() -> dict:
    return {"User-Agent": os.environ.get("SEC_USER_AGENT", DEFAULT_SEC_AGENT), "Accept-Encoding": "gzip, deflate"}


def _get(url: str, headers: dict, params: dict | None = None, attempts: int = 4, timeout: int = 60) -> requests.Response:
    delay = 1.0
    last_error = None
    for _ in range(attempts):
        try:
            response = requests.get(url, headers=headers, params=params, timeout=timeout)
            if response.status_code == 200:
                return response
            if response.status_code == 404:
                response.raise_for_status()
            last_error = RuntimeError(f"HTTP {response.status_code} from {url}")
        except requests.HTTPError:
            raise
        except requests.RequestException as exc:  # network hiccup: retry with backoff
            last_error = exc
        time.sleep(delay)
        delay *= 2
    raise RuntimeError(f"Could not fetch {url}: {last_error}")


# ------------------------------------------------------------------ ticker lists

def default_tickers() -> list[str]:
    text = resources.files("valuation").joinpath("default_tickers.txt").read_text(encoding="utf-8")
    return parse_ticker_list(text)


def parse_ticker_list(text: str) -> list[str]:
    """Accept one-per-line, comma-separated or quoted lists (e.g. 'AAPL','MSFT')."""
    out: list[str] = []
    for line in text.splitlines():
        line = line.split("#", 1)[0]
        for part in line.replace(";", ",").replace("\t", ",").split(","):
            for token in part.split():
                ticker = token.strip().strip("'\"[]()").upper()
                if ticker and ticker not in out:
                    out.append(ticker)
    return out


def read_ticker_file(path: str | Path) -> list[str]:
    return parse_ticker_list(Path(path).read_text(encoding="utf-8"))


_TICKER_MAP: dict[str, dict] | None = None


def ticker_map() -> dict[str, dict]:
    """Ticker -> {cik, title} from SEC's company_tickers.json (cached in memory)."""
    global _TICKER_MAP
    if _TICKER_MAP is None:
        data = _get(SEC_TICKERS_URL, sec_headers()).json()
        _TICKER_MAP = {}
        for item in data.values():
            _TICKER_MAP[str(item["ticker"]).upper()] = {"cik": str(item["cik_str"]).zfill(10), "title": item.get("title", "")}
    return _TICKER_MAP


def resolve(identifier: str) -> tuple[str, str]:
    """Ticker or CIK -> (ticker, 10-digit CIK)."""
    ident = identifier.strip().upper()
    if ident.isdigit():
        cik = ident.zfill(10)
        ticker = next((t for t, v in ticker_map().items() if v["cik"] == cik), "")
        return ticker, cik
    mapping = ticker_map()
    for candidate in (ident, ident.replace(".", "-"), ident.replace("-", ".")):
        if candidate in mapping:
            return candidate, mapping[candidate]["cik"]
    raise KeyError(
        f"{identifier}: not found in SEC company_tickers.json (the ticker may have changed or the company may no longer file); "
        "enter its CIK instead or update the ticker list"
    )


# ------------------------------------------------------------------ Input A

@dataclass
class FetchedFacts:
    cik: str
    data: dict
    url: str
    retrieved: str


def fetch_companyfacts(cik: str) -> FetchedFacts:
    url = SEC_FACTS_URL.format(cik=str(cik).zfill(10))
    response = _get(url, sec_headers())
    return FetchedFacts(str(cik).zfill(10), response.json(), url, datetime.now(timezone.utc).isoformat(timespec="seconds"))


# ------------------------------------------------------------------ Input C

@dataclass
class FetchedPrices:
    symbol: str
    frame: pd.DataFrame  # date, close, dividend, split_ratio
    meta: PriceMeta
    url: str
    retrieved: str
    exchange: str


def yahoo_symbol(ticker: str) -> str:
    return ticker.replace(".", "-")


def fetch_prices(ticker: str, start: str = "1990-01-01") -> FetchedPrices:
    symbol = yahoo_symbol(ticker)
    url = YAHOO_CHART_URL.format(symbol=symbol)
    params = {
        "period1": int(pd.Timestamp(start, tz="UTC").timestamp()),
        "period2": int(time.time()) + 86400,
        "interval": "1d",
        "events": "div,splits",
        "includeAdjustedClose": "true",
    }
    response = _get(url, {"User-Agent": "Mozilla/5.0"}, params=params)
    payload = response.json().get("chart", {})
    if payload.get("error") or not payload.get("result"):
        raise RuntimeError(f"Yahoo returned no data for {symbol}: {payload.get('error')}")
    result = payload["result"][0]
    meta = result.get("meta", {})
    tz = meta.get("exchangeTimezoneName") or "America/New_York"

    def to_dates(seconds) -> pd.DatetimeIndex:
        return pd.to_datetime(seconds, unit="s", utc=True).tz_convert(tz).tz_localize(None).normalize()

    stamps = result.get("timestamp") or []
    closes = (result.get("indicators", {}).get("quote") or [{}])[0].get("close") or []
    frame = pd.DataFrame({"date": to_dates(stamps), "close": pd.to_numeric(pd.Series(closes, dtype="float64"), errors="coerce")})
    frame = frame.drop_duplicates("date", keep="last")
    events = result.get("events", {}) or {}
    dividends = events.get("dividends", {}) or {}
    splits = events.get("splits", {}) or {}
    div = pd.DataFrame({"date": to_dates([d["date"] for d in dividends.values()]), "dividend": [float(d["amount"]) for d in dividends.values()]}) if dividends else pd.DataFrame(columns=["date", "dividend"])
    spl = (
        pd.DataFrame({"date": to_dates([s["date"] for s in splits.values()]), "split_ratio": [float(s["numerator"]) / float(s["denominator"]) for s in splits.values()]})
        if splits
        else pd.DataFrame(columns=["date", "split_ratio"])
    )
    frame = frame.merge(div.groupby("date", as_index=False)["dividend"].sum(), on="date", how="outer") if not div.empty else frame.assign(dividend=0.0)
    frame = frame.merge(spl.groupby("date", as_index=False)["split_ratio"].prod(), on="date", how="outer") if not spl.empty else frame.assign(split_ratio=float("nan"))
    frame = frame.sort_values("date").reset_index(drop=True)
    price_meta = PriceMeta(ticker=ticker, split_adjusted=True, dividend_adjusted=False, currency=str(meta.get("currency") or "").upper())
    return FetchedPrices(symbol, frame, price_meta, response.url.split("?")[0], datetime.now(timezone.utc).isoformat(timespec="seconds"), str(meta.get("exchangeName") or ""))


DEFAULT_TICKERS_TEXT = '# Default ticker list for the valuation review (one per line). Replace or pass --tickers to use another list.\nTTD\nCHTR\nSMCI\nCOIN\nIT\nFDS\nGDDY\nCSGP\nARE\nPAYC\nBAX\nCMG\nNFLX\nMOS\nNOW\nCARR\nHRL\nKMB\nVRSK\nCDW\nLYB\nBRO\nPYPL\nAMT\nSWKS\nDVA\nSBAC\nPAYX\nZBRA\nDXCM\nFTNT\nROP\nMOH\nERIE\nTYL\nZTS\nKVUE\nCCI\nLW\nPOOL\nINVH\nMDLZ\nVICI\nKDP\nIP\nCPRT\nCLX\nFIS\nHPQ\nKR\nAJG\nORCL\nALGN\nVST\nSTZ\nCMCSA\nLULU\nEFX\nLII\nAXON\nEOG\nDPZ\nCF\nPGR\nINTU\nCHD\nMMC\nTXN\nCPB\nGIS\nT\nCAG\nADP\nDOW\nVRSN\nESS\nNWS\nTT\nTMUS\nAMCR\nPTC\nNKE\nRSG\nKHC\nAVB\nUBER\nMGM\nTPL\nCTAS\nNWSA\nSW\nPG\nAPD\nTSCO\nAWK\nMETA\nFFIV\nPM\nOKE\nADBE\nCL\nMAA\nEQR\nIRM\nDOC\nIFF\nPPG\nOTIS\nCI\nWDAY\nDASH\nETN\nGEN\nUDR\nRCL\nDDOG\nICE\nAIG\nMSI\nCTVA\nAZO\nBLDR\nSNPS\nHON\nDLR\nSYK\nSO\nSBUX\nOXY\nMKC\nEXR\nAMP\nBR\nEQT\nDE\nBSX\nKKR\nABT\nDELL\nLIN\nPSKY\nDRI\nBALL\nDIS\nCOST\nCME\nKIM\nMSFT\nPNR\nPEG\nBBY\nMTCH\nSYY\nCPT\nTDG\nBKNG\nBX\nSJM\nVZ\nPSA\nFAST\nUNP\nIR\nADSK\nEXE\nCRWD\nTAP\nMPC\nBXP\nVRTX\nDTE\nPODD\nITW\nACN\nGRMN\nPNW\nZBH\nKMI\nARES\nAON\nRMD\nEG\nWM\nGPN\nED\nMO\nGWW\nWY\nV\nBLK\nAPO\nPANW\nCRM\nNVR\nNRG\nCPAY\nPFE\nDUK\nDECK\nCTRA\nWRB\nWEC\nNTAP\nCMS\nEXC\nLH\nVLTO\nIEX\nD\nTRMB\nMRNA\nAOS\nMSCI\nMCHP\nHD\nSHW\nGNRC\nREG\nTGT\nPPL\nKO\nLYV\nXYZ\nODFL\nMET\nCDNS\nORLY\nABNB\nDGX\nGPC\nECL\nRVTY\nAVY\nBEN\nYUM\nMA\nO\nCOP\nSPGI\nJBL\nLEN\nEMR\nES\nMAS\nJ\nPEP\nMCD\nFANG\nNXPI\nTRGP\nFTV\nBA\nMTB\nAEE\nLNT\nHBAN\nTDY\nHPE\nHSIC\nTROW\nJCI\nWMB\nON\nCNP\nRJF\nEVRG\nCSX\nWAB\nUPS\nMMM\nMCO\nFICO\nCVX\nWYNN\nDVN\nJKHY\nEQIX\nGILD\nPSX\nIBM\nPKG\nNCLH\nDOV\nNI\nXYL\nTSN\nTTWO\nWTW\nAFL\nNEE\nALLE\nFRT\nCSCO\nACGL\nSCHW\nOMC\nELV\nEW\nATO\nNDAQ\nXOM\nXEL\nAMGN\nCOO\nALL\nMDT\nROL\nNSC\nCEG\nCB\nPNC\nTXT\nTRV\nFE\nPRU\nCINF\nAEP\nCCL\nPFG\nAKAM\nCTSH\nPWR\nHST\nTFC\nAMZN\nSTE\nCBOE\nHUBB\nISRG\nSWK\nHIG\nQCOM\nFITB\nHOLX\nWAT\nHAS\nUHS\nDHI\nNVDA\nWSM\nADM\nABBV\nCOF\nCOR\nSNA\nSPG\nSOLV\nETR\nGEHC\nMCK\nBDX\nJPM\nNUE\nURI\nKEY\nPHM\nRF\nHSY\nUNH\nNTRS\nVMC\nBMY\nAES\nGEV\nTKO\nAXP\nL\nANET\nDHR\nUSB\nMAR\nWFC\nGD\nLVS\nHOOD\nLMT\nNDSN\nPCG\nCBRE\nLOW\nEBAY\nEME\nMLM\nVTR\nSRE\nBAC\nEIX\nROK\nDAY\nGL\nLUV\nF\nEPAM\nHUM\nPLD\nIBKR\nA\nVLO\nFCX\nMTD\nWST\nSTT\nNOC\nWMT\nTECH\nPCAR\nLDOS\nWELL\nAAPL\nSYF\nDLTR\nHWM\nCVS\nADI\nSTLD\nSLB\nEL\nAPA\nCAH\nPLTR\nHCA\nAVGO\nAIZ\nAPTV\nKEYS\nBKR\nCFG\nBK\nTJX\nDG\nGE\nDAL\nRL\nFOX\nUAL\nRTX\nPH\nLHX\nMPWR\nJNJ\nFOXA\nMS\nBG\nMNST\nFDX\nMRK\nIDXX\nTEL\nGS\nEXPD\nJBHT\nCVNA\nLLY\nEA\nCRL\nCRH\nDD\nTPR\nVTRS\nBIIB\nC\nULTA\nAMD\nREGN\nAPH\nCNC\nTSLA\nTMO\nFSLR\nROST\nIQV\nHAL\nHII\nKLAC\nCAT\nINCY\nAMAT\nGM\nEXPE\nCMI\nGLW\nIVZ\nCHRW\nNEM\nGOOG\nGOOGL\nAPP\nFIX\nINTC\nSTX\nLRCX\nALB\nTER\nWBD\nMU\nWDC\nSNDK\n'


def default_tickers() -> list[str]:
    return parse_ticker_list(DEFAULT_TICKERS_TEXT)


# cli.py refers to these as sources.<name>; in the single cell they are plain globals.
import types as _types

sources = _types.SimpleNamespace(
    resolve=resolve, fetch_companyfacts=fetch_companyfacts, fetch_prices=fetch_prices,
    default_tickers=default_tickers, read_ticker_file=read_ticker_file, parse_ticker_list=parse_ticker_list, ticker_map=ticker_map,
)

# ============================== valuation/workbook.py ==============================
"""Excel workbook with live formulas, built in memory.

Sheets (required): Snapshot, TTM_Quarterly, TTM_Stats, Ratios_Quarterly,
Valuation_Inputs, Data_Quality. Two supporting sheets keep every number
auditable: TTM_Build (the reported components behind each TTM value) and
Stats_Helpers (running counts and record-high flags used by TTM_Stats).
Daily values are never written here.

Colour convention: blue = value taken from Input A/C/D, black = formula,
green = link to another sheet.
"""


import io
import math

import pandas as pd
from openpyxl import Workbook
from openpyxl.comments import Comment
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


FONT = "Arial"
BLUE = Font(name=FONT, size=10, color="0000FF")
BLACK = Font(name=FONT, size=10, color="000000")
GREEN = Font(name=FONT, size=10, color="008000")
BOLD = Font(name=FONT, size=10, bold=True)
TITLE = Font(name=FONT, size=12, bold=True)
HEAD_FILL = PatternFill("solid", fgColor="DDE4EE")
NOTE = Font(name=FONT, size=9, italic=True, color="555555")
EXTREME_FILL = PatternFill("solid", fgColor="FCE4D6")

FMT = {
    "money": "#,##0;(#,##0);0",
    "shares": "#,##0",
    "per_share": "0.00;(0.00);0.00",
    "multiple": '0.0"x";(0.0"x");0.0"x"',
    "yield": "0.00%;(0.00%);0.00%",
    "ratio": "0.0%;(0.0%);0.0%",
    "pct": "0.0%;(0.0%);0.0%",
    "date": "yyyy-mm-dd",
    "count": "0",
    "factor": "0.######",
    "value": "0.00",
}


def _num(v) -> bool:
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(float(v))


def _val(v):
    if v is None:
        return None
    if isinstance(v, pd.Timestamp):
        return None if pd.isna(v) else v.to_pydatetime()
    if isinstance(v, float) and math.isnan(v):
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v


def _s(v) -> str:
    """Text field from a DataFrame row; NaN and None become ''."""
    return v if isinstance(v, str) else ""


def _q(sheet: str) -> str:
    return f"'{sheet}'"


class Sheet:
    def __init__(self, ws):
        self.ws = ws

    def header(self, row: int, labels: list[str], widths: list[int] | None = None):
        for c, label in enumerate(labels, start=1):
            cell = self.ws.cell(row=row, column=c, value=label)
            cell.font = BOLD
            cell.fill = HEAD_FILL
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        if widths:
            for c, w in enumerate(widths, start=1):
                self.ws.column_dimensions[get_column_letter(c)].width = w

    def put(self, row: int, col: int, value, fmt: str | None = None, font: Font = BLUE):
        cell = self.ws.cell(row=row, column=col, value=_val(value))
        if isinstance(value, str) and value.startswith("="):
            cell.font = GREEN if "!" in value and value.count("!") == 1 and value[1] == "'" and "(" not in value else BLACK
        else:
            cell.font = font
        if fmt:
            cell.number_format = FMT.get(fmt, fmt)
        return cell


def build_workbook(review) -> bytes:
    wb = Workbook()
    wb.remove(wb.active)
    names = ["Snapshot", "TTM_Quarterly", "TTM_Stats", "Ratios_Quarterly", "Valuation_Inputs", "Data_Quality", "TTM_Build", "Stats_Helpers"]
    sheets = {n: Sheet(wb.create_sheet(n)) for n in names}
    ttm = review.ttm.reset_index(drop=True)
    n = len(ttm)
    first, last = 2, n + 1  # data rows on the per-valuation-date sheets

    build_rows = _write_build(sheets["TTM_Build"], review)
    vi_cols, div_cols = _write_valuation_inputs(sheets["Valuation_Inputs"], review)
    tq_cols = _write_ttm_quarterly(sheets["TTM_Quarterly"], review, build_rows, vi_cols, div_cols)
    _write_ratios(sheets["Ratios_Quarterly"], review, tq_cols, vi_cols)
    helper_cols = _write_helpers(sheets["Stats_Helpers"], review, tq_cols)
    _write_stats(sheets["TTM_Stats"], review, tq_cols, helper_cols)
    _write_snapshot(sheets["Snapshot"], review, tq_cols, last)
    _write_dq(sheets["Data_Quality"], review)
    del first
    buffer = io.BytesIO()
    wb.save(buffer)
    return buffer.getvalue()


# ---------------------------------------------------------------- TTM_Build

BUILD_METRICS = ["revenue", "gross_profit", "operating_income", "net_income", "ocf", "capex", "da", "dividends_paid", "diluted_shares"]


def _write_build(sh: Sheet, review) -> dict[tuple[int, str], int]:
    ws = sh.ws
    ws["A1"] = "TTM_Build: reported components (Input A, latest restated, split-normalized) and the live formula for each TTM value"
    ws["A1"].font = TITLE
    sh.header(2, ["Quarter end", "Valuation date", "Metric", "Method", "Component A", "Component B", "Component C", "Component D", "TTM value",
                  "Alt: sum of 4 quarters", "Alt gap", "A is", "B is", "C is", "D is", "Source filings", "Reason if blank"],
              [12, 12, 16, 13, 18, 18, 18, 18, 18, 18, 9, 26, 26, 26, 26, 50, 40])
    ws["A3"] = "Methods: FY = annual value; FY+YTD-PYTD = A + B - C; 4Q = A + B + C + D; avg4Q = AVERAGE(A:D) (shares); blank = not computable (see reason)."
    ws["A3"].font = NOTE
    rows: dict[tuple[int, str], int] = {}
    build = review.build
    if build.empty:
        return rows
    ttm = review.ttm.reset_index(drop=True)
    index_of = {(ttm.at[i, "valuation_date"], ttm.at[i, "quarter_end"]): i for i in range(len(ttm))}
    r = 4
    for metric in BUILD_METRICS:
        sub = build[build["metric"] == metric]
        for _, b in sub.iterrows():
            i = index_of.get((b["valuation_date"], b["quarter_end"]))
            if i is None:
                continue
            rows[(i, metric)] = r
            unit = "shares" if metric == "diluted_shares" else "money"
            sh.put(r, 1, b["quarter_end"], "date")
            sh.put(r, 2, b["valuation_date"], "date")
            sh.put(r, 3, metric, font=BLACK)
            sh.put(r, 4, _s(b["method"]), font=BLACK)
            for c, key in enumerate(["a", "b", "c", "d"], start=5):
                if _num(b[key]):
                    sh.put(r, c, float(b[key]), unit)
            method = b["method"]
            formula = {
                "FY": f"=E{r}",
                "FY+YTD-PYTD": f"=E{r}+F{r}-G{r}",
                "4Q": f"=E{r}+F{r}+G{r}+H{r}",
                "avg4Q": f"=AVERAGE(E{r}:H{r})",
            }.get(method)
            if formula:
                sh.put(r, 9, formula, unit)
            if _num(b["alt"]):
                sh.put(r, 10, float(b["alt"]), unit)
                sh.put(r, 11, f'=IF(AND(ISNUMBER(I{r}),ISNUMBER(J{r}),I{r}<>0),(I{r}-J{r})/ABS(I{r}),"")', "pct")
            for c, key in enumerate(["a_label", "b_label", "c_label", "d_label"], start=12):
                sh.put(r, c, _s(b[key]), font=BLACK)
            sh.put(r, 16, _s(b["sources"]), font=BLACK)
            sh.put(r, 17, _s(b["reason"]), font=BLACK)
            r += 1
    ws.freeze_panes = "E4"
    return rows


# ---------------------------------------------------------------- Valuation_Inputs

VI_COLUMNS = [
    ("valuation_date", "Valuation date", "date"), ("quarter_end", "Quarter end", "date"), ("form", "Form", None), ("accn", "Accession", None),
    ("price_date", "Price date", "date"), ("raw_price", "Input C close (as supplied)", "per_share"), ("price_factor", "Split factor applied to close", "factor"),
    ("price", "Price (as-of split basis)", "per_share"), ("price_reason", "Price note", None),
    ("cover_shares_reported", "Cover-page shares (reported)", "shares"), ("cover_date", "Cover date", "date"), ("cover_accn", "Cover filing", None),
    ("share_factor", "Share split factor", "factor"), ("shares_out", "Shares outstanding (as-of basis)", "shares"),
    ("bs_instant", "Balance sheet date", "date"), ("bs_accn", "Balance sheet filing", None),
    ("cash", "Cash & equivalents", "money"), ("debt_current", "Borrowings, current", "money"), ("debt_noncurrent", "Borrowings, noncurrent", "money"),
    ("total_borrowings", "Total borrowings", "money"), ("nci", "Noncontrolling interest", "money"), ("preferred", "Preferred stock", "money"),
    ("equity", "Parent equity", "money"), ("goodwill", "Goodwill", "money"), ("intangibles", "Intangibles excl. goodwill", "money"), ("tbv", "Tangible book value", "money"),
    ("total_assets", "Total assets", "money"), ("total_liabilities", "Total liabilities", "money"), ("long_term_debt", "Long-term debt", "money"),
    ("mcap", "Market cap", "money"), ("ev", "Enterprise value", "money"), ("debt_tags", "Borrowing tags used", None), ("vi_notes", "Notes", None),
]


def _write_valuation_inputs(sh: Sheet, review) -> tuple[dict[str, str], tuple[str, str]]:
    ws = sh.ws
    cols = {key: get_column_letter(i) for i, (key, _, _) in enumerate(VI_COLUMNS, start=1)}
    sh.header(1, [label for _, label, _ in VI_COLUMNS], [12, 12, 8, 22, 12, 14, 12, 14, 30, 16, 12, 22, 10, 16, 12, 22] + [17] * 15 + [40, 50])
    ttm = review.ttm.reset_index(drop=True)
    c = cols
    for i, row in ttm.iterrows():
        r = i + 2
        for j, (key, _, fmt) in enumerate(VI_COLUMNS, start=1):
            if key == "price" and _num(row.get("raw_price")):
                sh.put(r, j, f'=IF(AND(ISNUMBER({c["raw_price"]}{r}),ISNUMBER({c["price_factor"]}{r})),{c["raw_price"]}{r}/{c["price_factor"]}{r},"")', fmt)
            elif key == "shares_out" and _num(row.get("cover_shares_reported")):
                sh.put(r, j, f'=IF(ISNUMBER({c["cover_shares_reported"]}{r}),{c["cover_shares_reported"]}{r}*{c["share_factor"]}{r},"")', fmt)
            elif key == "total_borrowings":
                sh.put(r, j, f'=IF(COUNT({c["debt_current"]}{r},{c["debt_noncurrent"]}{r})=0,"",SUM({c["debt_current"]}{r},{c["debt_noncurrent"]}{r}))', fmt)
            elif key == "tbv":
                sh.put(r, j, f'=IF(ISNUMBER({c["equity"]}{r}),IF(COUNT({c["goodwill"]}{r},{c["intangibles"]}{r})=0,"",{c["equity"]}{r}-SUM({c["goodwill"]}{r},{c["intangibles"]}{r})),"")', fmt)
            elif key == "mcap":
                sh.put(r, j, f'=IF(AND(ISNUMBER({c["price"]}{r}),ISNUMBER({c["shares_out"]}{r})),{c["price"]}{r}*{c["shares_out"]}{r},"")', fmt)
            elif key == "ev":
                sh.put(r, j, f'=IF(AND(ISNUMBER({c["mcap"]}{r}),ISNUMBER({c["cash"]}{r})),{c["mcap"]}{r}+SUM({c["total_borrowings"]}{r},{c["nci"]}{r},{c["preferred"]}{r})-{c["cash"]}{r},"")', fmt)
            elif key == "debt_tags":
                cur, non, note = _s(row.get("debt_current_tag")), _s(row.get("debt_noncurrent_tag")), _s(row.get("borrowings_note"))
                sh.put(r, j, "; ".join(t for t in (f"current: {cur}" if cur else "", f"noncurrent: {non}" if non else "", note) if t), font=BLACK)
            elif key == "vi_notes":
                notes = [x for x in (_s(row.get("tbv_reason")), _s(row.get("ev_reason")), _s(row.get("shares_source")) if "fallback" in _s(row.get("shares_source")) else "") if x]
                sh.put(r, j, "; ".join(notes), font=BLACK)
            elif key in ("form", "accn", "price_reason", "cover_accn", "bs_accn"):
                sh.put(r, j, _s(row.get(key)), font=BLACK)
            else:
                v = row.get(key)
                sh.put(r, j, v if (_num(v) or isinstance(v, pd.Timestamp)) else None, fmt)
    # Dividend events (Input C) for the SUMIFS behind TTM dividends per share.
    start_col = len(VI_COLUMNS) + 2
    ex_col, amt_col = get_column_letter(start_col), get_column_letter(start_col + 1)
    ws.cell(row=1, column=start_col, value="Dividend ex-date (Input C)").font = BOLD
    ws.cell(row=1, column=start_col + 1, value="Dividend per share (as-of split basis)").font = BOLD
    ws.cell(row=1, column=start_col).fill = HEAD_FILL
    ws.cell(row=1, column=start_col + 1).fill = HEAD_FILL
    ws.column_dimensions[ex_col].width = 14
    ws.column_dimensions[amt_col].width = 16
    prices = review.prices
    if prices is not None and prices.dividends is not None and not ttm.empty:
        lo = ttm["quarter_end"].min() - pd.DateOffset(years=1, days=7)
        d = prices.dividends[prices.dividends["ex_date"] >= lo]
        for k, (_, ev) in enumerate(d.iterrows(), start=2):
            sh.put(k, start_col, ev["ex_date"], "date")
            sh.put(k, start_col + 1, float(ev["amount"]), "0.0000")
    ws.freeze_panes = "E2"
    return cols, (ex_col, amt_col)


# ---------------------------------------------------------------- TTM_Quarterly

def _write_ttm_quarterly(sh: Sheet, review, build_rows, vi, div_cols) -> dict[str, str]:
    ws = sh.ws
    ttm = review.ttm.reset_index(drop=True)
    lead = [("quarter_end", "Quarter end (TTM period end)", "date"), ("fiscal_quarter", "Fiscal quarter", None), ("valuation_date", "Valuation date (first filing date)", "date"),
            ("form", "Form", None), ("accn", "Accession", None), ("regime", "Taxonomy / currency", None), ("in_stats", "In statistics", None)]
    labels = [label for _, label, _ in lead]
    cols: dict[str, str] = {}
    col = len(lead) + 1
    for s in SERIES:
        cols[s.key] = get_column_letter(col)
        cols[s.key + "__elig"] = get_column_letter(col + 1)
        unit = {"money": "", "shares": " (shares)", "per_share": " (per share)"}[s.unit]
        labels += [f"{s.label} [{s.basis}]{unit}", f"{s.label}: eligible for ADJUSTED (> 0)"]
        col += 2
    extra = [("eps_reported_ttm", "Reported EPS cross-check (FY+YTD-PYTD)", "per_share"), ("eps_check_gap", "EPS cross-check gap", "pct"),
             ("eps_cagr5", "5-year TTM EPS CAGR", "pct"), ("shares_method", "Diluted shares method", None), ("notes", "Missing-value reasons", None)]
    for key, label, _ in extra:
        cols[key] = get_column_letter(col)
        labels.append(label)
        col += 1
    sh.header(1, labels, [12, 8, 12, 8, 22, 14, 9] + [16, 9] * len(SERIES) + [14, 10, 10, 14, 60])
    c = cols
    ex, amt = div_cols
    for i, row in ttm.iterrows():
        r = i + 2
        sh.put(r, 1, row["quarter_end"], "date")
        sh.put(r, 2, row["fiscal_quarter"], font=BLACK)
        sh.put(r, 3, row["valuation_date"], "date")
        sh.put(r, 4, row["form"], font=BLACK)
        sh.put(r, 5, row["accn"], font=BLACK)
        sh.put(r, 6, row["regime"], font=BLACK)
        sh.put(r, 7, "Y" if i >= review.stats_start_index else "N (before regime break)", font=BLACK)
        shares = f"{c['avg_diluted_shares']}{r}"
        for s in SERIES:
            cell = f"{c[s.key]}{r}"
            key = s.key
            f = None
            metric = {"ttm_revenue": "revenue", "ttm_gross_profit": "gross_profit", "ttm_operating_income": "operating_income", "ttm_net_income": "net_income",
                      "ttm_ocf": "ocf", "ttm_capex": "capex", "ttm_da": "da", "ttm_dividends_paid": "dividends_paid", "avg_diluted_shares": "diluted_shares"}.get(key)
            if metric is not None:
                br = build_rows.get((i, metric))
                if br is not None and _num(row.get(key)):
                    f = f"={_q('TTM_Build')}!I{br}"
            elif key == "ttm_fcf":
                f = f'=IF(AND(ISNUMBER({c["ttm_ocf"]}{r}),ISNUMBER({c["ttm_capex"]}{r})),{c["ttm_ocf"]}{r}-{c["ttm_capex"]}{r},"")'
            elif key == "ttm_ebitda":
                f = f'=IF(AND(ISNUMBER({c["ttm_operating_income"]}{r}),ISNUMBER({c["ttm_da"]}{r})),{c["ttm_operating_income"]}{r}+{c["ttm_da"]}{r},"")'
            elif key == "equity":
                f = f'=IF(ISNUMBER({_q("Valuation_Inputs")}!{vi["equity"]}{r}),{_q("Valuation_Inputs")}!{vi["equity"]}{r},"")'
            elif key == "bvps":
                so = f"{_q('Valuation_Inputs')}!{vi['shares_out']}{r}"
                f = f'=IF(AND(ISNUMBER({c["equity"]}{r}),ISNUMBER({so})),IF({so}>0,{c["equity"]}{r}/{so},""),"")'
            elif key == "dps_ttm":
                if _num(row.get("dps_ttm")):
                    rng_d = f"{_q('Valuation_Inputs')}!${ex}:${ex}"
                    rng_a = f"{_q('Valuation_Inputs')}!${amt}:${amt}"
                    f = f'=SUMIFS({rng_a},{rng_d},">"&EDATE(A{r},-12),{rng_d},"<="&A{r})'
            else:
                src = {"revenue_ps": "ttm_revenue", "gp_ps": "ttm_gross_profit", "eps_ttm": "ttm_net_income", "ocf_ps": "ttm_ocf", "fcf_ps": "ttm_fcf",
                       "da_ps": "ttm_da", "ebitda_ps": "ttm_ebitda", "oi_ps": "ttm_operating_income"}[key]
                f = f'=IF(AND(ISNUMBER({c[src]}{r}),ISNUMBER({shares})),IF({shares}>0,{c[src]}{r}/{shares},""),"")'
            if f is not None:
                sh.put(r, ws[cell].column, f, s.unit)
            sh.put(r, ws[cell].column + 1, f'=IF(ISNUMBER({cell}),IF({cell}>0,"Y","N"),"")', font=BLACK)
        if _num(row.get("eps_reported_ttm")):
            sh.put(r, ws[f"{c['eps_reported_ttm']}1"].column, float(row["eps_reported_ttm"]), "per_share")
            sh.put(r, ws[f"{c['eps_check_gap']}1"].column, f'=IF(AND(ISNUMBER({c["eps_ttm"]}{r}),ISNUMBER({c["eps_reported_ttm"]}{r})),IF({c["eps_reported_ttm"]}{r}=0,"",ABS({c["eps_ttm"]}{r}-{c["eps_reported_ttm"]}{r})/ABS({c["eps_reported_ttm"]}{r})),"")', "pct")
        base = row.get("eps_cagr5_base_row")
        if _num(base):
            b = int(base) + 2
            e = c["eps_ttm"]
            sh.put(r, ws[f"{c['eps_cagr5']}1"].column, f'=IF(AND(ISNUMBER({e}{r}),ISNUMBER({e}{b})),IF(AND({e}{r}>0,{e}{b}>0),({e}{r}/{e}{b})^(1/5)-1,""),"")', "pct")
        sh.put(r, ws[f"{c['shares_method']}1"].column, _s(row.get("shares_method")), font=BLACK)
        sh.put(r, ws[f"{c['notes']}1"].column, _s(row.get("notes")), font=BLACK)
    ws.freeze_panes = "H2"
    return cols


# ---------------------------------------------------------------- Ratios_Quarterly

def ratio_formula(ratio, refs: dict[str, str]) -> str:
    """Excel formula for a ratio. `refs` maps input names to cell references."""
    if ratio.key == "graham":
        e, b = refs["eps"], refs["bvps"]
        return f'=IF(AND(ISNUMBER({e}),ISNUMBER({b})),IF(AND({e}>0,{b}>0),SQRT(22.5*{e}*{b}),"NM"),"")'
    if ratio.key == "peg":
        pe, g = refs["pe"], refs["cagr"]
        return f'=IF(AND(ISNUMBER({g}),{g}>0),IF(ISNUMBER({pe}),{pe}/({g}*100),IF({pe}="NM","NM","")),"")'
    n, d = refs[ratio.numerator], refs[ratio.denominator]
    return f'=IF(AND(ISNUMBER({n}),ISNUMBER({d})),IF({d}<=0,"NM",{n}/{d}),"")'


def _write_ratios(sh: Sheet, review, tq, vi):
    ws = sh.ws
    labels = ["Valuation date", "Quarter end", "Price", "Market cap", "Enterprise value"] + [f"{r.label}" for r in RATIOS] + ["Missing-input reasons"]
    sh.header(1, labels, [12, 12, 12, 18, 18] + [12] * len(RATIOS) + [70])
    ws.cell(row=1, column=1).comment = Comment("Formulas: " + " | ".join(f"{r.label} = {r.formula}" for r in RATIOS), "review")
    ttm = review.ttm.reset_index(drop=True)
    ratio_col = {r.key: get_column_letter(6 + k) for k, r in enumerate(RATIOS)}
    V, T = _q("Valuation_Inputs"), _q("TTM_Quarterly")
    pass  # was: relative import (names are global in the single cell)
    pass  # was: relative import (names are global in the single cell)

    for i, row in ttm.iterrows():
        r = i + 2
        sh.put(r, 1, f"={T}!C{r}", "date")
        sh.put(r, 2, f"={T}!A{r}", "date")
        sh.put(r, 3, f'=IF(ISNUMBER({V}!{vi["price"]}{r}),{V}!{vi["price"]}{r},"")', "per_share")
        sh.put(r, 4, f'=IF(ISNUMBER({V}!{vi["mcap"]}{r}),{V}!{vi["mcap"]}{r},"")', "money")
        sh.put(r, 5, f'=IF(ISNUMBER({V}!{vi["ev"]}{r}),{V}!{vi["ev"]}{r},"")', "money")
        refs = {
            "price": f"C{r}", "mcap": f"D{r}", "ev": f"E{r}",
            "eps": f"{T}!{tq['eps_ttm']}{r}", "revenue": f"{T}!{tq['ttm_revenue']}{r}", "gross_profit": f"{T}!{tq['ttm_gross_profit']}{r}",
            "equity": f"{T}!{tq['equity']}{r}", "tbv": f"{V}!{vi['tbv']}{r}", "ocf": f"{T}!{tq['ttm_ocf']}{r}", "fcf": f"{T}!{tq['ttm_fcf']}{r}",
            "dps": f"{T}!{tq['dps_ttm']}{r}", "operating_income": f"{T}!{tq['ttm_operating_income']}{r}", "ebitda": f"{T}!{tq['ttm_ebitda']}{r}",
            "bvps": f"{T}!{tq['bvps']}{r}", "pe": f"{ratio_col['pe']}{r}", "cagr": f"{T}!{tq['eps_cagr5']}{r}",
        }
        for k, ratio in enumerate(RATIOS):
            fmt = {"multiple": "multiple", "yield": "yield", "ratio": "ratio", "value": "per_share"}[ratio.kind]
            sh.put(r, 6 + k, ratio_formula(ratio, refs), fmt)
        values = compute_all(ratio_inputs(row))
        reasons = [f"{rd.label}: {values[rd.key][1]}" for rd in RATIOS if values[rd.key][0] is None and values[rd.key][1]]
        sh.put(r, 6 + len(RATIOS), "; ".join(reasons), font=BLACK)
    last = len(ttm) + 1
    for ratio in RATIOS:
        if ratio.kind == "multiple" and last >= 2:
            rng = f"{ratio_col[ratio.key]}2:{ratio_col[ratio.key]}{last}"
            ws.conditional_formatting.add(rng, CellIsRule(operator="greaterThan", formula=["100"], fill=EXTREME_FILL))
            ws.conditional_formatting.add(rng, CellIsRule(operator="lessThan", formula=["-100"], fill=EXTREME_FILL))
    note_row = last + 2
    ws.cell(row=note_row, column=1, value="Shaded multiples exceed 100x in absolute value: check inputs. NM = zero or negative denominator. Blank = input missing (reason in the last column).").font = NOTE
    ws.freeze_panes = "F2"


# ---------------------------------------------------------------- Stats_Helpers

def _write_helpers(sh: Sheet, review, tq) -> dict[str, tuple[str, str, str]]:
    ws = sh.ws
    ttm = review.ttm.reset_index(drop=True)
    s0 = review.stats_start_index + 2
    last = len(ttm) + 1
    T = _q("TTM_Quarterly")
    ws.cell(row=1, column=1, value="Valuation date").font = BOLD
    cols: dict[str, tuple[str, str, str]] = {}
    col = 2
    for s in SERIES:
        names = (get_column_letter(col), get_column_letter(col + 1), get_column_letter(col + 2))
        cols[s.key] = names
        for k, label in enumerate(["running count", "RAW record high", "ADJ record high"]):
            cell = ws.cell(row=1, column=col + k, value=f"{s.label} [{s.basis}]: {label}")
            cell.font = BOLD
            cell.fill = HEAD_FILL
            cell.alignment = Alignment(wrap_text=True)
        col += 3
    for r in range(s0, last + 1):
        sh.put(r, 1, f"={T}!C{r}", "date")
        for s in SERIES:
            x = tq[s.key]
            cnt, raw, adj = cols[s.key]
            sh.put(r, ws[f"{cnt}1"].column, f"=COUNT({T}!{x}${s0}:{x}{r})", "count")
            if r == s0:
                sh.put(r, ws[f"{raw}1"].column, "=0", "count")
                sh.put(r, ws[f"{adj}1"].column, "=0", "count")
            else:
                prev = f"{T}!{x}${s0}:{x}{r - 1}"
                cur = f"{T}!{x}{r}"
                sh.put(r, ws[f"{raw}1"].column, f"=IF(AND(ISNUMBER({cur}),COUNT({prev})>0),IF({cur}>MAX({prev}),1,0),0)", "count")
                sh.put(r, ws[f"{adj}1"].column, f'=IF(ISNUMBER({cur}),IF(AND({cur}>0,COUNTIF({prev},">0")>0),IF({cur}>MAX({prev}),1,0),0),0)', "count")
    ws.freeze_panes = "B2"
    return cols


# ---------------------------------------------------------------- TTM_Stats

STAT_COLUMNS = [
    ("series", "Series"), ("basis", "Basis"), ("focus", "Focus"), ("start", "Statistics start (valuation date)"),
    ("raw_n", "RAW N"), ("raw_comparisons", "RAW comparisons"), ("raw_increases", "RAW increases"), ("raw_decreases", "RAW decreases"), ("raw_unchanged", "RAW unchanged"),
    ("raw_increase_rate", "RAW increase rate"), ("raw_record_highs", "RAW record highs"), ("raw_latest_record", "RAW latest is record high"), ("raw_latest_rank", "RAW latest rank (1 = highest)"),
    ("adj_n", "ADJ N"), ("adj_comparisons", "ADJ comparisons"), ("adj_pairs_skipped", "ADJ pairs skipped"), ("adj_increases", "ADJ increases"), ("adj_decreases", "ADJ decreases"),
    ("adj_unchanged", "ADJ unchanged"), ("adj_increase_rate", "ADJ increase rate"), ("adj_record_highs", "ADJ record highs"), ("adj_latest_record", "ADJ latest is record high"),
    ("adj_excluded", "ADJ excluded observations (≤ 0)"), ("adj_sign_changes", "Sign changes"),
    ("latest", "Latest value"), ("latest_date", "Latest valuation date"), ("prior", "Prior value"), ("prior_date", "Prior valuation date"), ("pct_change", "% change from prior"),
    ("min", "Full-history minimum"), ("min_date", "Minimum date"), ("max", "Full-history maximum"), ("max_date", "Maximum date"),
    ("daily_obs", "Daily observations in display window (linked ratio)"), ("daily_ratio", "Linked daily ratio"),
]


def _write_stats(sh: Sheet, review, tq, helpers):
    ws = sh.ws
    ttm = review.ttm.reset_index(drop=True)
    s0 = review.stats_start_index + 2
    last = len(ttm) + 1
    T, H = _q("TTM_Quarterly"), _q("Stats_Helpers")
    ws["A1"] = "TTM statistics over quarter-end valuation dates (full history from the statistics start). RAW and ADJUSTED side by side."
    ws["A1"].font = TITLE
    ws["A2"] = ("RAW counts every numeric value (a move from -2 to -1 is an increase). ADJUSTED keeps values > 0 only; pairs with an excluded side are 'pairs skipped'. "
                "Record highs exclude the first (reference) observation. Daily observation counts are computed in Python from Input C (daily values are not stored in this workbook).")
    ws["A2"].font = NOTE
    sh.header(4, [label for _, label in STAT_COLUMNS], [34, 9, 7, 13] + [10] * 20 + [16, 12, 16, 12, 10, 16, 12, 16, 12, 12, 12])
    col = {key: get_column_letter(i) for i, (key, _) in enumerate(STAT_COLUMNS, start=1)}
    stats = review.stats.set_index("series") if not review.stats.empty else pd.DataFrame()
    for k, s in enumerate(SERIES):
        r = 5 + k
        x = tq[s.key]
        rng, rp, rn = f"{T}!${x}${s0}:${x}${last}", f"{T}!${x}${s0}:${x}${last - 1}", f"{T}!${x}${s0 + 1}:${x}${last}"
        dates = f"{T}!$C${s0}:$C${last}"
        cnt, raw_h, adj_h = (f"{H}!${c}${s0}:${c}${last}" for c in helpers[s.key])
        C = lambda key: f"{col[key]}{r}"  # noqa: E731
        single = last <= s0
        pair = "=0" if single else None
        both = f"ISNUMBER({rp})*ISNUMBER({rn})"
        adjp = f"{both}*({rp}>0)*({rn}>0)"
        fmt = s.unit
        sh.put(r, 1, f"{s.label}", font=BOLD if s.focus else BLACK)
        sh.put(r, 2, s.basis, font=BLACK)
        sh.put(r, 3, "yes" if s.focus else "", font=BLACK)
        sh.put(r, 4, f"={T}!C{s0}", "date")
        formulas = {
            "raw_n": f"=COUNT({rng})",
            "raw_comparisons": pair or f"=SUMPRODUCT({both})",
            "raw_increases": pair or f"=SUMPRODUCT({both}*({rn}>{rp}))",
            "raw_decreases": pair or f"=SUMPRODUCT({both}*({rn}<{rp}))",
            "raw_unchanged": pair or f"=SUMPRODUCT({both}*({rn}={rp}))",
            "raw_increase_rate": f'=IF({C("raw_comparisons")}>0,{C("raw_increases")}/{C("raw_comparisons")},"")',
            "raw_record_highs": f"=SUM({raw_h})",
            "raw_latest_record": f'=IF({C("raw_n")}<1,"n/a (no values)",IF({C("raw_n")}=1,"n/a (single value)",IF(INDEX({raw_h},MATCH({C("raw_n")},{cnt},0))=1,"yes","no")))',
            "raw_latest_rank": f'=IF({C("raw_n")}>0,1+COUNTIF({rng},">"&{C("latest")}),"")',
            "adj_n": f'=COUNTIF({rng},">0")',
            "adj_comparisons": pair or f"=SUMPRODUCT({adjp})",
            "adj_pairs_skipped": f'={C("raw_comparisons")}-{C("adj_comparisons")}',
            "adj_increases": pair or f"=SUMPRODUCT({adjp}*({rn}>{rp}))",
            "adj_decreases": pair or f"=SUMPRODUCT({adjp}*({rn}<{rp}))",
            "adj_unchanged": pair or f"=SUMPRODUCT({adjp}*({rn}={rp}))",
            "adj_increase_rate": f'=IF({C("adj_comparisons")}>0,{C("adj_increases")}/{C("adj_comparisons")},"")',
            "adj_record_highs": f"=SUM({adj_h})",
            "adj_latest_record": f'=IF({C("raw_n")}<1,"n/a (no values)",IF({C("latest")}<=0,"excluded (≤ 0)",IF({C("adj_n")}<=1,"n/a (single value)",IF(INDEX({adj_h},MATCH({C("raw_n")},{cnt},0))=1,"yes","no"))))',
            "adj_excluded": f'=COUNTIF({rng},"<=0")',
            "adj_sign_changes": pair or f"=SUMPRODUCT({both}*((({rp}>0)*({rn}<0))+(({rp}<0)*({rn}>0))))",
            "latest": f'=IF({C("raw_n")}>0,INDEX({rng},MATCH({C("raw_n")},{cnt},0)),"")',
            "latest_date": f'=IF({C("raw_n")}>0,INDEX({dates},MATCH({C("raw_n")},{cnt},0)),"")',
            "prior": f'=IF({C("raw_n")}>1,INDEX({rng},MATCH({C("raw_n")}-1,{cnt},0)),"")',
            "prior_date": f'=IF({C("raw_n")}>1,INDEX({dates},MATCH({C("raw_n")}-1,{cnt},0)),"")',
            "pct_change": f'=IF(AND(ISNUMBER({C("latest")}),ISNUMBER({C("prior")})),IF({C("prior")}=0,"NM",({C("latest")}-{C("prior")})/ABS({C("prior")})),"")',
            "min": f'=IF({C("raw_n")}>0,MIN({rng}),"")',
            "min_date": f'=IF({C("raw_n")}>0,INDEX({dates},MATCH({C("min")},{rng},0)),"")',
            "max": f'=IF({C("raw_n")}>0,MAX({rng}),"")',
            "max_date": f'=IF({C("raw_n")}>0,INDEX({dates},MATCH({C("max")},{rng},0)),"")',
        }
        fmts = {"raw_increase_rate": "pct", "adj_increase_rate": "pct", "pct_change": "pct", "latest": fmt, "prior": fmt, "min": fmt, "max": fmt,
                "latest_date": "date", "prior_date": "date", "min_date": "date", "max_date": "date"}
        for key, formula in formulas.items():
            sh.put(r, ws[f"{col[key]}1"].column, formula, fmts.get(key, "count"))
        daily = stats.at[s.key, "daily_obs"] if s.key in stats.index else None
        sh.put(r, ws[f"{col['daily_obs']}1"].column, int(daily) if _num(daily) else None, "count")
        sh.put(r, ws[f"{col['daily_ratio']}1"].column, s.daily_ratio or "", font=BLACK)
    ws.freeze_panes = "B5"


# ---------------------------------------------------------------- Snapshot

def _write_snapshot(sh: Sheet, review, tq, last_row: int):
    ws = sh.ws
    h, snap = review.header, review.snapshot_inputs
    ws["A1"] = f"{h['company']} ({h['ticker'] or 'no ticker'}) — CIK {h['cik']} — valuation snapshot as of {snap['as_of'].date()}"
    ws["A1"].font = TITLE
    ws["A2"] = f"Latest filing: {h['latest_filing']}. Taxonomy {h['taxonomy']}; currency {h['currency']}. Prices: {h['price_range']} ({h['price_basis']}). Splits: {h['splits']}."
    ws["A2"].font = NOTE
    ws.column_dimensions["A"].width = 40
    for c, w in zip("BCDEFGH", [20, 14, 60, 40, 36, 14, 50]):
        ws.column_dimensions[c].width = w
    T = _q("TTM_Quarterly")
    # ---- inputs block
    sh.header(4, ["Input", "Value", "Date", "Source / note"])
    inputs = [
        ("price", "Price (as-of split basis)", snap.get("price"), "per_share", snap.get("price_date"), snap.get("price_source") or snap.get("price_reason", "")),
        ("shares", "Shares outstanding (as-of split basis)", snap.get("shares_out"), "shares", snap.get("cover_date"), f"{snap.get('shares_source', '')}; {snap.get('cover_form', '')} {snap.get('cover_accn', '')}; split factor {snap.get('share_factor', '')}"),
        ("cash", "Cash & equivalents", snap.get("cash"), "money", snap.get("bs_instant"), f"{snap.get('bs_form', '')} {snap.get('bs_accn', '')}"),
        ("debt", "Total borrowings (current + noncurrent, as tagged)", snap.get("total_borrowings"), "money", snap.get("bs_instant"), snap.get("borrowings_note") or f"current: {snap.get('debt_current_tag') or '-'}; noncurrent: {snap.get('debt_noncurrent_tag') or '-'}"),
        ("nci", "Noncontrolling interest", snap.get("nci"), "money", snap.get("bs_instant"), "" if snap.get("nci") is not None else "not tagged (excluded from EV)"),
        ("pref", "Preferred stock", snap.get("preferred"), "money", snap.get("bs_instant"), "" if snap.get("preferred") is not None else "not tagged (excluded from EV)"),
        ("equity", "Parent equity", snap.get("equity"), "money", snap.get("bs_instant"), ""),
        ("goodwill", "Goodwill", snap.get("goodwill"), "money", snap.get("bs_instant"), "" if snap.get("goodwill") is not None else "not tagged"),
        ("intang", "Intangibles excl. goodwill", snap.get("intangibles"), "money", snap.get("bs_instant"), "" if snap.get("intangibles") is not None else "not tagged"),
    ]
    ref: dict[str, str] = {}
    r = 5
    for key, label, value, fmt, when, note in inputs:
        sh.put(r, 1, label, font=BLACK)
        sh.put(r, 2, value if _num(value) else None, fmt)
        sh.put(r, 3, when, "date")
        sh.put(r, 4, note or "", font=BLACK)
        ref[key] = f"B{r}"
        r += 1
    derived = [
        ("mcap", "Market cap", f'=IF(AND(ISNUMBER({ref["price"]}),ISNUMBER({ref["shares"]})),{ref["price"]}*{ref["shares"]},"")', "money", "Price x shares outstanding (cover page)"),
        ("ev", "Enterprise value", f'=IF(AND(ISNUMBER(B{r}),ISNUMBER({ref["cash"]})),B{r}+SUM({ref["debt"]},{ref["nci"]},{ref["pref"]})-{ref["cash"]},"")', "money", "Market cap + borrowings + NCI + preferred - cash (untagged components excluded; lease liabilities and short-term investments excluded)"),
        ("tbv", "Tangible book value", f'=IF(ISNUMBER({ref["equity"]}),IF(COUNT({ref["goodwill"]},{ref["intang"]})=0,"",{ref["equity"]}-SUM({ref["goodwill"]},{ref["intang"]})),"")', "money", "Parent equity - goodwill - intangibles (if tagged)"),
        ("bvps", "Book value per share", f'=IF(AND(ISNUMBER({ref["equity"]}),ISNUMBER({ref["shares"]})),IF({ref["shares"]}>0,{ref["equity"]}/{ref["shares"]},""),"")', "per_share", "Parent equity / cover-page shares outstanding"),
    ]
    for key, label, formula, fmt, note in derived:
        sh.put(r, 1, label, font=BLACK)
        sh.put(r, 2, formula, fmt)
        sh.put(r, 4, note, font=BLACK)
        ref[key] = f"B{r}"
        r += 1
    # ---- TTM block (links to the latest TTM_Quarterly row)
    r += 1
    sh.header(r, ["TTM / per-share TTM value", "Value", "Valuation date", "Inputs"])
    r += 1
    ttm = review.ttm
    latest = ttm.iloc[-1] if not ttm.empty else None
    for s in SERIES:
        sh.put(r, 1, f"{s.label} [{s.basis}]", font=BLACK)
        if latest is not None:
            sh.put(r, 2, f'=IF(ISNUMBER({T}!{tq[s.key]}{last_row}),{T}!{tq[s.key]}{last_row},"")', s.unit)
            sh.put(r, 3, f"={T}!C{last_row}", "date")
            sh.put(r, 4, f"TTM ending {latest['quarter_end'].date()} ({latest['form']} {latest['accn']})", font=BLACK)
        ref["ttm_" + s.key] = f"B{r}"
        r += 1
    sh.put(r, 1, "5-year TTM EPS CAGR", font=BLACK)
    sh.put(r, 2, f'=IF(ISNUMBER({T}!{tq["eps_cagr5"]}{last_row}),{T}!{tq["eps_cagr5"]}{last_row},"")', "pct")
    sh.put(r, 4, (latest.get("eps_cagr5_reason") or "") if latest is not None else "", font=BLACK)
    ref["cagr"] = f"B{r}"
    r += 2
    # ---- ratios
    sh.header(r, ["Ratio", "Value", "Valuation date", "Formula", "Inputs", "Share count used", "Percentile (full daily history)", "Reason if blank / NM"])
    ws.cell(row=r, column=7).comment = Comment("Computed in Python from the daily series (Input C price / held fundamentals). Daily values are not stored in this workbook.", "review")
    r += 1
    refs = {
        "price": ref["price"], "mcap": ref["mcap"], "ev": ref["ev"], "eps": ref["ttm_eps_ttm"], "revenue": ref["ttm_ttm_revenue"], "gross_profit": ref["ttm_ttm_gross_profit"],
        "equity": ref["equity"], "tbv": ref["tbv"], "ocf": ref["ttm_ttm_ocf"], "fcf": ref["ttm_ttm_fcf"], "dps": ref["ttm_dps_ttm"], "operating_income": ref["ttm_ttm_operating_income"],
        "ebitda": ref["ttm_ttm_ebitda"], "bvps": ref["bvps"], "cagr": ref["cagr"],
    }
    snapshot = review.snapshot.set_index("key") if not review.snapshot.empty else pd.DataFrame()
    pe_row = r + [x.key for x in RATIOS].index("pe")
    refs["pe"] = f"B{pe_row}"
    for ratio in RATIOS:
        fmt = {"multiple": "multiple", "yield": "yield", "ratio": "ratio", "value": "per_share"}[ratio.kind]
        sh.put(r, 1, ratio.label, font=BLACK)
        sh.put(r, 2, ratio_formula(ratio, refs), fmt)
        if latest is not None:
            sh.put(r, 3, latest["valuation_date"], "date")
        sh.put(r, 4, ratio.formula, font=BLACK)
        info = snapshot.loc[ratio.key] if ratio.key in snapshot.index else None
        sh.put(r, 5, info["inputs"] if info is not None else "", font=BLACK)
        sh.put(r, 6, ratio.shares, font=BLACK)
        pct = info["percentile"] if info is not None else None
        sh.put(r, 7, pct / 100 if _num(pct) else None, "pct")
        sh.put(r, 8, (info["reason"] or "") if info is not None else "", font=BLACK)
        r += 1
    rng = f"B{r - len(RATIOS)}:B{r - 1}"
    ws.conditional_formatting.add(rng, CellIsRule(operator="greaterThan", formula=["100"], fill=EXTREME_FILL))
    r += 1
    ws.cell(row=r, column=1, value="Blue = value from Inputs A/C/D; black = formula; green = link. Shaded multiples exceed 100x: check inputs. This sheet holds the only current-state values in the workbook.").font = NOTE


# ---------------------------------------------------------------- Data_Quality

def _write_dq(sh: Sheet, review):
    ws = sh.ws
    for c, w in zip("ABCDEF", [24, 10, 34, 12, 110, 28]):
        ws.column_dimensions[c].width = w
    r = 1
    ws.cell(row=r, column=1, value="Coverage (first and last fiscal year with usable data)").font = TITLE
    r += 1
    sh.header(r, ["Item", "First", "Last", "Years"])
    for c, w in zip("ABCDEF", [24, 10, 34, 12, 110, 28]):
        ws.column_dimensions[c].width = w
    r += 1
    for _, row in review.coverage_summary.iterrows():
        for c, key in enumerate(["item", "first", "last", "years"], start=1):
            sh.put(r, c, row[key], font=BLACK)
        r += 1
    r += 1
    ws.cell(row=r, column=1, value="Red flags (each cited to its filing)").font = TITLE
    r += 1
    sh.header(r, ["Flag", "Severity", "Source", "", "Evidence"])
    r += 1
    for _, row in review.red_flags.iterrows():
        sh.put(r, 1, row["flag"], font=BLACK)
        sh.put(r, 2, row["severity"], font=BLACK)
        sh.put(r, 3, row["source"], font=BLACK)
        sh.put(r, 5, row["evidence"], font=BLACK)
        r += 1
    r += 1
    ws.cell(row=r, column=1, value="Data-quality log").font = TITLE
    r += 1
    sh.header(r, ["Category", "Severity", "Item", "Period", "Detail", "Filing"])
    r += 1
    for _, row in review.dq.frame().iterrows():
        for c, key in enumerate(["category", "severity", "item", "period", "detail", "filing"], start=1):
            sh.put(r, c, row[key], font=BLACK)
        r += 1

# ============================== valuation/report.py ==============================
"""Markdown report in the order the review specifies (sections 1-10)."""


import math

import pandas as pd


LAUNCH = "python -B -m streamlit run valuation_app.py --server.headless true --browser.gatherUsageStats false"


def _num(v) -> bool:
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(float(v))


def fmt_value(v, unit: str) -> str:
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ""
    if isinstance(v, str):
        return v
    if unit == "money":
        return f"{v / 1e6:,.1f}"
    if unit == "shares":
        return f"{v / 1e6:,.1f}"
    if unit in ("per_share", "value"):
        return f"{v:,.2f}"
    if unit == "multiple":
        return f"{v:,.1f}x"
    if unit in ("yield", "ratio", "pct"):
        return f"{v * 100:,.2f}%"
    if unit == "count":
        return f"{int(v)}"
    return f"{v:,.4g}"


def _d(x) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    try:
        return str(pd.Timestamp(x).date())
    except (ValueError, TypeError):
        return str(x)


def _table(headers: list[str], rows: list[list[str]]) -> str:
    esc = lambda s: str(s).replace("|", "\\|").replace("\n", " ")  # noqa: E731
    out = ["| " + " | ".join(esc(h) for h in headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    out += ["| " + " | ".join(esc(c) for c in row) + " |" for row in rows]
    return "\n".join(out)


ANNUAL_ROWS = [
    ("revenue", "Revenue", "money"), ("gross_profit", "Gross profit", "money"), ("operating_income", "Operating income", "money"),
    ("net_income", "Net income (parent)", "money"), ("eps_diluted", "Diluted EPS (reported, split-normalized)", "per_share"),
    ("diluted_shares", "Diluted shares (millions)", "shares"), ("ocf", "Operating cash flow", "money"), ("capex", "Capex", "money"),
    ("fcf", "FCF (OCF - capex)", "money"), ("da", "D&A", "money"), ("dps_inputc", "Dividends per share (Input C ex-dates)", "per_share"),
    ("cash", "Cash", "money"), ("total_assets", "Total assets", "money"), ("total_liabilities", "Total liabilities", "money"),
    ("equity", "Parent equity", "money"), ("long_term_debt", "Long-term debt", "money"),
]
TTM_FOR_ANNUAL = {"revenue": "ttm_revenue", "gross_profit": "ttm_gross_profit", "operating_income": "ttm_operating_income", "net_income": "ttm_net_income",
                  "eps_diluted": "eps_ttm", "diluted_shares": "avg_diluted_shares", "ocf": "ttm_ocf", "capex": "ttm_capex", "fcf": "ttm_fcf", "da": "ttm_da",
                  "dps_inputc": "dps_ttm", "cash": "cash", "total_assets": "total_assets", "total_liabilities": "total_liabilities", "equity": "equity", "long_term_debt": "long_term_debt"}


def build_report(review, workbook_path: str | None = None, full_dq: bool = True) -> str:
    h = review.header
    snap = review.snapshot_inputs
    parts: list[str] = []
    title = f"{h['company']} ({h['ticker'] or 'no ticker'})"
    parts.append(f"# Valuation and fundamentals review: {title}\n")

    # 1. Header
    parts.append("## 1. Header\n")
    parts.append(_table(["Field", "Value"], [
        ["Company", h["company"]], ["Ticker", h["ticker"] or ""], ["CIK", h["cik"]],
        ["Taxonomy and currency", f"{h['taxonomy']}; {h['currency']}" + (f" (regimes: {', '.join(h['regimes'])})" if len(h["regimes"]) > 1 else "")],
        ["Latest filing", h["latest_filing"]], ["As-of date", _d(h["as_of"])], ["Display window", h["display_window"]],
        ["Full-history start (statistics)", _d(h["full_history_start"])], ["Price data range", h["price_range"]], ["Price adjustment basis", h["price_basis"]],
        ["Splits applied", h["splits"]],
    ]))
    if review.thin:
        parts.append(f"\n**Coverage too thin for a trend review.** {review.thin_reason} The review stops after the coverage check.\n")

    # 2. Coverage
    parts.append("\n## 2. Coverage matrix\n")
    parts.append("Fiscal years are labeled by the calendar year of the fiscal year end. A blank cell means no usable value.\n")
    cov = review.coverage
    if not cov.empty:
        parts.append(_table(["Item"] + list(cov.columns), [[idx] + list(row) for idx, row in cov.iterrows()]))
        cs = review.coverage_summary
        parts.append("\n" + _table(["Item", "First", "Last", "Years"], [[r["item"], r["first"], r["last"], str(r["years"])] for _, r in cs.iterrows()]))
    if review.thin:
        parts.append(_dq_section(review, full_dq))
        return "\n".join(parts) + "\n"

    # 3. Annual table
    parts.append("\n## 3. Annual table (display window)\n")
    parts.append(f"Amounts in millions of {h['currency']} except per-share values. Columns are fiscal year ends; the last column is the latest TTM.\n")
    annual = review.annual_all
    if not annual.empty and review.display_start is not None:
        window = annual[annual["fy_start"] >= review.display_start]
        latest = review.ttm.iloc[-1] if not review.ttm.empty else None
        headers = ["Item"] + [f"{_d(e)}{' (53w)' if w == '53-week' else ''}" for e, w in zip(window["fy_end"], window["weeks"])]
        add_ttm = latest is not None and (window.empty or latest["quarter_end"] > window["fy_end"].max())
        if add_ttm:
            headers.append(f"TTM {_d(latest['quarter_end'])}")
        rows = []
        for key, label, unit in ANNUAL_ROWS:
            row = [label] + [fmt_value(v, unit) for v in window[key]]
            if add_ttm:
                row.append(fmt_value(latest.get(TTM_FOR_ANNUAL[key]), unit))
            rows.append(row)
        parts.append(_table(headers, rows))

    # 4. Snapshot
    parts.append("\n## 4. Current snapshot\n")
    price = snap.get("price")
    money = lambda key: (fmt_value(snap.get(key), "money") + " m") if _num(snap.get(key)) else "not tagged"  # noqa: E731
    source = snap.get("shares_source") or ""
    where = f"cover page {_d(snap.get('cover_date'))}" if source.startswith("dei:") else (source or "no share count")
    parts.append(
        f"Price {fmt_value(price, 'per_share') or 'n/a'} on {_d(snap.get('price_date'))} ({snap.get('price_source') or snap.get('price_reason', '')}); "
        f"shares outstanding {fmt_value(snap.get('shares_out'), 'shares') or 'n/a'} million ({where}; {snap.get('cover_accn', '')}); "
        f"market cap {money('mcap') if _num(snap.get('mcap')) else 'n/a'}; EV {money('ev') if _num(snap.get('ev')) else 'n/a (' + (snap.get('ev_reason') or '') + ')'} "
        f"= market cap + borrowings {money('total_borrowings')} + NCI {money('nci')} + preferred {money('preferred')} - cash {money('cash')} "
        f"(balance sheet {_d(snap.get('bs_instant'))}, {snap.get('bs_accn', '')}; untagged components are excluded). "
        "Lease liabilities and short-term investments are excluded from EV.\n"
    )
    s = review.snapshot
    if not s.empty:
        ttm_rows = s[s["group"] != "Price ratios"]
        def shown(r):
            text = fmt_value(r["value"], r["unit"])
            return text + " m" if text and r["unit"] in ("money", "shares") else text

        parts.append(_table(["TTM / per-share item", "Value", "Valuation date", "Inputs"], [[r["item"], shown(r), _d(r["valuation_date"]), r["inputs"] or r["reason"] or ""] for _, r in ttm_rows.iterrows()]))
        rows = []
        for _, r in s[s["group"] == "Price ratios"].iterrows():
            pct = review.percentiles.get(r["key"])
            pct_text = f"{pct['percentile']:.1f}" if pct and pct["percentile"] >= 1 else (f"{pct['percentile']:.2f}" if pct else "")
            pct_text += f" (n={pct['n']:,}, {_d(pct['from'])} to {_d(pct['to'])})" if pct else ""
            value = fmt_value(r["value"], r["unit"] if r["unit"] != "value" else "per_share")
            if _num(r["value"]) and r["unit"] == "multiple" and abs(r["value"]) > 100:
                value += " (>100x: check inputs)"
            rows.append([r["item"], value or "", r["formula"], r["shares"], pct_text, r["reason"] or ""])
        parts.append("\n" + _table(["Ratio", "Value", "Formula", "Share count used", "Percentile in full daily history", "Blank / NM reason"], rows))

    # 5. Statistics
    parts.append("\n## 5. TTM statistics (focus series)\n")
    parts.append(f"Observations are quarter-end valuation dates from {_d(review.ttm.iloc[review.stats_start_index]['valuation_date']) if not review.ttm.empty else ''} to {_d(h['as_of'])}.\n")
    st = review.stats
    if not st.empty:
        rows = []
        for _, r in st[st["focus"]].iterrows():
            rows.append([
                f"{r['label']} [{r['basis']}]", str(r["raw_n"]), str(r["raw_increases"]), fmt_value(r["raw_increase_rate"], "pct"), str(r["raw_record_highs"]), r["raw_latest_record"],
                str(r["adj_n"]), str(r["adj_increases"]), fmt_value(r["adj_increase_rate"], "pct"), str(r["adj_record_highs"]), r["adj_latest_record"],
                str(r["adj_excluded"]), str(r["adj_pairs_skipped"]), str(r["adj_sign_changes"]),
            ])
        parts.append(_table(["Series", "RAW N", "RAW incr.", "RAW incr. rate", "RAW records", "RAW latest record", "ADJ N", "ADJ incr.", "ADJ incr. rate", "ADJ records", "ADJ latest record", "Excluded (≤0)", "Pairs skipped", "Sign changes"], rows))

    # 6. Growth
    parts.append("\n## 6. Growth summary (fiscal-year CAGRs)\n")
    g = review.growth
    if not g.empty:
        rows = []
        for _, r in g.iterrows():
            cells = [r["metric"]]
            for n in (5, 10, 15):
                v = r.get(f"cagr_{n}")
                note = r.get(f"note_{n}") or ""
                cells.append((fmt_value(v, "pct") if _num(v) else "") + (f" ({note})" if note else ""))
            rows.append(cells)
        parts.append(_table(["Metric", "5-year", "10-year", "15-year"], rows))
        parts.append(f"\nEnd year: fiscal year ending {g.iloc[0]['to']}. Diluted shares and EPS are on the as-of split basis.")
        if len(h["regimes"]) > 1:
            parts.append(" **Comparability break:** the reporting regime changed during the history (see the data-quality log).")
        parts.append("\n")

    parts.append(_dq_section(review, full_dq))

    # 8. Red flags
    parts.append("\n## 8. Red flags\n")
    rf = review.red_flags
    if rf.empty:
        parts.append("None found by the screens run.\n")
    else:
        parts.append(_table(["Flag", "Severity", "Evidence", "Source"], [[r["flag"], r["severity"], r["evidence"], r["source"]] for _, r in rf.iterrows()]))

    # 9. Deliverables
    parts.append("\n## 9. Deliverables\n")
    parts.append(f"- Excel workbook: {workbook_path or 'built in memory (download from the GUI, or pass an output folder off the C: drive to the batch runner)'}\n")
    parts.append(f"- GUI: `{LAUNCH}` then open http://localhost:8501\n")

    # 10. Summary
    parts.append("\n## 10. Summary\n")
    parts.append(summary_text(review) + "\n")
    return "\n".join(parts)


def _dq_section(review, full: bool) -> str:
    dq = review.dq.frame()
    out = ["\n## 7. Data-quality log\n"]
    if dq.empty:
        out.append("No entries.\n")
        return "\n".join(out)
    counts = dq.groupby("category").size()
    out.append("Entries by category: " + ", ".join(f"{k} {v}" for k, v in counts.items()) + ".\n")
    shown = dq if full else dq[dq["severity"].isin(["warning", "serious", "critical"]) & ~dq["category"].isin(["derived_quarter"])].head(60)
    out.append(_table(["Category", "Severity", "Item", "Period", "Detail", "Filing"], [[r["category"], r["severity"], r["item"], r["period"], r["detail"], r["filing"]] for _, r in shown.iterrows()]))
    return "\n".join(out)


def summary_text(review) -> str:
    """At most six factual sentences drawn from the tables above. No forecasts or recommendations."""
    sentences: list[str] = []
    st = review.stats.set_index("series") if not review.stats.empty else pd.DataFrame()

    def stat(key, field):
        return st.at[key, field] if key in st.index else None

    if "ttm_revenue" in st.index and stat("ttm_revenue", "raw_comparisons"):
        rec = stat("ttm_revenue", "raw_latest_record")
        sentences.append(
            f"TTM revenue rose in {stat('ttm_revenue', 'raw_increases')} of {stat('ttm_revenue', 'raw_comparisons')} consecutive valuation-date comparisons "
            f"({fmt_value(stat('ttm_revenue', 'raw_increase_rate'), 'pct')}), and the latest value {'is' if rec == 'yes' else 'is not'} a full-history record high."
        )
    if "eps_ttm" in st.index and stat("eps_ttm", "raw_n"):
        latest, pct = stat("eps_ttm", "latest"), stat("eps_ttm", "pct_change")
        sentences.append(
            f"TTM diluted EPS is {fmt_value(latest, 'per_share')}"
            + (f", {'up' if pct >= 0 else 'down'} {abs(pct) * 100:.1f}% from the prior valuation date" if _num(pct) else "")
            + f"; ADJUSTED EPS rose in {stat('eps_ttm', 'adj_increases')} of {stat('eps_ttm', 'adj_comparisons')} comparisons with {stat('eps_ttm', 'adj_excluded')} non-positive observation(s) excluded."
        )
    snap = review.snapshot.set_index("key") if not review.snapshot.empty else pd.DataFrame()
    for key in ("pe", "ev_ebitda"):
        if key in snap.index and _num(snap.at[key, "value"]):
            pct = review.percentiles.get(key)
            sentences.append(
                f"At the as-of price, {RATIO_BY_KEY[key].label} is {fmt_value(snap.at[key, 'value'], 'multiple')}"
                + (f", the {pct['percentile']:.0f}th percentile of its daily history since {_d(pct['from'])}." if pct else ".")
            )
        elif key in snap.index and snap.at[key, "value"] == "NM":
            sentences.append(f"{RATIO_BY_KEY[key].label} is NM because its denominator is zero or negative.")
    if "ttm_fcf" in st.index and stat("ttm_fcf", "raw_n"):
        sentences.append(f"TTM free cash flow is {fmt_value(stat('ttm_fcf', 'latest'), 'money')} million, with {stat('ttm_fcf', 'adj_excluded')} non-positive TTM observation(s) in the full history.")
    rf = review.red_flags
    serious = rf[rf["severity"].isin(["serious", "critical"])] if not rf.empty else rf
    dq = review.dq.frame()
    warn = int(dq["severity"].isin(["warning", "serious", "critical"]).sum()) if not dq.empty else 0
    if not serious.empty:
        sentences.append(f"Screens raised {len(serious)} serious or critical flag(s) ({', '.join(sorted(set(serious['flag'])))}) and the data-quality log holds {warn} warning(s).")
    else:
        sentences.append(f"No serious or critical red flags were raised by the data screens; the data-quality log holds {warn} warning(s).")
    return " ".join(sentences[:6])

# ============================== valuation/reference.py ==============================
"""Compare observation counts with a CompanyCharts-style reference workbook (built from Zacks data).

The reference reports, per ticker and metric M in Rev, EPS, FCF, GP, CFO, EQ, the columns
"{M} TTM #Obs", "#Inc", "#Rec", "#Inc Adj", "#Rec Adj" over a trailing window, plus a series
fingerprint ("{M} TTM Slope", "R2", "CV%") that pins down the exact values it counted.

The written spec's rules are the defaults. Each reference convention is a switch in
`ReferenceOptions`; none changes the review, the workbook or the review statistics.
Each metric has its own series builder below so metric-specific conventions stay separate.
"""


import math
from dataclasses import dataclass

import numpy as np
import pandas as pd

METRIC_COLUMNS = {"Rev": "ttm_revenue", "EPS": "eps_ttm", "FCF": "ttm_fcf", "GP": "ttm_gross_profit", "CFO": "ttm_ocf", "EQ": "equity"}
FIELDS = ["#Obs", "#Inc", "#Rec", "#Inc Adj", "#Rec Adj"]
MONEY_METRICS = {"Rev", "FCF", "GP", "CFO", "EQ"}  # reference fingerprints are in $ millions
ALIGN_COLUMNS = {"ttm_revenue": "Revenue TTM Total", "ttm_ocf": "CFO TTM Total", "ttm_net_income": "Net Income TTM Total"}


@dataclass
class ReferenceOptions:
    window: int = 57  # trailing TTM observations the reference counts over
    equity_window: int = 57  # the reference uses 60 for equity (no TTM build needed)
    record_counts_first: bool = False  # spec: the first observation is the reference, not a record
    emulate_first_ytd: bool = False  # reference artifact: the first window value of YTD-built series (CFO, FCF) includes the prior YTD
    missing_capex_as_zero: bool = False  # reference fills FCF = CFO where capex is missing
    align_window: bool = False  # end the window at the quarter the reference's latest TTM totals match (drops our newer quarters)

    @classmethod
    def reference_compatible(cls) -> "ReferenceOptions":
        return cls(window=57, equity_window=60, record_counts_first=True, emulate_first_ytd=True, missing_capex_as_zero=True, align_window=True)


def _num(v) -> bool:
    return isinstance(v, (int, float, np.floating)) and not isinstance(v, bool) and math.isfinite(float(v))


# ---------------------------------------------------------------- counting

def window_counts(values: list, window: int, record_counts_first: bool = False) -> dict:
    vals = list(values[-window:]) if window else list(values)
    vals = [float(v) if _num(v) else None for v in vals]
    present = [v for v in vals if v is not None]
    pairs = [(vals[i], vals[i + 1]) for i in range(len(vals) - 1) if vals[i] is not None and vals[i + 1] is not None]

    def records(seq):
        n, best = 0, None
        for v in seq:
            if best is None:
                n += 1 if record_counts_first else 0
                best = v
            else:
                n += 1 if v > best else 0
                best = max(best, v)
        return n

    return {
        "#Obs": len(present),
        "#Inc": sum(1 for a, b in pairs if b > a),
        "#Rec": records(present),
        "#Inc Adj": sum(1 for a, b in pairs if a > 0 and b > 0 and b > a),
        "#Rec Adj": records([v for v in present if v > 0]),
    }


def fingerprint(values: list, window: int, metric: str) -> dict:
    """Slope (per observation), R2 and CV% (population sd) of the window's values, as the reference computes them."""
    vals = [float(v) for v in values[-window:] if _num(v)]
    if metric in MONEY_METRICS:
        vals = [v / 1e6 for v in vals]
    if len(vals) < 3:
        return {"slope": np.nan, "r2": np.nan, "cv": np.nan, "last": vals[-1] if vals else np.nan}
    y = np.asarray(vals)
    x = np.arange(len(y))
    slope, intercept = np.polyfit(x, y, 1)
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ((y - (intercept + slope * x)) ** 2).sum() / ss_tot if ss_tot else np.nan
    cv = y.std() / y.mean() * 100 if y.mean() else np.nan
    return {"slope": float(slope), "r2": float(r2), "cv": float(cv), "last": float(y[-1])}


# ---------------------------------------------------------------- alignment

def alignment_offset(review, reference_row: pd.Series | None) -> int:
    """How many of our latest TTM rows are newer than the reference's latest quarter (0 or more).

    Uses the reference's latest totals (revenue, CFO, net income, $ millions): if they match one of
    our earlier rows on at least two metrics (within 0.5%) and do not match our latest row, the
    window ends at that earlier row.
    """
    if reference_row is None or review.ttm.empty:
        return 0
    ttm = review.ttm.reset_index(drop=True)

    def matches(i):
        hits = 0
        for col, ref_col in ALIGN_COLUMNS.items():
            theirs = pd.to_numeric(str(reference_row.get(ref_col, "")).replace(",", ""), errors="coerce")
            ours = ttm.at[i, col] if col in ttm else None
            if _num(theirs) and _num(ours) and theirs != 0 and abs(float(ours) / 1e6 - theirs) <= 0.005 * abs(theirs):
                hits += 1
        return hits

    n = len(ttm)
    if matches(n - 1) >= 2:
        return 0
    for back in (1, 2):
        if n - 1 - back >= 0 and matches(n - 1 - back) >= 2:
            return back
    return 0


# ---------------------------------------------------------------- per-metric series builders

def _ttm(review) -> pd.DataFrame:
    return review.ttm.reset_index(drop=True)


def _first_window_ytd(review, key: str, ttm: pd.DataFrame, window: int) -> tuple[int | None, float | None]:
    """Index of the first non-blank value in the window and the YTD the reference adds to it."""
    col = {"ocf": "ttm_ocf", "capex": "ttm_capex"}[key]
    values = pd.to_numeric(ttm[col], errors="coerce")
    start = max(0, len(ttm) - window)
    idx = next((i for i in range(start, len(ttm)) if pd.notna(values.iloc[i])), None)
    if idx is None:
        return None, None
    cal = review.calendar
    quarter = next((q for q in cal.quarters if q.end == pd.Timestamp(ttm.at[idx, "quarter_end"])), None)
    four = cal.previous_quarters(quarter, 4) if quarter is not None else None
    if not four:
        return idx, None
    earliest = four[0]
    if earliest.k == 1:
        return idx, 0.0
    prior = cal.quarter(earliest.fy, earliest.k - 1)
    row = review.tables[key].get(cal.years[earliest.fy].start, prior.end) if prior is not None else None
    return idx, (None if row is None else float(row["value"]))


def series_rev(review, ttm: pd.DataFrame, options: ReferenceOptions) -> list:
    return list(pd.to_numeric(ttm["ttm_revenue"], errors="coerce"))


def series_gp(review, ttm: pd.DataFrame, options: ReferenceOptions) -> list:
    return list(pd.to_numeric(ttm["ttm_gross_profit"], errors="coerce"))


def series_cfo(review, ttm: pd.DataFrame, options: ReferenceOptions) -> list:
    values = list(pd.to_numeric(ttm["ttm_ocf"], errors="coerce"))
    if options.emulate_first_ytd:
        i, extra = _first_window_ytd(review, "ocf", ttm, options.window)
        if i is not None and extra is not None:
            values[i] = values[i] + extra
    return values


def series_fcf(review, ttm: pd.DataFrame, options: ReferenceOptions) -> list:
    ocf = pd.to_numeric(ttm["ttm_ocf"], errors="coerce")
    capex = pd.to_numeric(ttm["ttm_capex"], errors="coerce")
    if options.missing_capex_as_zero:
        capex = capex.where(capex.notna() | ocf.isna(), 0.0)
    values = list(ocf - capex)
    if options.emulate_first_ytd:
        i, extra_ocf = _first_window_ytd(review, "ocf", ttm, options.window)
        _, extra_capex = _first_window_ytd(review, "capex", ttm, options.window)
        if i is not None and extra_ocf is not None and _num(values[i]):
            values[i] = values[i] + extra_ocf - (extra_capex or 0.0)
    return values


def series_eps(review, ttm: pd.DataFrame, options: ReferenceOptions) -> list:
    return list(pd.to_numeric(ttm["eps_ttm"], errors="coerce"))


def series_eq(review, ttm: pd.DataFrame, options: ReferenceOptions) -> list:
    return list(pd.to_numeric(ttm["equity"], errors="coerce"))


SERIES_BUILDERS = {"Rev": series_rev, "EPS": series_eps, "FCF": series_fcf, "GP": series_gp, "CFO": series_cfo, "EQ": series_eq}


def series_for(review, metric: str, options: ReferenceOptions, offset: int = 0) -> list:
    ttm = _ttm(review)
    if ttm.empty:
        return []
    if offset:
        ttm = ttm.iloc[: len(ttm) - offset].reset_index(drop=True)
    return SERIES_BUILDERS[metric](review, ttm, options)


def review_counts(review, options: ReferenceOptions | None = None, reference_row: pd.Series | None = None) -> dict[str, dict]:
    options = options or ReferenceOptions()
    offset = alignment_offset(review, reference_row) if options.align_window else 0
    out = {}
    for metric in METRIC_COLUMNS:
        window = options.equity_window if metric == "EQ" else options.window
        values = series_for(review, metric, options, offset)
        counts = window_counts(values, window, options.record_counts_first)
        counts["offset"] = offset
        counts["fingerprint"] = fingerprint(values, window, metric)
        out[metric] = counts
    return out


# ---------------------------------------------------------------- comparison

def load_reference(path) -> pd.DataFrame:
    frame = pd.read_excel(path, dtype=str)
    if "Ticker" not in frame.columns:
        raise ValueError("Reference workbook needs a 'Ticker' column.")
    return frame.set_index(frame["Ticker"].str.strip().str.upper())


def _ref_num(reference: pd.DataFrame, ticker: str, col: str):
    if col not in reference.columns:
        return np.nan
    return pd.to_numeric(str(reference.at[ticker, col]).replace(",", "").replace("%", ""), errors="coerce")


def per_ticker(reviews: dict, reference: pd.DataFrame, options: ReferenceOptions | None = None) -> pd.DataFrame:
    options = options or ReferenceOptions()
    rows = []
    for ticker, review in reviews.items():
        if ticker not in reference.index or review is None or review.ttm.empty:
            continue
        mine = review_counts(review, options, reference.loc[ticker])
        for metric in METRIC_COLUMNS:
            row = {"Ticker": ticker, "metric": metric, "offset": mine[metric]["offset"]}
            for field in FIELDS:
                row[f"mine {field}"] = mine[metric][field]
                row[f"theirs {field}"] = _ref_num(reference, ticker, f"{metric} TTM {field}")
            fp = mine[metric]["fingerprint"]
            row.update({"mine slope": fp["slope"], "mine r2": fp["r2"], "mine cv": fp["cv"],
                        "theirs slope": _ref_num(reference, ticker, f"{metric} TTM Slope"),
                        "theirs r2": _ref_num(reference, ticker, f"{metric} TTM R2"),
                        "theirs cv": _ref_num(reference, ticker, f"{metric} TTM CV%")})
            rows.append(row)
    return pd.DataFrame(rows)


def summarize(per: pd.DataFrame) -> pd.DataFrame:
    summary = []
    for metric in METRIC_COLUMNS:
        sub = per[per["metric"] == metric] if not per.empty else per
        if sub.empty:
            continue
        same = sub["mine #Obs"] == sub["theirs #Obs"]
        slope_ok = ((sub["mine slope"] - sub["theirs slope"]).abs() <= 0.005 * sub["theirs slope"].abs()) & ((sub["mine r2"] - sub["theirs r2"]).abs() <= 0.002)
        for field in FIELDS:
            a, b = sub[f"mine {field}"], sub[f"theirs {field}"]
            both = a.notna() & b.notna()
            d = (a - b)[both]
            ds = (a - b)[both & same]
            summary.append({
                "metric": metric, "field": field, "tickers": int(both.sum()),
                "exact": round(float((d == 0).mean()), 3) if len(d) else np.nan,
                "within 1": round(float((d.abs() <= 1).mean()), 3) if len(d) else np.nan,
                "within 2": round(float((d.abs() <= 2).mean()), 3) if len(d) else np.nan,
                "same #Obs exact": round(float((ds == 0).mean()), 3) if len(ds) else np.nan,
                "series match": round(float(slope_ok[both].mean()), 3) if both.any() else np.nan,
            })
    return pd.DataFrame(summary)


def compare(reviews: dict, reference: pd.DataFrame, options: ReferenceOptions | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(agreement per metric and field, per-ticker mine/theirs) for reviews keyed by ticker."""
    per = per_ticker(reviews, reference, options)
    return summarize(per), per

# ============================== valuation/cli.py ==============================
"""Command line: review one company or a whole ticker list.

Nothing is written unless --out-dir (or VALUATION_OUTPUT_DIR) names a folder
that is not on the C: drive. Without it, results print to the console only.

  python -B -m valuation.cli review AAPL --as-of 2026-11-23 --out-dir D:\\valuation
  python -B -m valuation.cli batch --as-of 2026-11-23 --out-dir D:\\valuation --workers 4
  python -B -m valuation.cli batch --tickers my_list.txt          # summary only, nothing written
  python -B -m valuation.cli review --facts CIK0000320193.json --prices prices.csv \\
         --split-adjusted yes --dividend-adjusted no --currency USD --as-of 2026-11-23
"""


import argparse
import gzip
import json
import sys
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import date
from pathlib import Path

import pandas as pd



def fetch_inputs(identifier: str, as_of: date, current_price: float | None = None, docs: list[InputBDocument] | None = None) -> ReviewInputs:
    ticker, cik = sources.resolve(identifier)
    facts = sources.fetch_companyfacts(cik)
    notes = [f"Input A: SEC companyfacts {facts.url} retrieved {facts.retrieved} (UTC)."]
    prices, meta = None, PriceMeta(ticker=ticker)
    try:
        px = sources.fetch_prices(ticker)
        prices, meta = px.frame, px.meta
        notes.append(
            f"Input C: Yahoo Finance chart API {px.url} retrieved {px.retrieved} (UTC); exchange {px.exchange}; "
            "close is split-adjusted, not dividend-adjusted; dividends are cash amounts on ex-dates; split events with ex-dates."
        )
    except Exception as exc:  # prices are optional: price ratios become "no price data"
        notes.append(f"Input C: not available ({exc}).")
    return ReviewInputs(facts=facts.data, as_of=as_of, prices=prices, price_meta=meta, ticker=ticker, current_price=current_price, filings_text=docs or [], source_notes=notes)


def local_inputs(args) -> ReviewInputs:
    meta = PriceMeta.from_json(args.price_meta) if args.price_meta else PriceMeta()
    pass  # was: relative import (names are global in the single cell)

    if args.split_adjusted is not None:
        meta.split_adjusted = _as_bool(args.split_adjusted)
    if args.dividend_adjusted is not None:
        meta.dividend_adjusted = _as_bool(args.dividend_adjusted)
    if args.currency:
        meta.currency = args.currency.upper()
    facts_path = Path(args.facts)
    data = json.loads(gzip.decompress(facts_path.read_bytes()) if facts_path.suffix == ".gz" else facts_path.read_text(encoding="utf-8"))
    docs = [InputBDocument(Path(p).name, Path(p).read_text(encoding="utf-8", errors="replace")) for p in (args.filing_text or [])]
    notes = [f"Input A: local file {facts_path.name}."]
    if args.prices:
        notes.append(f"Input C: local file {Path(args.prices).name}.")
    return ReviewInputs(
        facts=data, as_of=args.as_of, prices=args.prices, price_meta=meta, dividends=args.dividends, ticker=args.ticker or meta.ticker,
        current_price=args.price, filings_text=docs, source_notes=notes,
    )


def summarize(review) -> dict:
    snap = review.snapshot.set_index("key") if not review.snapshot.empty else pd.DataFrame()
    stats = review.stats.set_index("series") if not review.stats.empty else pd.DataFrame()
    rf = review.red_flags
    dq = review.dq.frame()

    def val(key):
        if key not in snap.index:
            return None
        v = snap.at[key, "value"]
        return v if isinstance(v, str) or v is None else float(v)

    def st(key, field):
        return stats.at[key, field] if key in stats.index else None

    return {
        "ticker": review.header["ticker"],
        "company": review.header["company"],
        "cik": review.header["cik"],
        "status": "thin coverage" if review.thin else "ok",
        "latest_filing": review.header["latest_filing"],
        "price": review.snapshot_inputs.get("price"),
        "price_date": str(review.snapshot_inputs.get("price_date").date()) if review.snapshot_inputs.get("price_date") is not None else "",
        "pe": val("pe"), "ps": val("ps"), "pb": val("pb"), "pfcf": val("pfcf"), "ev_ebitda": val("ev_ebitda"), "fcf_yield": val("fcf_yield"), "div_yield": val("div_yield"),
        "pe_percentile": review.percentiles.get("pe", {}).get("percentile"),
        "revenue_raw_increase_rate": st("ttm_revenue", "raw_increase_rate"),
        "eps_raw_increase_rate": st("eps_ttm", "raw_increase_rate"),
        "eps_adj_increase_rate": st("eps_ttm", "adj_increase_rate"),
        "eps_latest_record": st("eps_ttm", "raw_latest_record"),
        "serious_flags": int(rf["severity"].isin(["serious", "critical"]).sum()) if not rf.empty else 0,
        "flags": "; ".join(sorted(set(rf.loc[rf["severity"].isin(["serious", "critical"]), "flag"]))) if not rf.empty else "",
        "dq_warnings": int(dq["severity"].isin(["warning", "serious", "critical"]).sum()) if not dq.empty else 0,
        "error": "",
    }


def _safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in text)


def run_one(identifier: str, as_of: date, out: str | None, current_price: float | None = None) -> dict:
    """Fetch, review and (optionally) write one company. Returns a summary row."""
    try:
        inputs = fetch_inputs(identifier, as_of, current_price)
        review = run_review(inputs)
        row = summarize(review)
        if out:
            folder = Path(out)
            stem = _safe_name(f"{review.header['ticker'] or review.header['cik']}_{as_of.isoformat()}")
            workbook_path = None
            if not review.thin:
                workbook_path = folder / f"{stem}_valuation.xlsx"
                workbook_path.write_bytes(build_workbook(review))
            (folder / f"{stem}_report.md").write_text(build_report(review, str(workbook_path) if workbook_path else None), encoding="utf-8")
            row["workbook"] = str(workbook_path) if workbook_path else ""
        return row
    except Exception as exc:
        return {"ticker": identifier, "status": "error", "error": f"{type(exc).__name__}: {exc}", "trace": traceback.format_exc(limit=3)}



# ===================================== RUN ============================================
import os as _os
import threading as _threading
from concurrent.futures import ThreadPoolExecutor as _ThreadPool
from datetime import date as _date

_os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT

try:
    from IPython.display import Markdown as _Markdown, display as _display
except Exception:  # plain Python: print instead
    _Markdown = None

    def _display(obj):
        print(obj)


def _as_of() -> _date:
    return _date.fromisoformat(AS_OF) if isinstance(AS_OF, str) else (AS_OF or _date.today())


def _ticker_list() -> list[str]:
    if TICKERS is None:
        tickers = default_tickers()
    elif isinstance(TICKERS, str):
        tickers = parse_ticker_list(TICKERS)
    else:
        tickers = parse_ticker_list(",".join(TICKERS))
    return tickers[:LIMIT] if LIMIT else tickers


_lock = _threading.Lock()


def review_one(ticker: str):
    """Fetch Inputs A and C into memory and run the review. Returns (ticker, review or None, error)."""
    try:
        price = CURRENT_PRICES.get(ticker)
        review = run_review(fetch_inputs(ticker, _as_of(), price))
        return ticker, review, ""
    except Exception as exc:
        return ticker, None, f"{type(exc).__name__}: {exc}"


def run_all() -> tuple[dict, dict, pd.DataFrame]:
    tickers = _ticker_list()
    reviews, errors, rows = {}, {}, []
    print(f"Reviewing {len(tickers)} ticker(s) as of {_as_of()} with {WORKERS} thread(s); "
          f"output: {OUTPUT_DIR or 'memory only (nothing written)'}")
    with _ThreadPool(max_workers=max(1, WORKERS)) as pool:
        for k, (ticker, review, error) in enumerate(pool.map(review_one, tickers), start=1):
            if review is None:
                errors[ticker] = error
                rows.append({"ticker": ticker, "status": "error", "error": error})
            else:
                reviews[ticker] = review
                rows.append(summarize(review))
            if k % 25 == 0 or k == len(tickers):
                print(f"  {k}/{len(tickers)} done ({len(errors)} error(s))")
    return reviews, errors, pd.DataFrame(rows)


def workbook(ticker: str) -> bytes:
    """The Excel workbook (live formulas) for one reviewed ticker, as bytes."""
    return build_workbook(REVIEWS[ticker.upper()])


def report(ticker: str, show: bool = True) -> str:
    """The markdown report (sections 1-10) for one reviewed ticker."""
    text = build_report(REVIEWS[ticker.upper()])
    if show:
        _display(_Markdown(text) if _Markdown else text)
    return text


def daily(ticker: str) -> pd.DataFrame | None:
    """The daily price-ratio table (held fundamentals) for one reviewed ticker."""
    return REVIEWS[ticker.upper()].daily


def save(ticker: str, folder: str) -> list[str]:
    """Write one company's workbook and report to `folder` (refused on the C: drive)."""
    out = output_dir(folder)
    review = REVIEWS[ticker.upper()]
    stem = _safe_name(f"{review.header['ticker'] or review.header['cik']}_{_as_of().isoformat()}")
    paths = []
    if not review.thin:
        (out / f"{stem}_valuation.xlsx").write_bytes(build_workbook(review))
        paths.append(str(out / f"{stem}_valuation.xlsx"))
    (out / f"{stem}_report.md").write_text(build_report(review), encoding="utf-8")
    paths.append(str(out / f"{stem}_report.md"))
    return paths


def chart(ticker: str, ratio: str = "pe", series: str = "eps_ttm"):
    """Daily ratio over the display window plus the held TTM value as a step line (needs altair)."""
    import altair as alt

    review = REVIEWS[ticker.upper()]
    d = review.daily
    if d is None:
        raise ValueError(f"No daily series for {ticker}: {review.price_block_reason or 'no price data'}")
    w = d[d["in_display_window"]].reset_index()
    df = pd.DataFrame({"date": w["date"], "ratio": w[ratio], "held": pd.to_numeric(w[series], errors="coerce"),
                       "filing": w["form"].astype(str) + " " + w["accn"].astype(str), "valuation_date": w["valuation_date"]})
    tip = ["date:T", alt.Tooltip("ratio:Q", format=",.2f"), alt.Tooltip("held:Q", format=",.4g"), "filing:N", "valuation_date:T"]
    rules = alt.Chart(pd.DataFrame({"v": sorted(set(df["valuation_date"].dropna()))})).mark_rule(strokeDash=[2, 3], opacity=0.4).encode(x="v:T")
    top = alt.layer(rules, alt.Chart(df).mark_line(color="#2a78d6", invalid=None).encode(x="date:T", y=alt.Y("ratio:Q", title=RATIO_BY_KEY[ratio].label), tooltip=tip)).properties(width=760, height=260)
    bottom = alt.layer(rules, alt.Chart(df).mark_line(color="#eb6834", interpolate="step-after", invalid=None).encode(x="date:T", y=alt.Y("held:Q", title=series), tooltip=tip)).properties(width=760, height=180)
    return alt.vconcat(top, bottom)


REVIEWS, ERRORS, SUMMARY = run_all()
_display(SUMMARY)

if OUTPUT_DIR and WRITE_FILES:
    _out = output_dir(OUTPUT_DIR)  # raises StorageRefused for a C: drive folder
    for _t in REVIEWS:
        save(_t, str(_out))
    SUMMARY.to_csv(_out / f"batch_summary_{_as_of().isoformat()}.csv", index=False)
    print(f"Wrote {len(REVIEWS)} report(s) and workbooks to {_out}")

COMPARISON = COMPARISON_DETAIL = None
if REFERENCE_XLSX:
    _opts = ReferenceOptions.reference_compatible() if REFERENCE_COMPATIBLE else ReferenceOptions()
    COMPARISON, COMPARISON_DETAIL = compare(REVIEWS, load_reference(REFERENCE_XLSX), _opts)
    if COMPARISON.empty:
        print(f"No reviewed ticker is in {REFERENCE_XLSX}; nothing to compare.")
    else:
        print(f"Share of tickers whose counts match {REFERENCE_XLSX} exactly ({'reference conventions' if REFERENCE_COMPATIBLE else 'written spec'}):")
        _display(COMPARISON.pivot(index="field", columns="metric", values="exact").reindex(FIELDS))

if SHOW_FIRST_REPORT and REVIEWS:
    report(next(iter(REVIEWS)))

print("Done. Use REVIEWS[ticker], SUMMARY, ERRORS, report(t), workbook(t) -> bytes, daily(t), chart(t), save(t, folder)"
      + (", COMPARISON, COMPARISON_DETAIL" if REFERENCE_XLSX else "") + ".")
